# Workup: Scans 110789, 110790, 110791, 110792
KSR-216&emsp;-&emsp;PS/SiNP nanocomposite with D=50nm SiNPs, Mw=259kDa PS, unwashed

Repeat energy scans (Energy = 284.8, 286 eV)

Measured April 2026 at SST1

In [ ]:
gpu_to_use = 2
workup_pass = "05"

# Install necessary packages

I recommend creating a new environment, then install the necesary packages in this section with the following pip commands. After running these for the first time in your new environment, you shouldn't need to run these again. (Although it doesn't hurt.)

In [ ]:
%pip install numpy
%pip install tiled
%pip install databroker
%pip install xarray
%pip install cupy-cuda12x
%pip install joblib
%pip install PyHyperScattering[bluesky,ui]==0.2.10

# Imports & Checking versions

These cells need to be run each time the kernel is restarted

In [ ]:
import PyHyperScattering as phs
from PyHyperScattering import __version__

print(f"USING PYHYPER VERSION: {__version__}")

In [ ]:
import xarray
from xarray import __version__

print(f"USING XARRAY VERSION: {__version__}")

import numpy as np
from numpy import __version__

print(f"USING NUMPY VERSION: {__version__}")

import tiled
from tiled import __version__

print(f"USING TILED VERSION: {__version__}")

import databroker
from databroker import __version__

print(f"USING DATABROKER VERSION: {__version__}")

import holoviews
from holoviews import __version__
print(f"USING HOLOVIEWS VERSION: {__version__}")

import hvplot
from hvplot import __version__
print(f"USING HVPLOT VERSION: {__version__}")

In [ ]:
import pickle
import sys
import os
import pyopencl as cl
import psutil
from joblib import Parallel, delayed
from pathlib import Path

import pandas as pd
import datetime
import time

import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.colors import LogNorm
from matplotlib.pyplot import cm
from scipy.signal import savgol_filter
import matplotlib.patches as patches
import scipy.optimize
from scipy.optimize import curve_fit
from scipy.integrate import odeint
from scipy import stats
import matplotlib.patches as mpatches
from scipy.signal import find_peaks
from skimage.draw import polygon2mask
import textwrap
from matplotlib.gridspec import GridSpec
from scipy.signal import periodogram
from scipy.stats import entropy
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import brute, minimize
from matplotlib.patches import Rectangle, Circle
import scipy.ndimage
import matplotlib.ticker as mticker
from matplotlib.colors import ListedColormap
from matplotlib.ticker import ScalarFormatter
from matplotlib.ticker import LogLocator, FuncFormatter
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import imageio
from matplotlib.ticker import MultipleLocator

import cupy as cp
import cupyx.scipy.ndimage

import pathlib
basePath = pathlib.Path('.').absolute()

import math
import warnings

from tiled.client import from_uri

In [ ]:
#pin to specific GPU
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_to_use}"
print(f'USING GPU: {os.environ["CUDA_VISIBLE_DEVICES"]}\n')

# Load Scans

Here, we're setting it up with `corr_mode="none"`, which means no I₀ correction is being applied at this stage. I expect in the future we'll revise I₀ handling, but for now we'll load the uncorrected spectra and plan to make corrections later in the workflow.

In [ ]:
#NOTE: run this cell, then, without clicking away, type in your BNL login. You should get a DUO push, and then you'll have access.

import httpx
# for long scans, use the httpx.Timeout(None), which turns off timeout

cat = from_uri("https://tiled.nsls2.bnl.gov", timeout=httpx.Timeout(None))
load = phs.load.SST1RSoXSDB(corr_mode="none", catalog=cat["rsoxs"]["raw"])

In [ ]:
# # Display Catalog

# #You will only be able to see scans that your BNL ID is attached to

# # load.searchCatalog flags include cycle="2024", sample = "Dow", plan = "rsoxs" etc
# df = load.searchCatalog(cycle="2025-1", debugWarnings=True, project="Randazzo", plan = "list_scan", proposal_id=316980)
# from IPython.display import display, HTML

# # Puts the scrollbar next to the DataFrame
# display(
#     HTML(
#         "<div style='height: 400px; overflow: auto; width: fit-content'>"
#         + df.to_html(notebook=True)
#         + "</div>"
#     )
# )

In [ ]:
#Loading the scans. Right now I'm only examining individual samples, so a set of scans of pol=0,90,45,135 corresponding to a single sample. 
#A dictionary works well for this.

#dictionary for scan polarization: scan ID
scan_ids = {
    0: 110789,
    90: 110790,
    45: 110791,
    135: 110792,
}

#I include these because I like to popular figure titles with this info, which helps me keep track of my data better. 
sample_name = "KSR-216"
diameter = "50nm"
NP_vendor = "Nanocomposix"
mw = "259kDa"
mn = "253.5kDa"
mw_exact = 259000
mn_exact = 253500
pdi = 1.02
polymer = "PS"
wash_state = "unwashed"
scan_type = "repeats"
beam_decay_status = "No beam decay"
proposal = "319665"
cycle = "2026-1"
adc_mode = "4adc"
base_description = (f"Nanocomposite with D={diameter} SiNPs, Mw={mw} {polymer}, {wash_state}")

scan = {}

for pol, sid in scan_ids.items():
    data = load.loadRun(sid)
    scan[pol] = {
        "scan_id": sid,
        "base_description": base_description,
        "description": f"{base_description}, polarization={pol}°",
        "data": data,
        #I only include sdd and pixel_size so there's an option to manually calculate q and chi from x and y. You may not be interested in doing that.
        #Here, you can add other attributes if you want to be able to pull them up later.
        "sdd": data.attrs["sdd"], #in mm(?)
        "pixel_size": data.attrs["pixel1"], #in m(?)
        #some other things too:
        #"au_mesh_current": xarray.DataArray(data.attrs["RSoXS Au Mesh Current"], dims=["energy"], coords={"energy": data.energy}),
        "au_mesh_current": xarray.DataArray(data.attrs["RSoXS Au Mesh Current"], dims=["system"], coords={"energy": ("system", data.energy.values)}),
        "ring_current":data.attrs["NSLS-II Ring Current"],
        "date_time":data.attrs["meas_time"],
        "sample_name": sample_name,
        "diameter": diameter,
        "NP_vendor": NP_vendor,
        "mw": mw,
        "mn": mn,
        "mw_exact": mw_exact,
        "mn_exact": mn_exact,
        "pdi": pdi,
        "polymer": polymer,
        "wash_state": wash_state,
        "scan_type": scan_type,
        "beam_decay_status": beam_decay_status,
        "proposal": proposal,
        "cycle": cycle,
        "adc_mode": adc_mode,
        "workup_pass": workup_pass,
        # although here we're only defining a dictionary, when this cell is run the loop executes, 
        # and the result (the loaded data) is stored in the dictionary under the "data" key for each polarization
    }

#we'll also fill in this dictionary as we go. it involves anisotropy calculated from multiple pol scans
A_calcs = {}

In [ ]:
#sanity check

def check_polarizations(scan0, scan90, scan45, scan135):
    try:
        pol0 = scan0.attrs["polarization"]
        pol90 = scan90.attrs["polarization"]
        pol45 = scan45.attrs["polarization"]
        pol135 = scan135.attrs["polarization"]
        if pol0 == 0.0 and pol90 == 90.0 and pol45 == 45.0 and pol135 == 135.0:
            print("OK: polarizations are scan0 = 0.0, scan90 = 90.0, scan45 = 45.0, scan135 = 135.0")
        else:
            print("WARNING: Unexpected polarization values:")
            print(f"scan0 polarization = {pol0} (expected 0.0)")
            print(f"scan90 polarization = {pol90} (expected 90.0)")
            print(f"scan45 polarization = {pol45} (expected 45.0)")
            print(f"scan135 polarization = {pol135} (expected 135.0)")
    except KeyError as e: 
        print(f"WARNING: Missing polarization attribute in {e}")

def pol_description_check():
    #checking sample description text to ensure correctly labeled pols
    if "polarization=0" not in scan[0]["description"]:
        print("WARNING: Description for pol=0 does not indicate expected polarization=0")
    else:
        print("OK: Description correctly indicates polarization=0")
    if "polarization=90" not in scan[90]["description"]:
        print("WARNING: Description for pol=90 does not indicate expected polarization=90")
    else:
        print("OK: Description correctly indicates polarization=90")
    if "polarization=45" not in scan[45]["description"]:
        print("WARNING: Description for pol=45 does not indicate expected polarization=45")
    else:
        print("OK: Description correctly indicates polarization=45")
    if "polarization=135" not in scan[135]["description"]:
        print("WARNING: Description for pol=135 does not indicate expected polarization=135")
    else:
        print("OK: Description correctly indicates polarization=135")

#check for polarization & labeling errors
check_polarizations(scan[0]["data"], scan[90]["data"], scan[45]["data"], scan[135]["data"])
pol_description_check()

# Inspect Raw Scans and Frames

In [ ]:
#inspect scan at pol=0
scan[0]["data"]

In [ ]:
#inspect scan at pol=90
scan[90]["data"]

In [ ]:
#inspect scan at pol=45
scan[45]["data"]

In [ ]:
#inspect scan at pol=135
scan[135]["data"]

In [ ]:
#inspect raw frames at selected E

plt.style.use("dark_background") #Dean's darkmode preference

energy_indices_to_plot = [0, 1] 

for pol, scan_data in scan.items():

    fig, axs = plt.subplots(1, 2, figsize=(15, 8), constrained_layout=True)
    axs = axs.flatten()

    energy_vals = scan_data["data"].energy.values
    energy_frames = [scan_data["data"].isel(system=idx).values for idx in energy_indices_to_plot]

    for ax, frame_img, energy_val in zip(axs, energy_frames, energy_vals[energy_indices_to_plot]):
        im = ax.imshow(frame_img, norm=LogNorm(vmin=1e2, vmax=1e6), cmap="jet", origin="lower")
        ax.set_title(f"Energy = {energy_val:.1f} eV")
        ax.axis("off")

    cbar = fig.colorbar(im, ax=axs, shrink = 0.75, label="Intensity")

    fig.suptitle(f"Selected energy frames for Scan {scan_data["scan_id"]}: {sample_name} \n{scan_data["description"]}", fontsize=16)

    plt.show()
    plt.close()

Let's also visualize all of the darks taken. The next few cells run very slowly, presumably due to some bottleneck somewhere in the data loading.

In [ ]:
for pol, scan_data in scan.items():

    #number of unique dark scan_ids referenced in this scan's data
    unique_dark_ids = scan[pol]["data"].dark_id.values
    num_unique_darks = len(set(unique_dark_ids.flatten()))
    print(f"# of darks taken for pol={pol}: {num_unique_darks}")

    #accessing scan id and run
    scan_id = scan[pol]["scan_id"]
    run = load.c[scan_id]

    #getting the dark_id from the first image in the data array
    first_dark_id = int(scan[pol]["data"].dark_id.isel(system=0).item())

    #getting full dark data array and loading it into memory
    dark_data = run["dark"]["data"].read() #now this is an xarray.DataArray
    dark_img_stack = dark_data["Wide Angle CCD Detector_image"] #relevant variable

    print(f"dark_img_stack shape: {dark_img_stack.shape}    expected: (num_darks, 1, 1024, 1026)") #gives (num_darks, 1, 1024, 1026)

    #squeeze out the useless dimension to get shape (num_darks, 1024, 1026)
    darks = dark_img_stack.squeeze().load() #the .load() puts it fully in memory
                                            #otherwise it's suuuuper slow doing anything with the darks
                                            #something about lazy loading.
                                            #Follow-up note: Still very slow. Will probably just live with it.

    print(f"darks shape:          {darks.shape}       expected: (num_darks, 1024, 1026)")
    print(f"darks dims: \n    before rename: {darks.dims}")

    # renaming spatial dims only
    darks = darks.rename({"dim_1": "pix_y", "dim_2": "pix_x"})
    print(f"    after rename:  {darks.dims}   expected: (time, pix_y, pix_x)\n")

    #storing in scan dictionary
    scan[pol]["darks"] = darks

For the next part, the assumption is that darks data is essentially equivalent across respective darks images. If that's not the case, your approach to identifying and handling outliers might need revising.

In [ ]:
low_ile = 0
high_ile = 99.98 #a little arbitrary. snuck up on it. could go higher.

for pol, scan_data in scan.items():

    darks = scan[pol]["darks"]

    #flattening array and removing removing NaNs
    darks_flat = darks.values.ravel()
    darks_flat = darks_flat[~np.isnan(darks_flat)]

    low, high = np.nanpercentile(darks_flat, [low_ile, high_ile])

    fig, axs = plt.subplots(1,2,figsize=(15,4))
    axs[0].hist(darks_flat, bins=200)
    axs[1].hist(darks_flat, range=(low, high), bins=200)
    axs[0].set_xlabel("Dark Intensity")
    axs[1].set_xlabel("Dark Intensity")
    axs[0].set_ylabel("Pixel Count")
    axs[1].set_ylabel("Pixel Count")
    axs[0].set_title("Full Intensity Range")
    axs[1].set_title(f"Intensity from {low_ile} to {high_ile} %ile")

    fig.suptitle(f"Histogram of Intensity in Darks Data for Scan {scan_data['scan_id']} \n{scan_data['description']}")

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
def plot_dark_frames(low_ile_quadrant, high_ile_quadrant, quadrant, plot_full_intensity=True):
#inspect dark frames for each pol with two scaling modes (full range, limited percentile range)

    for pol, scan_data in scan.items():

        if quadrant == "all":
            darks = scan_data["darks"]
        elif quadrant == "lower left":
            darks = scan_data["darks"].isel(pix_y=slice(0, 512), pix_x=slice(0, 513))
        elif quadrant == "lower right":
            darks = scan_data["darks"].isel(pix_y=slice(0, 512), pix_x=slice(513, 1026))
        elif quadrant == "upper left":
            darks = scan_data["darks"].isel(pix_y=slice(512, 1024), pix_x=slice(0, 513))
        elif quadrant == "upper right":
            darks = scan_data["darks"].isel(pix_y=slice(512, 1024), pix_x=slice(513, 1026))

        if "time" in darks.dims:
            num_darks = darks.sizes["time"]
        else:
            num_darks = 1
            darks = darks.expand_dims("time")

        scaling_modes = {}

        #only add full intensity if we want it
        if plot_full_intensity:
            scaling_modes["Full Intensity Range"] = {"vmin": np.nanmin(darks), "vmax": np.nanmax(darks)}

        scaling_modes["Percentile Intensity Range"] = {"vmin": np.nanpercentile(darks, low_ile_quadrant), "vmax": np.nanpercentile(darks, high_ile_quadrant)}

        for mode_name, scale in scaling_modes.items():

            vmin = scale["vmin"]
            vmax = scale["vmax"]

            fig_w = num_darks * 4 + 2
            fig, axs = plt.subplots(1, num_darks, figsize=(fig_w, 6), constrained_layout=True)
            axs = np.atleast_1d(axs).flatten()

            for idx, ax in enumerate(axs):

                frame = darks.isel(time=idx)

                min_val = float(frame.min())
                max_val = float(frame.max())

                percentile_low_ile = float(np.nanpercentile(frame, low_ile_quadrant))
                percentile_high_ile = float(np.nanpercentile(frame, high_ile_quadrant))

                print(f"[pol{pol} - {mode_name}] Frame {idx}: {low_ile_quadrant} %ile = {percentile_low_ile:.1f}, {high_ile_quadrant} %ile = {percentile_high_ile:.1f}")

                im = ax.imshow(frame, norm=LogNorm(vmin=vmin, vmax=vmax), cmap="jet", origin="lower")
                ax.set_title(f"Index {idx}\nMin: {min_val:.0f}    Max: {max_val:.0f}\n{low_ile_quadrant} %ile: {percentile_low_ile:.0f}   {high_ile_quadrant} %ile: {percentile_high_ile:.0f}")
                ax.axis("off")

            cbar = fig.colorbar(im, ax=axs, shrink=0.75, label="Intensity")

            if mode_name == "Full Intensity Range":
                mode_label = "Full Intensity Range"
            else:
                mode_label = f"{low_ile_quadrant}-{high_ile_quadrant} Percentile Intensity Range"

            if quadrant == "all":
                fig.suptitle(f"Dark Frames for Scan {scan_data['scan_id']}\n{mode_label}\n{scan_data['description']}\n", fontsize=14)
            else:
                fig.suptitle(f"Dark Frames for Scan {scan_data['scan_id']} ({quadrant} quadrant)\n{mode_label}\n{scan_data['description']}\n", fontsize=14)

            plt.show()
            plt.close()

In [ ]:
#view dark frames in their entirety
plot_dark_frames(low_ile_quadrant=low_ile, high_ile_quadrant=high_ile, quadrant="all", plot_full_intensity=True)

In [ ]:
#view upper right quadrant
plot_dark_frames(low_ile_quadrant=0, high_ile_quadrant=80, quadrant="upper right", plot_full_intensity=False)

In [ ]:
#view lower right quadrant
plot_dark_frames(low_ile_quadrant=0, high_ile_quadrant=80, quadrant="lower right", plot_full_intensity=False)

In [ ]:
#view upper left quadrant
plot_dark_frames(low_ile_quadrant=0, high_ile_quadrant=80, quadrant="upper left", plot_full_intensity=False)

In [ ]:
#view lower left quadrant
plot_dark_frames(low_ile_quadrant=0, high_ile_quadrant=80, quadrant="lower left", plot_full_intensity=False)

The outliers in the darks usually correspond to spikes in the data readouts, so we're best off masking them out now. Let's build a 2D boolean mask of shape (pix_y, pix_x) that flags any pixel location that ever goes above our upper percentile bound in any dark frame. That way, once we've identified those bad pixel locations, we can reuse the mask the clean all our data.

In [ ]:
darks_masks = {}  #going to store one mask per polarization

for pol, scan_data in scan.items():
    darks = scan_data["darks"]   # expected shape: (time, pix_y, pix_x)

    #calculate cutoff across *all values* in this polarization's darks
    cutoff = np.nanpercentile(darks, high_ile)
    print(f"pol={pol}")
    print(f"    {high_ile} %ile cutoff = {cutoff:.2f}")

    #building mask
    #True where a pixel EVER exceeds cutoff in ANY frame
    if "time" in darks.dims:
        mask = (darks.max(dim="time") > cutoff) #max over time compares pixel-wise maxima to cutoff
    else:
        mask = (darks > cutoff)

    #save mask
    darks_masks[pol] = mask

    print(f"    mask shape = {mask.shape}")
    print(f"    masked pixels = {mask.values.sum()}\n")

# Now we can use masks[pol] later to clean our data, e.g.:
# cleaned = scan[pol]["darks"].where(~masks[pol])

We now have a darks-based mask associated with each polarization in `darks_masks`

# Pre-processing: Masking, Beam Centering, & Integrating

First we'll build a mask. Then we'll use the masked scan to rigorously calculate the center, based on the function i developed in July 2025. 

Then we'll use that center to feed into the integration function, which asks for 3 things: integrator, center, mask.

Because we have strong bragg rings, we can check on the goodness of fit for the center using Dan's/Julia's method too.

## Construct a beamstop mask

This is one of the key steps to "reduced" data that can be compared, integrated. 

In [ ]:
# SUM SCATTERING SAMPLE TO DEFINE BEAMSTOP

# NOTE sometimes loadRun doesn't automatically identify the correct dims;  you can hint it with dims
# as below.
# loaded_for_mask= load.loadRun(90588,dims = ['energy','polarization'])
# NOTE when loading fails, it's frequently a dim hinting issue

# data can be loaded using LoadRun or loadSeries. Be aware that they seem to structure the data differently.
# Generally, loadRun is preferred, but loadSeries can be tried if loadRun fails
# note that loadrun_kwargs sometimes needs to be hinted as well. The automatic id of dims seems to be
# where there are differences betweeen loadRun and loadSeries
# loaded_for_mask= load.loadSeries([90588],'sample_name',loadrun_kwargs={'dims':['energy','polarization']})

#We don't usually analyze data this way, but for the beamstop mask, 
#we sum all the energies and polarizations to get the silhouette of the beamstop

#creating dictionary with all loaded summed data
summed_frames = {pol: scan_data["data"].sum("system") for pol, scan_data in scan.items()}

fig, axs = plt.subplots(2, 2, figsize=(12 ,12), constrained_layout=True)

for ax, pol in zip(axs.flat, [0,90,45,135]):
    im = summed_frames[pol].plot(ax=ax, norm=LogNorm(vmin=1e0, vmax=1e9), cmap="plasma", add_colorbar=False)
    ax.set_aspect("equal")
    ax.set_title(f"Polarization = {pol}°")

fig.colorbar(im, ax=axs, orientation="vertical", shrink=0.75, label="Summed Intensity")

plt.show()
plt.close()

Previous notebooks used an annotator tool to draw the mask. Drawing the mask was super glitchy for both Matt and I, so I don't tend to use it. I have an easier time manually defining the vertices of polygons which form the mask. However, the annotator tool may be useful to decide on vertices because when you hover your mouse of a point it (usually) displays coordinates. I'm including the (commented out) cell below in case you wish to try your luck with the annotator, but I usually don't bother.

In [ ]:
# draw_mask = phs.IntegrationUtils.DrawMask(summed_frames[0], clim=(1e0,1e1))
# draw_mask.ui()

In [ ]:
#we'll use this to draw the beamstop mask

def rotated_rectangle(cx, cy, L, W, theta_deg):
    theta = np.deg2rad(theta_deg)

    # half dimensions
    dx = L / 2
    dy = W / 2

    corners = np.array([[-dx, -dy], [ dx, -dy], [ dx,  dy], [-dx,  dy]]) #rectangle in local coordinates
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta),  np.cos(theta)] ]) #rotation matrix

    rotated = corners @ R.T
    rotated[:, 0] += cx
    rotated[:, 1] += cy

    return rotated

def outside_circle_mask(shape, center_x, center_y, radius):

    yy, xx = np.indices(shape)
    
    return (xx - center_x)**2 + (yy - center_y)**2 > radius**2

In [ ]:
#Make a mask out of polygons

#specify vertices manually in (x,y) format
#comment out and/or add polygons as needed

beamstop_polygon = rotated_rectangle(
    cx=743, cy=189,     # <-- set these
    L=900, W=54,        # <-- set these
    theta_deg=-50       # <-- set this
)

print([(float(x), float(y)) for x, y in beamstop_polygon])

polygon_list = [
    [  # Polygon 1 - beamstop (from rotated rectangle)
        tuple(p) for p in beamstop_polygon
    ],
    [  # Polygon 2 - left hand side
        (2, 0),
        (0, 0),
        (0, 1023),
        (2, 1023)
         
    ],
    [  # Polygon 3 - right hand side
        (1025, 0),
        (1023, 0),
        (1023, 1023),
        (1025, 1023)
    ],
    [  # Polygon 4 - bottom
        (1025, 0),
        (0, 0),
        (0, 5),
        (1025, 5)
    ],
    [  # Polygon 5 - top
        (1025, 1023),
        (1025, 1018),
        (0, 1018),
        (0, 1023)
    ],
    # [  # Polygon 6 - center
    #     (900, 516),
    #     (900, 506),
    #     (490, 506),
    #     (490, 516)
    # ],
    #Add more polygons as needed
]

polygon_masks = np.any([polygon2mask(summed_frames[0].shape, [(y, x) for x, y in np.round(polygon).astype(int)]) for polygon in polygon_list], axis=0)
all_masks = polygon_masks + [darks_masks[pol] for pol in darks_masks.keys()] #collapse all darks masks + beamstop mask
static_mask = np.any(all_masks, axis=0)
inverted_mask = ~static_mask

fig, axs = plt.subplots(4, 4, figsize=(16, 18))

pols = [0, 90, 45, 135]

for row_idx, pol in enumerate(pols):
    data = summed_frames[pol]

    #summed scattering
    axs[row_idx, 0].imshow(data, norm=LogNorm(vmin=1e0,vmax=1e9), cmap="plasma", origin="lower")
    axs[row_idx, 0].set_title(f"total summed scattering, pol={pol}°")

    #polygon outline overlay
    axs[row_idx, 1].imshow(data, norm=LogNorm(vmin=1e0,vmax=1e9), cmap="plasma", origin="lower")
    for polygon in polygon_list:
        patch = mpatches.Polygon(np.round(polygon).astype(int), closed=True, edgecolor="cyan", facecolor="none", linewidth=1)
        axs[row_idx, 1].add_patch(patch)
    axs[row_idx, 1].set_title(f"w/ polygon overlay, pol={pol}°")

    #shaded mask overlay        
    axs[row_idx, 2].imshow(data, norm=LogNorm(vmin=1e0,vmax=1e9), cmap="plasma", origin="lower")
    axs[row_idx, 2].imshow(static_mask, cmap="gray", alpha=0.4, origin="lower")
    axs[row_idx, 2].set_title(f"w/ full manual mask overlay, pol={pol}°")

    #zoomed in
    axs[row_idx, 3].imshow(data, norm=LogNorm(vmin=1e0,vmax=1e9), cmap="plasma", origin="lower")
    axs[row_idx, 3].imshow(static_mask, cmap="gray", alpha=0.4, origin="lower")
    axs[row_idx, 3].set_title(f"w/ full mask, zoomed in, pol={pol}°")
    axs[row_idx, 3].set_ylim(400, 600)
    axs[row_idx, 3].set_xlim(400, 600)

    for col in range(4):
        axs[row_idx, col].set_xlabel("pix_x")
        axs[row_idx, col].set_ylabel("pix_y")
        axs[row_idx, col].set_aspect("equal")

caption = f"List of vertices for building mask: {polygon_list}"
wrapped_caption = textwrap.fill(caption, width=180)
fig.text(0.05, 0.04, wrapped_caption, fontsize=12, ha='left', va='bottom')
fig.suptitle(
    f'Masking of summed energy frames for Scans ' 
    f'{scan[0]["scan_id"]} (pol=0°), {scan[90]["scan_id"]} (pol=90°), {scan[45]["scan_id"]} (pol=45°), & {scan[135]["scan_id"]} (pol=135°) of {sample_name}\n' 
    f'{base_description}\n', 
    fontsize=16
)

#fig.subplots_adjust(bottom=0.1, top=0.92, hspace=0.35, wspace=0.2)
fig.subplots_adjust(left=0.05, right=0.98, bottom=0.10, top=0.92, hspace=0.35, wspace=0.05)
plt.show()
plt.close()

#confirming that mask is true and outside the mask is false
print("Vals removed from diffraction image will be where mask=False:")
print(f'top row of static_mask (masked vals=true, unmasked=false): \n {static_mask[-1]}')
print(f"top row of static_mask after inversion (masked vals=true, unmasked=false): \n {inverted_mask[-1]}")

#adding to dictionary
scan[0]["masked_data_static"] = scan[0]["data"].where(inverted_mask)
scan[90]["masked_data_static"] = scan[90]["data"].where(inverted_mask)
scan[45]["masked_data_static"] = scan[45]["data"].where(inverted_mask)
scan[135]["masked_data_static"] = scan[135]["data"].where(inverted_mask)

#confirming that the masked data we just made is structured similarly to the scan made during integration during my "standard" workup
#(it's a little different because i've been relying on the built-in PHS function that applies the mask during integration, 
#but here i'd like to see the masked unintegrated scan)

print(f'\nscan[0]["data"] type: {type(scan[0]["data"])}')
print(f'scan[0]["masked_data_static"] type: {type(scan[0]["masked_data_static"])}')
print(f'scan[0]["data"] dims: {scan[0]["data"].dims}')
print(f'scan[0]["masked_data_static"] dims: {scan[0]["masked_data_static"].dims}')
print(f'scan[0]["data"] shape: {scan[0]["data"].shape}')
print(f'scan[0]["masked_data_static"] shape: {scan[0]["masked_data_static"].shape}\n')

print(f'Masked values should now be nan\nTop row of scan[0]["masked_data_static"]:\n{scan[0]["masked_data_static"].values[-1]}')

So now the polygons in `polygon_masks` and all pol-specific masks in `darks_masks` have been collapsed into a single `static_mask` which is the same for all energies and polarizations.

Next we work on masking out saturated pixels, which show up as negative intensity values in the final scans. Keep in mind that pixels might saturate differently at different energies or pols.

`combined_neg_masks` will look across all pols to find a pixel that is saturated at any pol for a given energy.

In [ ]:
#inspect masked frames --> saturated pixels will show up in white

energy_indices_to_plot = [0, 1]

combined_neg_masks = [np.zeros(scan[0]["masked_data_static"].shape[1:], dtype=bool) for _ in energy_indices_to_plot]

for pol, scan_data in scan.items():

    fig, axs = plt.subplots(1, 2, figsize=(15, 8), constrained_layout=True)
    axs = axs.flatten()

    energy_vals = scan_data["masked_data_static"].energy.values
    energy_frames = [scan_data["masked_data_static"].isel(system=idx).values for idx in energy_indices_to_plot]

    for i, (ax, frame_img, energy_val) in enumerate(zip(axs, energy_frames, energy_vals[energy_indices_to_plot])):

        im = ax.imshow(frame_img, norm=LogNorm(vmin=1e1, vmax=1e6), cmap="jet", origin="lower")
        ax.set_title(f"Energy = {energy_val:.1f} eV")
        ax.axis("off")

        #make negative pixels (which is how saturation shows up) appear in white
        white_cmap = ListedColormap(["white"])
        neg_mask = frame_img < 0
        combined_neg_masks[i] |= neg_mask   #combine masks across polarizations for this energy only

        ax.imshow(np.ma.masked_where(~neg_mask, neg_mask), cmap=white_cmap, origin="lower")

    cbar = fig.colorbar(im, ax=axs, shrink = 0.75, label="Intensity")

    fig.suptitle(f'Selected energy frames after masking for Scan {scan_data["scan_id"]}: {sample_name} \n{scan_data["description"]}', fontsize=16)

    plt.show()
    plt.close()

Visualize saturated pixels and check for distortions around the polygon mask boundaries

In [ ]:
#zoomed in masked frames
#adjust vmin, vmax as necessary to make sure no distortions are unmasked around beamstop

for pol, scan_data in scan.items():

    fig, axs = plt.subplots(1, 2, figsize=(15, 8), constrained_layout=True)
    axs = axs.flatten()

    energy_vals = scan_data["masked_data_static"].energy.values
    frames = scan_data["masked_data_static"].values  # full (energy, y, x)

    for ax, idx in zip(axs, energy_indices_to_plot):

        frame_img = frames[idx]

        zoom = frame_img[360:560, 420:620]
        zoom_masked = np.ma.masked_where(zoom < 0, zoom)
        im = ax.imshow(zoom_masked, norm=LogNorm(vmin=1e1, vmax=1e6), cmap="jet", origin="lower")

        ax.set_title(f"Energy = {energy_vals[idx]:.1f} eV")
        ax.axis("off")

        neg_mask = (zoom < 0)
        white_cmap = ListedColormap(["white"])
        ax.imshow(np.ma.masked_where(~neg_mask, neg_mask), cmap=white_cmap, origin="lower")

    fig.colorbar(im, ax=axs, shrink=0.75, label="Intensity")
    fig.suptitle(f"Selected energy frames after masking for Scan {scan_data['scan_id']}: {sample_name}\n{scan_data['description']}", fontsize=16)

    plt.show()
    plt.close()

In [ ]:
#zoomed in masked frames
#adjust vmin, vmax as necessary to make sure no distortions are unmasked around beamstop

for pol, scan_data in scan.items():

    fig, axs = plt.subplots(1, 2, figsize=(15, 8), constrained_layout=True)
    axs = axs.flatten()

    energy_vals = scan_data["masked_data_static"].energy.values
    frames = scan_data["masked_data_static"].values  # full (energy, y, x)

    for ax, idx in zip(axs, energy_indices_to_plot):

        frame_img = frames[idx]

        zoom = frame_img[180:380, 550:750]
        zoom_masked = np.ma.masked_where(zoom < 0, zoom)
        im = ax.imshow(zoom_masked, norm=LogNorm(vmin=1e1, vmax=1e4), cmap="jet", origin="lower")

        ax.set_title(f"Energy = {energy_vals[idx]:.1f} eV")
        ax.axis("off")

        neg_mask = (zoom < 0)
        white_cmap = ListedColormap(["white"])
        ax.imshow(np.ma.masked_where(~neg_mask, neg_mask), cmap=white_cmap, origin="lower")

    fig.colorbar(im, ax=axs, shrink=0.75, label="Intensity")
    fig.suptitle(f"Selected energy frames after masking for Scan {scan_data['scan_id']}: {sample_name}\n{scan_data['description']}", fontsize=16)

    plt.show()
    plt.close()

In [ ]:
#zoomed in masked frames
#adjust vmin, vmax as necessary to make sure no distortions are unmasked around beamstop

for pol, scan_data in scan.items():

    fig, axs = plt.subplots(1, 2, figsize=(15, 8), constrained_layout=True)
    axs = axs.flatten()

    energy_vals = scan_data["masked_data_static"].energy.values
    frames = scan_data["masked_data_static"].values  # full (energy, y, x)

    for ax, idx in zip(axs, energy_indices_to_plot):

        frame_img = frames[idx]

        zoom = frame_img[0:200, 700:900]
        zoom_masked = np.ma.masked_where(zoom < 0, zoom)
        im = ax.imshow(zoom_masked, norm=LogNorm(vmin=1e1, vmax=1e3), cmap="jet", origin="lower")

        ax.set_title(f"Energy = {energy_vals[idx]:.1f} eV")
        ax.axis("off")

        neg_mask = (zoom < 0)
        white_cmap = ListedColormap(["white"])
        ax.imshow(np.ma.masked_where(~neg_mask, neg_mask), cmap=white_cmap, origin="lower")

    fig.colorbar(im, ax=axs, shrink=0.75, label="Intensity")
    fig.suptitle(f"Selected energy frames after masking for Scan {scan_data['scan_id']}: {sample_name}\n{scan_data['description']}", fontsize=16)

    plt.show()
    plt.close()

In [ ]:
#visualize neg_mask

energy_indices_to_plot = [0, 1]

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

for ax, energy_idx in zip(axs, energy_indices_to_plot):

    ys_neg, xs_neg = np.where(combined_neg_masks[energy_idx])

    ax.scatter(xs_neg, ys_neg, c="cyan", s=1)
    ax.set_title(f"Energy idx = {energy_idx}")

plt.show()
plt.close()

#plot lims are automatic default right now

Next, we use the saturated pixels near the center to calculate the `center_masks`, which give a radius-based mask to cut out saturated pixels near the diffraction image center, or else to remove the center of the image which is largely useless due to the beamstop interfering with the angular range for calculating anisotropy.

In [ ]:
#do a guess if just starting out. otherwise put the fitted beamcenter val here
beamcenter_x_guess = 473.1
beamcenter_y_guess = 514.7
saturation_radius = 90  #range around beamcenter to look over for saturated pixels

beamcenter = np.array([beamcenter_x_guess, beamcenter_y_guess])
xmin_idx = int(beamcenter_x_guess - saturation_radius / 2)
xmax_idx = int(beamcenter_x_guess + saturation_radius / 2)
ymin_idx = int(beamcenter_y_guess - saturation_radius / 2)
ymax_idx = int(beamcenter_y_guess + saturation_radius / 2)

#beamstop vertex calculations
beamstop_vertices = np.array([(x, y) for x, y in beamstop_polygon if x >= 0 and y >= 0])
dx_beamstop = beamstop_vertices[:, 0] - beamcenter[0]
dy_beamstop = beamstop_vertices[:, 1] - beamcenter[1]
distances = (dx_beamstop**2 + dy_beamstop**2)**0.5
farthest_beamstop = beamstop_vertices[np.argmax(distances)] #np.argmax returns an index
rmax_beamstop = distances[np.argmax(distances)]

print(f"beamstop_vertices: {[(round(float(x), 2), round(float(y), 2)) for x, y in beamstop_vertices]}")
print(f"farthest_beamstop: ({float(farthest_beamstop[0]):.2f}, {float(farthest_beamstop[1]):.2f})\n")

center_masks = np.zeros((len(energy_indices_to_plot), *mask.shape), dtype=bool)     #initialize masks, one per energy

for energy_i, energy_idx in enumerate(energy_indices_to_plot):
    print(f"Calculating center mask for energy_idx={energy_idx}")

    mask = combined_neg_masks[energy_idx]

    cropped = mask[ymin_idx:ymax_idx, xmin_idx:xmax_idx] #crop for plotting (need to make adjustments in computation)
    y_cropped, x_cropped = np.where(cropped) #grab only the True (saturated) pixels 

    fig, axs = plt.subplots(1,2,figsize=(10,5))
    axs[0].imshow(mask, origin="lower", cmap="gray")

    if len(x_cropped) == 0:
        rmax_pt = farthest_beamstop
        rmax = rmax_beamstop
        print(f"No masked (saturated) pixels in search area. Making center mask with max radius from beamstop vertex at {rmax_pt[0]:.2f}, {rmax_pt[1]:.2f}) (rmax = {rmax})")
        
    else:
        #convert cropped coords -> full detector coords
        x_full = x_cropped + xmin_idx
        y_full = y_cropped + ymin_idx
        points_full = np.column_stack([x_full, y_full]) #turn x and y pts into coord pairs
        dx = points_full[:, 0] - beamcenter[0]
        dy = points_full[:, 1] - beamcenter[1]
        distances = np.sqrt(dx**2 + dy**2)
        farthest_sat = points_full[np.argmax(distances)]
        rmax_sat = distances[np.argmax(distances)]
        
        if rmax_sat >= rmax_beamstop: #if saturation extends beyond reach of beamstop vertices
            rmax_pt = farthest_sat
            rmax = rmax_sat
            axs[0].scatter(x_full, y_full, s=5, c="cyan", label="Masked (saturated) pixels")
            print(f"Making center mask with max radius from saturation extent at {rmax_pt[0]:.2f}, {rmax_pt[1]:.2f}) (rmax = {rmax})")
        else:
            rmax_pt = farthest_beamstop
            rmax = rmax_beamstop
            print(f"Making center mask with max radius from beamstop vertex at {rmax_pt[0]:.2f}, {rmax_pt[1]:.2f}) (rmax = {rmax})")

    axs[0].scatter(rmax_pt[0], rmax_pt[1], c="goldenrod", s=100, label="rmax pt")

    midmask_y, midmask_x = np.indices(mask.shape) #indices for middle mask
    midmask_coords = np.stack([midmask_x, midmask_y], axis=-1) #shape (y,x,2)
    dx_all = midmask_coords[:, :, 0] - beamcenter[0]
    dy_all = midmask_coords[:, :, 1] - beamcenter[1]
    r_all = np.sqrt(dx_all**2 + dy_all**2)
    center_mask = r_all < rmax
    center_masks[energy_i] = center_mask

    axs[1].imshow(center_mask, origin="lower", cmap="gray")
    axs[1].set_title("Center mask")

    for ax in axs:
        ax.scatter(beamcenter[0], beamcenter[1], c="red", s=80, label="beam center")
        ax.set_xlim(xmin_idx, xmax_idx)
        ax.set_ylim(ymin_idx, ymax_idx)

    plt.tight_layout()
    plt.show()
    plt.close()

Now we combine the `static_mask` (same for all energies/pols) with the energy-dependent `center_masks` and `combined_neg_masks` (both energy dependent, but collapsed across all pols)

In [ ]:
#combining center_masks and combined_neg_masks (for each energy) with static_mask (constant) 
final_masks = [static_mask | center_mask | neg_mask for center_mask, neg_mask in zip(center_masks, combined_neg_masks)]
print(f"final_masks[0].shape: {final_masks[0].shape}")

fig, axs = plt.subplots(1,2,figsize=(12,6))

axs[0].imshow(final_masks[0], origin="lower", cmap="gray")
axs[1].imshow(final_masks[1], origin="lower", cmap="gray")

plt.tight_layout()
plt.show()
plt.close()

Define `masked_data_final` and identify out radius of accessible range to determine the max allowable q

In [ ]:
outer_radius = 459

for pol, scan_data in scan.items():

    scan[pol]["final_masks"] = final_masks
    scan[pol]["masked_data_final"] = scan[pol]["data"].copy()
    
    for i, mask in enumerate(final_masks):
        scan[pol]["masked_data_final"][i] = (scan[pol]["masked_data_final"][i].where(~mask))

    fig, axs = plt.subplots(1, 2, figsize=(15, 8), constrained_layout=True)
    axs = axs.flatten()

    energy_vals = scan_data["masked_data_final"].energy.values
    energy_frames = [scan_data["masked_data_final"].isel(system=idx).values for idx in energy_indices_to_plot]

    for i, (ax, frame_img, energy_val) in enumerate(zip(axs, energy_frames, energy_vals[energy_indices_to_plot])):

        im = ax.imshow(frame_img, norm=LogNorm(vmin=1e1, vmax=1e6), cmap="jet", origin="lower")
        circle = Circle((beamcenter_x_guess, beamcenter_y_guess), outer_radius, fill=False, edgecolor="fuchsia", linestyle="--", linewidth=2)
        ax.add_patch(circle)

        ax.set_title(f"Energy = {energy_val:.1f} eV")
        ax.axis("off")

    cbar = fig.colorbar(im, ax=axs, shrink = 0.75, label="Intensity")

    fig.suptitle(f'Selected energy frames after masking for Scan {scan_data["scan_id"]}: {sample_name} \n{scan_data["description"]}', fontsize=16)

    plt.show()
    plt.close()

In [ ]:
#checking dims for consistency

print(scan[0]["masked_data_final"].dims)
print(scan[0]["data"].dims) #system dim is energy
print(scan[0]["data"].shape) #expected (energy, pix_y, pix_X) = (2,1024,1026)
print(scan[pol]["final_masks"][0].shape) #this is energy idx=0
print(scan[pol]["final_masks"][1].shape) #this is energy idx=1

In [ ]:
#run through here

## Beam centering calculations

This beam-centering approach works best when your data contain several well-defined rings. If, instead, your pattern shows a more gradual, smooth decrease in intensity away from the beam center with no distinct ring features it may not perform as reliably. In such cases, you might consider alternative centering methods or simply approximate the center visually for now. I plan to work on  more robust centering strategies in the future.

The cell immediately below defines some functions, and accordingly only needs to be run once.

In [ ]:
#calculate best beam center for 
    #pol 0, 45, 90, 135
        #normal masking, reflected masking
#compare all calculated centers


#incorporate:
    #-center cutout with edges equal to q=distance between beam center and beamstop mask vertices farthest away
    #-center cutout that takes into acount saturation somehow... same q range cutout thing
    #-energy specific fits


The following 3 cells must be run iteratively. Change the center coordinates each time

In [ ]:
def wrap_chi(vals):
    return ((np.asarray(vals) + 180) % 360) - 180

def get_chi_slice(data, chi_center, chi_halfwidth): #lifted directly from pyhyperscattering
    """
    Return a slice centered at `chi_center` ± chi_halfwidth, handling wrapping manually.
    Assumes chi ranges from ~-180 to 180.
    """
    chi_vals = data['chi'].values
    chi_min = chi_center - chi_halfwidth
    chi_max = chi_center + chi_halfwidth

    # Wrap chi into [-180, 180)
    chi_vals_wrapped = ((chi_vals + 180) % 360) - 180
    chi_min_wrapped = ((chi_min + 180) % 360) - 180
    chi_max_wrapped = ((chi_max + 180) % 360) - 180

    if chi_min_wrapped <= chi_max_wrapped:
        mask = (chi_vals_wrapped >= chi_min_wrapped) & (chi_vals_wrapped <= chi_max_wrapped)
    else:
        mask = (chi_vals_wrapped >= chi_min_wrapped) | (chi_vals_wrapped <= chi_max_wrapped)

    return data.sel(chi=mask)

def rotate_mask(masked_pixels, reflection_mode): #chi-based reflection

    if reflection_mode == "none":
        return masked_pixels

    masked_pixels_2d = masked_pixels.any(dim="energy") #cllapse energy
    
    chi_grid = masked_pixels_2d['chi'].values
    q_grid = masked_pixels_2d['q'].values
    
    masked_i, masked_j = np.where(masked_pixels_2d)  #masked pts in indices, as a boolean
    masked_chi = chi_grid[masked_i]
    masked_q = q_grid[masked_j]

    chi_shifts = {
        "twofold": [180],
        "fourfold": [0, 90, 180, -90],
        "eightfold": [0, 45, 90, 135, 180, -45, -90, -135],
    }
    
    reflection = np.zeros_like(masked_pixels_2d.values)

    all_chi = []
    all_q = []

    for shift in chi_shifts[reflection_mode]:
        all_chi.append(wrap_chi(masked_chi + shift))
        all_q.append(masked_q)
    
    all_chi = np.concatenate(all_chi)
    all_q = np.concatenate(all_q)


    for chi_val, q_val in zip(all_chi, all_q):
        i = np.argmin(np.abs(chi_grid - chi_val))
        j = np.argmin(np.abs(q_grid - q_val))
        reflection[i, j] = True

    full_mask_2d = masked_pixels_2d| reflection #original bad pixels + their reflections

    #expand back to 3d by adding energy dim
    energy_dim = masked_pixels.sizes["energy"]
    full_mask_3d = masked_pixels.copy()
    full_mask_3d.values = np.broadcast_to(full_mask_2d.values, (energy_dim, *full_mask_2d.shape)).transpose(0, 1, 2)
    
    return full_mask_3d

def integrate(data, pol, manual_mask, beamcenter_x, beamcenter_y, scan_id, sample_name, sample_base_description, threshold, energy_idx_to_plot, mask_reflection_mode, apply_postmask, verbose, show_plots):

    loaded_scan = data.unstack("system") #I understand this to mean that unstacking by system means that we can separately access x, y, and energy from a given scan
    manual_mask = np.array(manual_mask, copy=True)  # make writable copy for this worker

    integrator = phs.integrate.PFEnergySeriesIntegrator(geomethod="template_xr", template_xr=loaded_scan, integration_method="csr")
    integrator.ni_beamcenter_x = beamcenter_x
    integrator.ni_beamcenter_y = beamcenter_y
    integrator.mask = manual_mask
    
    remeshed = integrator.integrateImageStack(loaded_scan)
    if verbose == True:
        print(f"loaded (unstacked) scan shape: \n{loaded_scan.shape} \n(pix_y, pix_x, energy)")
        print(f'\nintegrated scan shape: \n{remeshed.shape} \n(energy, chi, q)')

    if apply_postmask == True:

        loaded_mask_scan = xarray.DataArray(np.repeat(~manual_mask[:, :, np.newaxis], loaded_scan.sizes['energy'], axis=2), 
                                            dims=loaded_scan.dims, coords={dim: loaded_scan[dim] for dim in loaded_scan.dims})

        integrator.mask = None
        remeshed_mask = integrator.integrateImageStack(loaded_mask_scan)
        if verbose == True:
            print(f"\nloaded (unstacked) mask scan shape: \n{loaded_mask_scan.shape} \n{loaded_mask_scan.dims}")
            print(f'\nintegrated mask shape: \n{remeshed_mask.shape} \n{remeshed_mask.dims}')

        ones_input = xarray.full_like(loaded_mask_scan, 1)
        remeshed_ones = integrator.integrateImageStack(ones_input)
        if verbose == True:
            print(f"\nones input shape: \n{ones_input.shape} \n{ones_input.dims}")
            print(f'\nintegrated ones shape: \n{remeshed_ones.shape} \n{remeshed_ones.dims}')

        #APPLYING POSTMASK
        source_frac = remeshed_mask / remeshed_ones #fraction of eligible source pixels contributed to a given integrated pixel. 
                                                    #1 = exclude any integrated pixels missing contributions.
                                                    #0 = do no extra excluding based on missing contributinos from source pixels
        masked_pixels = (source_frac < threshold) | np.isnan(source_frac) #boolean mask where True = source_frac is below threshold or nan
        clean = remeshed.copy() #make a copy so don't overwrite the original
        clean.values[masked_pixels.values] = np.nan #apply mask: wherever masked_pixels is True, set to nan
        if verbose == True:
            print(f'\nclean integrated scan shape:: \n{clean.shape} \n{clean.dims}')

        symmetry_excluded = None #initialize in case mask_reflection is None
        if mask_reflection_mode == "twofold": #1 axis of symmetry only
            full_mask = rotate_mask(masked_pixels, reflection_mode="twofold")
            symmetry_excluded = remeshed.copy()
            symmetry_excluded.values[full_mask.values] = np.nan
        elif mask_reflection_mode == "fourfold": #2 axes of symmetry
            full_mask = rotate_mask(masked_pixels, reflection_mode="fourfold")
            symmetry_excluded = remeshed.copy()
            symmetry_excluded.values[full_mask.values] = np.nan
        elif mask_reflection_mode == "eightfold": #2 axes of symmetry
            full_mask = rotate_mask(masked_pixels, reflection_mode="eightfold")
            symmetry_excluded = remeshed.copy()
            symmetry_excluded.values[full_mask.values] = np.nan
        elif mask_reflection_mode is None:
            pass

        if show_plots == True:

            fig, axs = plt.subplots(2,3,figsize=(18,10))

            if mask_reflection_mode is None:
            
                datasets = [remeshed, remeshed_mask, remeshed_ones, source_frac, clean]

                titles = ["Integrated Data \n(with built-in PyHyper masking)", "Integrated Manual Mask", "Integrated Ones", 
                    "Normalized Fraction \nof Unmasked Source Pixels", "Clean Integrated Data"]
            
                cbar_labels = ["Intensity", "Source pixel count for each integrated pixel",  "Source pixel count for each integrated pixel", 
                    "Fraction of eligible source pixels \ncontributing to each integrated pixel", "Intensity"]

            else:

                datasets = [remeshed, remeshed_mask, remeshed_ones, source_frac, clean, symmetry_excluded]

                titles = ["Integrated Data \n(with built-in PyHyper masking)", "Integrated Manual Mask", "Integrated Ones", 
                    "Normalized Fraction \nof Unmasked Source Pixels", "Clean Integrated Data", "Symmetry-excluded"]
            
                cbar_labels = ["Intensity", "Source pixel count for each integrated pixel",  "Source pixel count for each integrated pixel", 
                    "Fraction of eligible source pixels \ncontributing to each integrated pixel", "Intensity", "Intensity"]

            #test display vals
            qvals = [0.01, 0.04, 0.11]
            chivals = [-50, 50, -100]
            colors = ["red", "cyan", "lawngreen"]

            for ax, ds, title, cbar_label in zip(axs.flat, datasets, titles, cbar_labels):
                
                test_slice = ds.isel(energy=energy_idx_to_plot)
                im = ax.pcolormesh(test_slice['q'], test_slice['chi'], test_slice, shading='auto')

                ax.set_title(title)
                ax.set_xlabel("q")
                ax.set_ylabel("chi")

                for qv, chv, c in zip(qvals, chivals, colors):
                    intensity = test_slice.sel(q=qv, chi=chv, method="nearest").values
                    ax.plot(qv, chv, marker="o", color=c, markersize=8, label=f"{intensity:.2f}")

                ax.legend(loc="upper right", fontsize=10)
                cbar = fig.colorbar(im, ax=ax, shrink=0.5)
                cbar.set_label(cbar_label)

            fig.suptitle(
                f"Intermediate Integration Results for Scan {scan_id} ({sample_name}): {sample_base_description}, pol={pol}\n"
                f"Representative Frames at Energy Index={energy_idx_to_plot}\n"
                f"Fitted Beam Center = ({beamcenter_x}, {beamcenter_y})"
            )

            plt.tight_layout()
            plt.show()
            plt.close()

        return remeshed, remeshed_mask, threshold, clean, remeshed_ones, symmetry_excluded

    else:

        return remeshed

def gaussian(x, A, x0, sigma, C):
    return A * np.exp(-(x - x0)**2 / (2 * sigma**2)) + C

def fit_peak(q, I, q_target, q_window_width):
    
    mask = (q > q_target - q_window_width) & (q < q_target + q_window_width)
    
    q_fit = q[mask]
    I_fit = I[mask]

    if len(q_fit) < 5 or np.all(~np.isfinite(I_fit)):
        return np.nan, (q_fit, I_fit), None

    #initial guesses
    A0 = I_fit.max() - I_fit.min()
    x0 = q_fit[np.argmax(I_fit)]
    sigma0 = (q_fit.max() - q_fit.min()) / 10
    C0 = I_fit.min()

    try:
        popt, _ = curve_fit(gaussian, q_fit, I_fit, p0=[A0, x0, sigma0, C0])
    except RuntimeError:
        return np.nan, (q_fit, I_fit), None
    
    A, x0, sigma, C = popt

    #smooth curve to be plotted 
    q_smooth = np.linspace(q_fit.min(), q_fit.max(), 300)
    I_smooth = gaussian(q_smooth, *popt)

    return x0, (q_fit, I_fit), (q_smooth, I_smooth)

def locate_peak_q(clean, pol, chi_halfwidth, energy_val, q_target, q_window_width):

    if energy_val == "all":
        clean_Esel = clean.mean('energy')
    else:
        clean_Esel = clean.sel(energy=energy_val, method="nearest")        

    if pol in [0, 90]:
        centers = [90, 180, -90, 0]

    elif pol in [45, 135]:
        centers = [45, 135, -45, -135]

    I_curves = {}
    peaks = {}
    fit_curves = {}

    for c in centers:
        I_vs_q = get_chi_slice(clean_Esel, c, chi_halfwidth).mean("chi")

        q_vals = I_vs_q['q'].values
        I_vals = I_vs_q.values
    
        peak_q, (q_fit, I_fit), fit_curve = fit_peak(q_vals, I_vals,  q_target=q_target, q_window_width=q_window_width)
        if fit_curve is None:
            q_smooth, I_smooth = None, None
        else:
            q_smooth, I_smooth = fit_curve
        
        I_curves[c] = {"q": q_vals, "I": I_vals}
        peaks[c] = peak_q
        fit_curves[c] = {"q_fit": q_fit, "I_fit": I_fit, "q_smooth": q_smooth, "I_smooth": I_smooth}

    return clean_Esel, I_curves, fit_curves, peaks

def beam_center_cost(data, pol, manual_mask, beamcenter_x, beamcenter_y, scan_id, sample_name, sample_base_description, chi_halfwidth, threshold, energy_val, energy_idx_to_plot, 
                     q_target, q_window_width, mask_reflection_mode, apply_postmask, verbose, show_plots):

    print(f"Testing beam center (x, y) = ({beamcenter_x}, {beamcenter_y})")

    _, _, _, clean, _, _ = integrate(data=data, pol=pol, manual_mask=manual_mask, beamcenter_x=beamcenter_x, beamcenter_y=beamcenter_y, 
                                     scan_id=scan_id, sample_name=sample_name, sample_base_description=sample_base_description, threshold=threshold,
                                     energy_idx_to_plot=energy_idx_to_plot, mask_reflection_mode=mask_reflection_mode,
                                     apply_postmask=apply_postmask, verbose=verbose, show_plots=show_plots) 

    _, _, _, peaks = locate_peak_q(clean=clean, pol=pol, chi_halfwidth=chi_halfwidth, energy_val=energy_val, q_target=q_target, q_window_width=q_window_width)

    if pol in [0, 90]:
        centers = [90, 180, -90, 0]
    elif pol in [45, 135]:
        centers = [45, 135, -45, -135]
    else:
        return np.nan

    peak_vals = np.array([peaks[c] for c in centers])
    peak_vals = peak_vals[np.isfinite(peak_vals)]

    if len(peak_vals) < 2:
        return np.nan

    return np.mean((peak_vals - np.mean(peak_vals))**2)

def cost_wrapper(params, #scipy optimize expects that the function passed only take the params being optimized as first arg
                 data, pol, manual_mask, scan_id, sample_name, sample_base_description, chi_halfwidth, threshold, energy_val, energy_idx_to_plot, q_target, q_window_width,
                 mask_reflection_mode, apply_postmask, verbose, show_plots): 
    
    beamcenter_x, beamcenter_y = params
    return beam_center_cost(data=data, pol=pol, manual_mask=manual_mask, beamcenter_x=beamcenter_x, beamcenter_y=beamcenter_y, scan_id=scan_id, sample_name=sample_name, 
                            sample_base_description=sample_base_description, chi_halfwidth=chi_halfwidth, threshold=threshold, energy_val=energy_val,
                            energy_idx_to_plot=energy_idx_to_plot,  q_target=q_target, q_window_width=q_window_width, mask_reflection_mode=mask_reflection_mode, 
                            apply_postmask=apply_postmask, verbose=verbose, show_plots=show_plots)

def plot_beam_center_fit(data, pol, manual_mask, beamcenter_x, beamcenter_y, scan_id, sample_name, sample_base_description, chi_halfwidth, threshold, energy_val, energy_idx_to_plot, q_target, q_window_width, 
                         best_x_coarse, best_y_coarse, search_step_coarse, cost_grid_coarse, best_x_fine, best_y_fine, cost_grid_fine, search_step_fine,
                         mask_reflection_mode, apply_postmask, verbose, show_plots):

    remeshed, _, _, clean, _, symmetry_excluded = integrate(data=data, pol=pol, manual_mask=manual_mask, beamcenter_x=beamcenter_x, beamcenter_y=beamcenter_y, 
                                                            scan_id=scan_id, sample_name=sample_name, sample_base_description=sample_base_description, threshold=threshold, 
                                                            energy_idx_to_plot=energy_idx_to_plot, mask_reflection_mode=mask_reflection_mode, apply_postmask=apply_postmask, 
                                                            verbose=verbose, show_plots=show_plots)

    clean_Esel, I_curves, fit_curves, peaks = locate_peak_q(clean=clean, pol=pol, chi_halfwidth=chi_halfwidth, energy_val=energy_val, q_target=q_target, q_window_width=q_window_width)

    if energy_val == "all":
        data_Esel = data.mean('system')
    else:      
        data_Esel = data.isel(system=energy_idx_to_plot)

    if energy_val == "all":
        print(f"\nCalculating center based on average across all energies")
    else:
        print(f"\nCalculating center based on energy = {energy_val}")
    
    print("\nFitted peak maxima located at:")
    for c in peaks:
        print(peaks[c])

    fig, axs = plt.subplots(2,3,figsize=(18,10))

    im = axs[0,0].pcolormesh(clean_Esel['q'], clean_Esel['chi'], clean_Esel.values, shading='auto')
    axs[0,0].set_xlim(0, clean_Esel['q'].max().item())
    axs[0,0].set_ylabel("chi")
    axs[0,0].set_xlabel("q")

    for c, curve in I_curves.items():
        axs[0,1].plot(curve["q"], curve["I"], label=str(c))
    axs[0,1].set_yscale("log")
    axs[0,1].set_ylabel("Intensity")
    axs[0,1].set_xlabel("q")
    axs[0,1].legend(loc="upper right")

    for c, curve in fit_curves.items():
        axs[0,2].scatter(curve["q_fit"], curve["I_fit"], s=10, label=f"{c} data")  #always plot data
        if curve["q_smooth"] is None:
            continue  #only skip the gaussian line
        axs[0,2].plot(curve["q_smooth"], curve["I_smooth"], label=f"{c} fit")

    axs[0,2].legend(loc="lower center")
    axs[0,2].set_ylabel("Intensity")
    axs[0,2].set_xlabel("q")

    axs[1,0].imshow(data_Esel, norm=LogNorm(vmin=1, vmax=1e5), origin='lower')
    axs[1,0].add_patch(plt.Circle((beamcenter_x, beamcenter_y), 5, color='deeppink'))
    axs[1,0].add_patch(plt.Circle((beamcenter_x, beamcenter_y), 50, color='deeppink', fill=False))
    axs[1,0].add_patch(plt.Circle((beamcenter_x, beamcenter_y), 150, color='deeppink', fill=False))
    axs[1,0].set_aspect("equal")

    coarse_mesh = axs[1,1].pcolormesh(cost_grid_coarse["x"], cost_grid_coarse["y"], cost_grid_coarse.values, shading="auto")
    axs[1,1].scatter(best_x_coarse, best_y_coarse, marker="x", color="cyan", s=100)
    cbar = fig.colorbar(coarse_mesh, ax=axs[1,1], shrink=0.5)
    cbar.set_label("cost")
    axs[1,1].set_xlabel("x")
    axs[1,1].set_ylabel("y")
    axs[1,1].set_xlim((cost_grid_coarse.x.min() - search_step_coarse/2), (cost_grid_coarse.x.max() + search_step_coarse/2))
    axs[1,1].set_ylim((cost_grid_coarse.y.min() - search_step_coarse/2), (cost_grid_coarse.y.max() + search_step_coarse/2))
    axs[1,1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    axs[1,1].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    axs[1,1].set_aspect('equal', adjustable='box')

    fine_mesh = axs[1,2].pcolormesh(cost_grid_fine["x"], cost_grid_fine["y"], cost_grid_fine.values, shading="auto")
    axs[1,2].scatter(best_x_fine, best_y_fine, marker="x", color="cyan", s=100)
    cbar = fig.colorbar(fine_mesh, ax=axs[1,2], shrink=0.5)
    cbar.set_label("cost")
    axs[1,2].set_xlabel("x")
    axs[1,2].set_ylabel("y")
    axs[1,2].set_xlim((cost_grid_fine.x.min() - search_step_fine/2), (cost_grid_fine.x.max() + search_step_fine/2))
    axs[1,2].set_ylim((cost_grid_fine.y.min() - search_step_fine/2), (cost_grid_fine.y.max() + search_step_fine/2))
    axs[1,2].xaxis.set_major_locator(mticker.MultipleLocator(0.1))
    axs[1,2].yaxis.set_major_locator(mticker.MultipleLocator(0.1))
    axs[1,2].set_aspect('equal', adjustable='box')

    fig.suptitle(f"Beam Center Calculation for Scan {scan_id} ({sample_name}): {sample_base_description}, pol={pol} \nFitted Beam Center = ({beamcenter_x}, {beamcenter_y})")

    plt.tight_layout()
    plt.show()
    plt.close()

    return remeshed, clean, symmetry_excluded


def find_best_beam_center(data, pol, manual_mask, beamcenter_x_guess, beamcenter_y_guess, scan_id, sample_name, sample_base_description, 
                          search_radius_coarse, search_step_coarse, search_radius_fine, search_step_fine, n_jobs, chi_halfwidth=45, threshold=1,
                          energy_val="all", energy_idx_to_plot=5, q_target=0.02201, q_window_width=0.003, mask_reflection_mode=None, apply_postmask=True, verbose=True, show_plots=True):

    if energy_val != "all":
        loaded_scan = data.unstack("system")
        energy_arr = loaded_scan["energy"].values
        energy_idx_to_plot = int(np.argmin(np.abs(energy_arr - energy_val)))
        print(f"Plots will be prepared using energy_val={energy_val}, energy_idx_to_plot={energy_idx_to_plot}\n")

    xs_coarse = np.arange(beamcenter_x_guess - search_radius_coarse, beamcenter_x_guess + search_radius_coarse + search_step_coarse/2, search_step_coarse)
    ys_coarse = np.arange(beamcenter_y_guess - search_radius_coarse, beamcenter_y_guess + search_radius_coarse + search_step_coarse/2, search_step_coarse)

    results_coarse = Parallel(n_jobs=n_jobs)(
        delayed(cost_wrapper)(
            np.array([x, y]), 
            data, pol, manual_mask, scan_id, sample_name, sample_base_description, chi_halfwidth, threshold, energy_val, energy_idx_to_plot, 
            q_target, q_window_width, mask_reflection_mode, apply_postmask, False, False)
        for y in ys_coarse
        for x in xs_coarse)

    cost_grid_coarse = xarray.DataArray(np.array(results_coarse).reshape(len(ys_coarse), len(xs_coarse)), coords={"y": ys_coarse, "x": xs_coarse}, dims=("y", "x"))

    best_coarse = cost_grid_coarse.argmin(...)  # finds flat index across all dims
    best_x_coarse = float(cost_grid_coarse.x[best_coarse["x"]])
    best_y_coarse = float(cost_grid_coarse.y[best_coarse["y"]])
    best_cost_coarse = float(cost_grid_coarse.sel(x=best_x_coarse, y=best_y_coarse))

    print(f"Best point (Coarse search): ({best_x_coarse}, {best_y_coarse}) with cost {best_cost_coarse:.6e}\n")

    xs_fine = np.arange(best_x_coarse - search_radius_fine, best_x_coarse + search_radius_fine + search_step_fine/2, search_step_fine)
    ys_fine = np.arange(best_y_coarse - search_radius_fine, best_y_coarse + search_radius_fine + search_step_fine/2, search_step_fine)
    xs_fine = np.round(xs_fine, 3) #round to 3 dec places to match expected val (i.e,. "fixing" float point precision)
    ys_fine = np.round(ys_fine, 3)

    results_fine = Parallel(n_jobs=n_jobs)(
        delayed(cost_wrapper)(
            np.array([x, y]), 
            data, pol, manual_mask, scan_id, sample_name, sample_base_description, chi_halfwidth, threshold, energy_val, energy_idx_to_plot, 
            q_target, q_window_width, mask_reflection_mode, apply_postmask, False, False)
        for y in ys_fine
        for x in xs_fine)

    cost_grid_fine = xarray.DataArray(np.array(results_fine).reshape(len(ys_fine), len(xs_fine)), coords={"y": ys_fine, "x": xs_fine}, dims=("y", "x"))

    best_fine = cost_grid_fine.argmin(...)          # <-- this is the "find lowest cost" step, since argmin(...) returns the indices of the minimum cost cell across all dims
    best_x_fine = float(cost_grid_fine.x[best_fine["x"]])   # x coord of that minimum
    best_y_fine = float(cost_grid_fine.y[best_fine["y"]])   # y coord of that minimum
    best_cost_fine = float(cost_grid_fine.sel(x=best_x_fine, y=best_y_fine))  # the cost value there, for printing

    print(f"Best point (Fine search): ({best_x_fine}, {best_y_fine}) with cost {best_cost_fine:.6e}")

    remeshed, clean, symmetry_excluded = plot_beam_center_fit(data=data, pol=pol, manual_mask=manual_mask, beamcenter_x=best_x_fine, beamcenter_y=best_y_fine, 
                                                              scan_id=scan_id, sample_name=sample_name, sample_base_description=sample_base_description, 
                                                              chi_halfwidth=chi_halfwidth, threshold=threshold, energy_val=energy_val, energy_idx_to_plot=energy_idx_to_plot, 
                                                              q_target=q_target, q_window_width=q_window_width, 
                                                              best_x_coarse=best_x_coarse, best_y_coarse=best_y_coarse, 
                                                              search_step_coarse=search_step_coarse, cost_grid_coarse=cost_grid_coarse, 
                                                              best_x_fine=best_x_fine, best_y_fine=best_y_fine, 
                                                              search_step_fine=search_step_fine, cost_grid_fine=cost_grid_fine,
                                                              mask_reflection_mode=mask_reflection_mode, apply_postmask=apply_postmask, verbose=verbose, show_plots=show_plots)

    return best_x_coarse, best_y_coarse, best_x_fine, best_y_fine, threshold, remeshed, clean, mask_reflection_mode, symmetry_excluded, cost_grid_coarse, cost_grid_fine


In [ ]:
from joblib import parallel_backend

#suppressing warnings globally (including in joblib workers)
os.environ["PYTHONWARNINGS"] = "ignore"

warnings.filterwarnings("ignore", message="Module pyFAI.azimuthalIntegrator is deprecated") # pyFAI deprecation warning
warnings.filterwarnings("ignore", message="Since mask was none") # PyHyperScattering mask warnings
warnings.filterwarnings("ignore", message="No mask defined")
warnings.filterwarnings("ignore", message="Unable to load optional dependency pygix") #pygix dependency warning

class SuppressStderr: #suppress stderr temporarily (kill the big stack trace spam)
    def __enter__(self):
        self._stderr = sys.stderr
        sys.stderr = open(os.devnull, "w")
    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stderr.close()
        sys.stderr = self._stderr

In [ ]:
cpu_percentages = psutil.cpu_percent(percpu=True, interval=1)
used = sum(1 for p in cpu_percentages if p > 10)
idle = sum(1 for p in cpu_percentages if p <= 10)

print(f"Total cores: {len(cpu_percentages)}")
print(f"Cores in use (>10% utilization): {used}")
print(f"Idle cores: {idle}")
print(f"Suggested n_jobs: {min((0.75*idle), 32)}")  #capping at 32 to be a good citizen

In [ ]:
#with saturation mask:
    #pol0: 473.1, 514.7
    #pol90: 473.1, 514.8
    #pol45:  473.0, 514.7
    #pol135:  473.0, 514.7

#FOURFOLD SYMMETRY, MASKING OUT MIDDLE STRIP
    #pol0:
    #pol90: 
    #pol45: 
    #pol135: 
#FOURFOLD SYMMETRY (NO MIDDLE MASK)
    #pol0: 473, 514.7
    #pol90:
    #pol45: 472.9, 514.7
    #pol135:

### pol=0 beam centering

In [ ]:
pol = 0

beamcenter_x_guess = 472
beamcenter_y_guess = 515
n_jobs = 32
data=scan[pol]["masked_data_final"]
#data = scan[pol]["data"].isel(system=slice(14, 15))  #to do 1 energy at a time. keeps the system dim, just with 1 energy at index (first index listed)
#data = scan[pol]["data"].isel(system=slice(34, 67)) #slice(34, 67) covers energy=283.8 to 287
scan_id=scan[pol]["scan_id"]
sample_name=scan[pol]["sample_name"]
sample_base_description=scan[pol]["base_description"]
with parallel_backend("threading"):
    with SuppressStderr():
        best_x_coarse, best_y_coarse, best_x_fine, best_y_fine, threshold, remeshed, clean, mask_reflection, clean_with_symmetry, cost_grid_coarse, cost_grid_fine, = find_best_beam_center(
            data=data, pol=pol, manual_mask=final_masks[0], beamcenter_x_guess=beamcenter_x_guess, beamcenter_y_guess=beamcenter_y_guess, 
            scan_id=scan_id, sample_name=sample_name, sample_base_description=sample_base_description,
            search_radius_coarse=4, search_step_coarse=1, search_radius_fine=0.8, search_step_fine=0.1, n_jobs=n_jobs, chi_halfwidth=45, threshold=1, 
            energy_val=284.8, energy_idx_to_plot=0,  q_target=0.02201, q_window_width=0.003, #energy_val can be a number or "all"
            mask_reflection_mode="fourfold", apply_postmask=True, #mask reflection mode can be None, "twofold", "fourfold", or "eightfold"
            verbose=True, show_plots=True)

#add to dict
scan[pol]["beamcenter_x_fit_coarse"] = best_x_coarse
scan[pol]["beamcenter_y_fit_coarse"] = best_y_coarse
scan[pol]["beamcenter_x_fit_fine"] = best_x_fine
scan[pol]["beamcenter_y_fit_fine"] = best_y_fine
scan[pol]["threshold"] = threshold
scan[pol]["remeshed"] = remeshed
scan[pol]["clean"] = clean
scan[pol]["mask_reflection"] = mask_reflection
scan[pol]["clean_with_symmetry"] = clean_with_symmetry
scan[pol]["cost_grid_coarse"] = cost_grid_coarse
scan[pol]["cost_grid_fine"] = cost_grid_fine

print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")


In [ ]:
print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")

In [ ]:
save_dir = "/homes/ksr2/rsoxs_workup/sst1_2026apr/beamcenter_pickles"
bc_data = {key: scan[pol][key] for key in ["beamcenter_x_fit_coarse", "beamcenter_y_fit_coarse", "beamcenter_x_fit_fine", "beamcenter_y_fit_fine",
                                             "threshold", "remeshed", "clean", "mask_reflection", "clean_with_symmetry",
                                             "cost_grid_coarse", "cost_grid_fine"]}

filename_bc = f"beamcenter_{scan[pol]['scan_id']}_{scan[pol]['mask_reflection']}_p{workup_pass}.pkl"
filepath_bc = os.path.join(save_dir, filename_bc)

print("starting beam center save")
t0 = time.time()
with open(filepath_bc, "wb") as f:
    pickle.dump(bc_data, f)
print(f"finished beam center save in {time.time() - t0:.1f} s")

In [ ]:
data=scan[pol]["data"]
pol=0
manual_mask = final_masks[0]#static_mask
beamcenter_x = best_x_fine
beamcenter_y = best_y_fine
scan_id=scan[pol]["scan_id"]
sample_name=scan[pol]["sample_name"]
sample_base_description=scan[pol]["base_description"]

_, _, _ = plot_beam_center_fit(data, pol, manual_mask, beamcenter_x, beamcenter_y, scan_id, sample_name, sample_base_description, chi_halfwidth=45, threshold=1, energy_val="all", energy_idx_to_plot=0, q_target=0.02201, q_window_width=0.003, 
                         best_x_coarse=best_x_coarse, best_y_coarse=best_y_coarse, search_step_coarse=1, cost_grid_coarse=cost_grid_coarse, best_x_fine=best_x_fine, best_y_fine=best_y_fine, cost_grid_fine=cost_grid_fine, search_step_fine=0.1,
                         mask_reflection_mode="fourfold", apply_postmask=True, verbose=True, show_plots=True)

print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")

If all looks good: restart the kernel, re-run the notebook right up to beamcentering, then skip this pol and do the next.

### pol=90 beam centering

In [ ]:
pol = 90

beamcenter_x_guess = 472
beamcenter_y_guess = 515
n_jobs = 32
data=scan[pol]["masked_data_final"]
#data = scan[pol]["data"].isel(system=slice(14, 15))  #to do 1 energy at a time. keeps the system dim, just with 1 energy at index (first index listed)
#data = scan[pol]["data"].isel(system=slice(34, 67)) #slice(34, 67) covers energy=283.8 to 287
scan_id=scan[pol]["scan_id"]
sample_name=scan[pol]["sample_name"]
sample_base_description=scan[pol]["base_description"]
with parallel_backend("threading"):
    with SuppressStderr():
        best_x_coarse, best_y_coarse, best_x_fine, best_y_fine, threshold, remeshed, clean, mask_reflection, clean_with_symmetry, cost_grid_coarse, cost_grid_fine, = find_best_beam_center(
            data=data, pol=pol, manual_mask=final_masks[0], beamcenter_x_guess=beamcenter_x_guess, beamcenter_y_guess=beamcenter_y_guess, 
            scan_id=scan_id, sample_name=sample_name, sample_base_description=sample_base_description,
            search_radius_coarse=4, search_step_coarse=1, search_radius_fine=0.8, search_step_fine=0.1, n_jobs=n_jobs, chi_halfwidth=45, threshold=1, 
            energy_val=284.8, energy_idx_to_plot=0,  q_target=0.02201, q_window_width=0.003, #energy_val can be a number or "all"
            mask_reflection_mode="fourfold", apply_postmask=True, #mask reflection mode can be None, "twofold", "fourfold", or "eightfold"
            verbose=True, show_plots=True)

#add to dict
scan[pol]["beamcenter_x_fit_coarse"] = best_x_coarse
scan[pol]["beamcenter_y_fit_coarse"] = best_y_coarse
scan[pol]["beamcenter_x_fit_fine"] = best_x_fine
scan[pol]["beamcenter_y_fit_fine"] = best_y_fine
scan[pol]["threshold"] = threshold
scan[pol]["remeshed"] = remeshed
scan[pol]["clean"] = clean
scan[pol]["mask_reflection"] = mask_reflection
scan[pol]["clean_with_symmetry"] = clean_with_symmetry
scan[pol]["cost_grid_coarse"] = cost_grid_coarse
scan[pol]["cost_grid_fine"] = cost_grid_fine

print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")


In [ ]:
print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")

In [ ]:
save_dir = "/homes/ksr2/rsoxs_workup/sst1_2026apr/beamcenter_pickles"
bc_data = {key: scan[pol][key] for key in ["beamcenter_x_fit_coarse", "beamcenter_y_fit_coarse", "beamcenter_x_fit_fine", "beamcenter_y_fit_fine",
                                             "threshold", "remeshed", "clean", "mask_reflection", "clean_with_symmetry",
                                             "cost_grid_coarse", "cost_grid_fine"]}

filename_bc = f"beamcenter_{scan[pol]['scan_id']}_{scan[pol]['mask_reflection']}_p{workup_pass}.pkl"
filepath_bc = os.path.join(save_dir, filename_bc)

print("starting beam center save")
t0 = time.time()
with open(filepath_bc, "wb") as f:
    pickle.dump(bc_data, f)
print(f"finished beam center save in {time.time() - t0:.1f} s")

In [ ]:
data=scan[pol]["data"]
pol=90
manual_mask = final_masks[0]#static_mask
beamcenter_x = best_x_fine
beamcenter_y = best_y_fine
scan_id=scan[pol]["scan_id"]
sample_name=scan[pol]["sample_name"]
sample_base_description=scan[pol]["base_description"]

_, _, _ = plot_beam_center_fit(data, pol, manual_mask, beamcenter_x, beamcenter_y, scan_id, sample_name, sample_base_description, chi_halfwidth=45, threshold=1, energy_val="all", energy_idx_to_plot=0, q_target=0.02201, q_window_width=0.003, 
                         best_x_coarse=best_x_coarse, best_y_coarse=best_y_coarse, search_step_coarse=1, cost_grid_coarse=cost_grid_coarse, best_x_fine=best_x_fine, best_y_fine=best_y_fine, cost_grid_fine=cost_grid_fine, search_step_fine=0.1,
                         mask_reflection_mode="fourfold", apply_postmask=True, verbose=True, show_plots=True)

print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")

If all looks good: restart the kernel, re-run the notebook right up to beamcentering, then skip previous pols and do the next.

### pol=45 beam centering

In [ ]:
pol = 45

beamcenter_x_guess = 472
beamcenter_y_guess = 515
n_jobs = 32
data=scan[pol]["masked_data_final"]
#data = scan[pol]["data"].isel(system=slice(14, 15))  #to do 1 energy at a time. keeps the system dim, just with 1 energy at index (first index listed)
#data = scan[pol]["data"].isel(system=slice(34, 67)) #slice(34, 67) covers energy=283.8 to 287
scan_id=scan[pol]["scan_id"]
sample_name=scan[pol]["sample_name"]
sample_base_description=scan[pol]["base_description"]
with parallel_backend("threading"):
    with SuppressStderr():
        best_x_coarse, best_y_coarse, best_x_fine, best_y_fine, threshold, remeshed, clean, mask_reflection, clean_with_symmetry, cost_grid_coarse, cost_grid_fine, = find_best_beam_center(
            data=data, pol=pol, manual_mask=final_masks[0], beamcenter_x_guess=beamcenter_x_guess, beamcenter_y_guess=beamcenter_y_guess, 
            scan_id=scan_id, sample_name=sample_name, sample_base_description=sample_base_description,
            search_radius_coarse=4, search_step_coarse=1, search_radius_fine=0.8, search_step_fine=0.1, n_jobs=n_jobs, chi_halfwidth=45, threshold=1, 
            energy_val=284.8, energy_idx_to_plot=0,  q_target=0.02201, q_window_width=0.003, #energy_val can be a number or "all"
            mask_reflection_mode="fourfold", apply_postmask=True, #mask reflection mode can be None, "twofold", "fourfold", or "eightfold"
            verbose=True, show_plots=True)

#add to dict
scan[pol]["beamcenter_x_fit_coarse"] = best_x_coarse
scan[pol]["beamcenter_y_fit_coarse"] = best_y_coarse
scan[pol]["beamcenter_x_fit_fine"] = best_x_fine
scan[pol]["beamcenter_y_fit_fine"] = best_y_fine
scan[pol]["threshold"] = threshold
scan[pol]["remeshed"] = remeshed
scan[pol]["clean"] = clean
scan[pol]["mask_reflection"] = mask_reflection
scan[pol]["clean_with_symmetry"] = clean_with_symmetry
scan[pol]["cost_grid_coarse"] = cost_grid_coarse
scan[pol]["cost_grid_fine"] = cost_grid_fine

print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")


In [ ]:
print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")

In [ ]:
save_dir = "/homes/ksr2/rsoxs_workup/sst1_2026apr/beamcenter_pickles"
bc_data = {key: scan[pol][key] for key in ["beamcenter_x_fit_coarse", "beamcenter_y_fit_coarse", "beamcenter_x_fit_fine", "beamcenter_y_fit_fine",
                                             "threshold", "remeshed", "clean", "mask_reflection", "clean_with_symmetry",
                                             "cost_grid_coarse", "cost_grid_fine"]}

filename_bc = f"beamcenter_{scan[pol]['scan_id']}_{scan[pol]['mask_reflection']}_p{workup_pass}.pkl"
filepath_bc = os.path.join(save_dir, filename_bc)

print("starting beam center save")
t0 = time.time()
with open(filepath_bc, "wb") as f:
    pickle.dump(bc_data, f)
print(f"finished beam center save in {time.time() - t0:.1f} s")

In [ ]:
data=scan[pol]["data"]
pol=45
manual_mask = final_masks[0]#static_mask
beamcenter_x = best_x_fine
beamcenter_y = best_y_fine
scan_id=scan[pol]["scan_id"]
sample_name=scan[pol]["sample_name"]
sample_base_description=scan[pol]["base_description"]

_, _, _ = plot_beam_center_fit(data, pol, manual_mask, beamcenter_x, beamcenter_y, scan_id, sample_name, sample_base_description, chi_halfwidth=45, threshold=1, energy_val="all", energy_idx_to_plot=0, q_target=0.02201, q_window_width=0.003, 
                         best_x_coarse=best_x_coarse, best_y_coarse=best_y_coarse, search_step_coarse=1, cost_grid_coarse=cost_grid_coarse, best_x_fine=best_x_fine, best_y_fine=best_y_fine, cost_grid_fine=cost_grid_fine, search_step_fine=0.1,
                         mask_reflection_mode="fourfold", apply_postmask=True, verbose=True, show_plots=True)

print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")

If all looks good: restart the kernel, re-run the notebook right up to beamcentering, then skip previous pols and do the next.

### pol=135 beam centering

In [ ]:
pol = 135

beamcenter_x_guess = 472
beamcenter_y_guess = 515
n_jobs = 32
data=scan[pol]["masked_data_final"]
#data = scan[pol]["data"].isel(system=slice(14, 15))  #to do 1 energy at a time. keeps the system dim, just with 1 energy at index (first index listed)
#data = scan[pol]["data"].isel(system=slice(34, 67)) #slice(34, 67) covers energy=283.8 to 287
scan_id=scan[pol]["scan_id"]
sample_name=scan[pol]["sample_name"]
sample_base_description=scan[pol]["base_description"]
with parallel_backend("threading"):
    with SuppressStderr():
        best_x_coarse, best_y_coarse, best_x_fine, best_y_fine, threshold, remeshed, clean, mask_reflection, clean_with_symmetry, cost_grid_coarse, cost_grid_fine, = find_best_beam_center(
            data=data, pol=pol, manual_mask=final_masks[0], beamcenter_x_guess=beamcenter_x_guess, beamcenter_y_guess=beamcenter_y_guess, 
            scan_id=scan_id, sample_name=sample_name, sample_base_description=sample_base_description,
            search_radius_coarse=4, search_step_coarse=1, search_radius_fine=0.8, search_step_fine=0.1, n_jobs=n_jobs, chi_halfwidth=45, threshold=1, 
            energy_val=284.8, energy_idx_to_plot=0,  q_target=0.02201, q_window_width=0.003, #energy_val can be a number or "all"
            mask_reflection_mode="fourfold", apply_postmask=True, #mask reflection mode can be None, "twofold", "fourfold", or "eightfold"
            verbose=True, show_plots=True)

#add to dict
scan[pol]["beamcenter_x_fit_coarse"] = best_x_coarse
scan[pol]["beamcenter_y_fit_coarse"] = best_y_coarse
scan[pol]["beamcenter_x_fit_fine"] = best_x_fine
scan[pol]["beamcenter_y_fit_fine"] = best_y_fine
scan[pol]["threshold"] = threshold
scan[pol]["remeshed"] = remeshed
scan[pol]["clean"] = clean
scan[pol]["mask_reflection"] = mask_reflection
scan[pol]["clean_with_symmetry"] = clean_with_symmetry
scan[pol]["cost_grid_coarse"] = cost_grid_coarse
scan[pol]["cost_grid_fine"] = cost_grid_fine

print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")


In [ ]:
print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")

In [ ]:
save_dir = "/homes/ksr2/rsoxs_workup/sst1_2026apr/beamcenter_pickles"
bc_data = {key: scan[pol][key] for key in ["beamcenter_x_fit_coarse", "beamcenter_y_fit_coarse", "beamcenter_x_fit_fine", "beamcenter_y_fit_fine",
                                             "threshold", "remeshed", "clean", "mask_reflection", "clean_with_symmetry",
                                             "cost_grid_coarse", "cost_grid_fine"]}

filename_bc = f"beamcenter_{scan[pol]['scan_id']}_{scan[pol]['mask_reflection']}_p{workup_pass}.pkl"
filepath_bc = os.path.join(save_dir, filename_bc)

print("starting beam center save")
t0 = time.time()
with open(filepath_bc, "wb") as f:
    pickle.dump(bc_data, f)
print(f"finished beam center save in {time.time() - t0:.1f} s")

In [ ]:
data=scan[pol]["data"]
pol=135
manual_mask = final_masks[0]#static_mask
beamcenter_x = best_x_fine
beamcenter_y = best_y_fine
scan_id=scan[pol]["scan_id"]
sample_name=scan[pol]["sample_name"]
sample_base_description=scan[pol]["base_description"]

_, _, _ = plot_beam_center_fit(data, pol, manual_mask, beamcenter_x, beamcenter_y, scan_id, sample_name, sample_base_description, chi_halfwidth=45, threshold=1, energy_val="all", energy_idx_to_plot=0, q_target=0.02201, q_window_width=0.003, 
                         best_x_coarse=best_x_coarse, best_y_coarse=best_y_coarse, search_step_coarse=1, cost_grid_coarse=cost_grid_coarse, best_x_fine=best_x_fine, best_y_fine=best_y_fine, cost_grid_fine=cost_grid_fine, search_step_fine=0.1,
                         mask_reflection_mode="fourfold", apply_postmask=True, verbose=True, show_plots=True)

print(f"Best center (coarse search) (x,y): ({best_x_coarse}, {best_y_coarse})")
print(f"Best center (fine search) (x,y): ({best_x_fine}, {best_y_fine})")

If all looks good: restart the kernel, re-run the notebook right up to beamcentering, then skip beamcentering calcs and move onto analysis.

## Integration

In [ ]:
#with saturation mask:
    #pol0: 473.1, 514.7
    #pol90: 473.1, 514.8
    #pol45:  473.0, 514.7
    #pol135:  473.0, 514.7

In [ ]:
def wrap_chi(vals):
    return ((np.asarray(vals) + 180) % 360) - 180

def get_chi_slice(data, chi_center, chi_halfwidth): #lifted directly from pyhyperscattering
    """
    Return a slice centered at `chi_center` ± chi_halfwidth, handling wrapping manually.
    Assumes chi ranges from ~-180 to 180.
    """
    chi_vals = data['chi'].values
    chi_min = chi_center - chi_halfwidth
    chi_max = chi_center + chi_halfwidth

    # Wrap chi into [-180, 180)
    chi_vals_wrapped = ((chi_vals + 180) % 360) - 180
    chi_min_wrapped = ((chi_min + 180) % 360) - 180
    chi_max_wrapped = ((chi_max + 180) % 360) - 180

    if chi_min_wrapped <= chi_max_wrapped:
        mask = (chi_vals_wrapped >= chi_min_wrapped) & (chi_vals_wrapped <= chi_max_wrapped)
    else:
        mask = (chi_vals_wrapped >= chi_min_wrapped) | (chi_vals_wrapped <= chi_max_wrapped)

    return data.sel(chi=mask)

def integrate_with_energy_masks(scan_at_pol, pol):

    loaded_scan = scan_at_pol["data"].unstack("system")
    integrator = phs.integrate.PFEnergySeriesIntegrator(geomethod="template_xr", template_xr=loaded_scan, integration_method="csr")
    integrator.ni_beamcenter_x = scan_at_pol["beamcenter_x"]
    integrator.ni_beamcenter_y = scan_at_pol["beamcenter_y"]

    remeshed_list = []
    remeshed_mask_list = []
    remeshed_ones_list = []

    for energy_idx in range(loaded_scan.sizes["energy"]):

        print(f"Integrating energy index {energy_idx}")

        mask = scan_at_pol["final_masks"][energy_idx]

        #INTEGRATE DATA--------------------------------------------------------------------------------

        integrator.mask = mask #keep full energy stack, but apply current energy mask
        remeshed = integrator.integrateImageStack(loaded_scan)
        remeshed_list.append(remeshed.isel(energy=energy_idx)) #select only this energy after integration

        #INTEGRATE MASK-------------------------------------------------------------------------------

        loaded_mask_scan = xarray.DataArray(np.repeat(~mask[:, :, np.newaxis], loaded_scan.sizes["energy"], axis=2), 
            dims=loaded_scan.dims, coords={dim: loaded_scan[dim] for dim in loaded_scan.dims})

        integrator.mask = None
        remeshed_mask = integrator.integrateImageStack(loaded_mask_scan)
        remeshed_mask_list.append(remeshed_mask.isel(energy=energy_idx))

        #INTEGRATE ONES------------------------------------------------------------------------------------

        ones_input = xarray.full_like(loaded_mask_scan, 1) #Make an" all ones" image with the same shape as input
        remeshed_ones = integrator.integrateImageStack(ones_input)
        remeshed_ones_list.append(remeshed_ones.isel(energy=energy_idx))


    #COMBINE ENERGIES--------------------------------------------------------------------------------

    scan_at_pol["remeshed"] = xarray.concat(remeshed_list, dim="energy")
    scan_at_pol["remeshed_mask"] = xarray.concat(remeshed_mask_list, dim="energy")
    scan_at_pol["remeshed_ones"] = xarray.concat(remeshed_ones_list, dim="energy")

    #for debugging/checking:
    #print(f"loaded (unstacked) scan shape:\n{loaded_scan.shape}\n(pix_y, pix_x, energy)")
    #print(f"\nintegrated scan shape:\n{scan_at_pol['remeshed'].shape}\n(energy, chi, q)")
    #print(f"\nintegrated mask shape:\n{scan_at_pol['remeshed_mask'].shape}\n{scan_at_pol['remeshed_mask'].dims}")
    #print(f"\nintegrated ones shape:\n{scan_at_pol['remeshed_ones'].shape}")
    
    #APPLY POSTMASK--------------------------------------------------------------------------------------

    unmasked_pixel_fraction = scan_at_pol["remeshed_mask"] / scan_at_pol["remeshed_ones"]
    masked_pixels = (unmasked_pixel_fraction < scan_at_pol["threshold"]) | np.isnan(unmasked_pixel_fraction) #boolean mask where True = pixel_fraction is below threshold

    scan_at_pol["clean_uncapped_q"] = scan_at_pol["remeshed"].copy() #make a copy so don't overwrite the original
    scan_at_pol["clean_uncapped_q"].values[masked_pixels.values] = np.nan #apply mask: wherever masked_pixels is True, set to NaN

    #APPLY QCAP--------------------------------------------------------------------------------------

    q_cap_mask = scan_at_pol["clean_uncapped_q"]["q"] <= scan_at_pol["q_cap"]
    scan_at_pol["clean"] = scan_at_pol["clean_uncapped_q"].where(q_cap_mask)
    
    #CHECKING-----------------------------------------------------------------------------------------
      
    for energy_idx in range(scan_at_pol["remeshed"].sizes["energy"]):

        energy_val = scan_at_pol["remeshed"].energy.values[energy_idx]
        q_cap_val = scan_at_pol["q_cap"].sel(energy=energy_val, method="nearest").item()

        fig, axs = plt.subplots(2,3,figsize=(12,8))

        datasets = [scan_at_pol["remeshed"], scan_at_pol["remeshed_mask"], scan_at_pol["remeshed_ones"], unmasked_pixel_fraction, scan_at_pol["clean_uncapped_q"], scan_at_pol["clean"]]
        titles = ["Integrated Data\n(with built-in PyHyper masking)", "Integrated Mask", "Integrated Ones", "Normalized Fraction\nof Unmasked Source Pixels", "Clean Integrated Data\n(No q Cap)", f"Clean Integrated Data\nq ≤ {q_cap_val:.4f} Å⁻¹",]
        cbar_labels = ["Intensity", "Source pixel count for each integrated pixel",  "Source pixel count for each integrated pixel", "Fraction of eligible source pixels\ncontributing to each integrated pixel", "Intensity", "Intensity"]

        qvals = [0.01, 0.04, 0.11]
        chivals = [-50, 50, -100]
        colors = ["red", "cyan", "lawngreen"]

        for ax, ds, title, cbar_label in zip(axs.flat, datasets, titles, cbar_labels):

            if "energy" in ds.dims:
                data = ds.isel(energy=energy_idx)
            else:
                data = ds

            im = ax.pcolormesh(data["q"], data["chi"], data, shading="auto")
            ax.set_title(title)
            ax.set_xlabel("q")
            ax.set_ylabel("chi")

            for qv, chv, c in zip(qvals, chivals, colors):
                intensity = data.sel(q=qv, chi=chv, method="nearest").values
                ax.plot(qv, chv, marker="o", color=c, markersize=8, label=f"{intensity:.2f}")

            ax.legend(loc="upper right", fontsize=10)
            cbar = fig.colorbar(im, ax=ax, shrink=0.5)
            cbar.set_label(cbar_label)

        fig.suptitle(f"Integration of Scan {scan_at_pol['scan_id']} (pol={pol}) using Beamcenter (x,y)=({scan_at_pol["beamcenter_x"]}, {scan_at_pol["beamcenter_y"]})\nEnergy = {energy_val:.1f}", fontsize=16)

        plt.tight_layout()
        plt.show()
        plt.close()

    return scan_at_pol["remeshed"], scan_at_pol["remeshed_mask"], scan_at_pol["remeshed_ones"], scan_at_pol["clean_uncapped_q"], scan_at_pol["clean"]

def integrate_darks_with_energy_masks(scan_at_pol, pol, high_ile):

    darks = scan_at_pol["darks"]
    data = scan_at_pol["data"]

    #expand dark image to match energy dimension
    if "time" in darks.dims:
        darks = darks.isel(time=0)

    loaded_scan = data.unstack("system")
    dark_stack = darks.expand_dims(energy=loaded_scan.energy)
    dark_stack = dark_stack.transpose("pix_y", "pix_x", "energy")

    integrator = phs.integrate.PFEnergySeriesIntegrator(geomethod="template_xr", template_xr=loaded_scan, integration_method="csr")
    integrator.ni_beamcenter_x = scan_at_pol["beamcenter_x"]
    integrator.ni_beamcenter_y = scan_at_pol["beamcenter_y"]

    remeshed_list = []
    remeshed_mask_list = []
    remeshed_ones_list = []

    for energy_idx in range(loaded_scan.sizes["energy"]):

        print(f"Integrating darks: energy index {energy_idx}")

        mask = scan_at_pol["final_masks"][energy_idx]

        #INTEGRATE DARKS---------------------------------------------------------------------------

        integrator.mask = mask
        darks_remeshed = integrator.integrateImageStack(dark_stack)
        remeshed_list.append(darks_remeshed.isel(energy=energy_idx))

        #INTEGRATE MASK----------------------------------------------------------------------------

        loaded_mask_scan = xarray.DataArray(np.repeat(~mask[:, :, np.newaxis], loaded_scan.sizes["energy"], axis=2),
            dims=loaded_scan.dims, coords={dim: loaded_scan[dim] for dim in loaded_scan.dims})

        integrator.mask = None
        darks_remeshed_mask = integrator.integrateImageStack(loaded_mask_scan)
        remeshed_mask_list.append(darks_remeshed_mask.isel(energy=energy_idx))

        #INTEGRATE ONES----------------------------------------------------------------------------

        ones_input = xarray.full_like(loaded_mask_scan, 1)
        darks_remeshed_ones = integrator.integrateImageStack(ones_input)
        remeshed_ones_list.append(darks_remeshed_ones.isel(energy=energy_idx))

    #COMBINE ENERGIES--------------------------------------------------------------------------

    scan_at_pol["darks_remeshed"] = xarray.concat(remeshed_list, dim="energy")
    scan_at_pol["darks_remeshed_mask"] = xarray.concat(remeshed_mask_list, dim="energy")
    scan_at_pol["darks_remeshed_ones"] = xarray.concat(remeshed_ones_list, dim="energy")

    #for checking/debugging
    # print(f"loaded dark stack shape:\n{dark_stack.shape}")
    # print(f"\nintegrated darks shape:\n{scan_at_pol['darks_remeshed'].shape}")
    # print(f"\nintegrated mask shape:\n{scan_at_pol['darks_remeshed_mask'].shape}")
    # print(f"\nintegrated ones shape:\n{scan_at_pol['darks_remeshed_ones'].shape}")

    # APPLY POSTMASK ----------------------------------------------------------------

    unmasked_pixel_fraction = scan_at_pol["darks_remeshed_mask"] / scan_at_pol["darks_remeshed_ones"]
    masked_pixels = (unmasked_pixel_fraction < scan_at_pol["threshold"]) | np.isnan(unmasked_pixel_fraction)

    scan_at_pol["darks_clean"] = scan_at_pol["darks_remeshed"].copy()
    scan_at_pol["darks_clean"].values[masked_pixels.values] = np.nan

    #CHECKING--------------------------------------------------------------------------------

    for energy_idx in range(scan_at_pol["darks_clean"].sizes["energy"]):

        fig, axs = plt.subplots(1, 2, figsize=(12, 6))

        original_dark_frame = dark_stack.isel(energy=energy_idx) #unintegrated dark for this energy
        integrated_dark_frame = scan_at_pol["darks_clean"].isel(energy=energy_idx) #integrated + postmasked dark

        vmin_original = np.nanpercentile(original_dark_frame, 0)
        vmax_original = np.nanpercentile(original_dark_frame, high_ile)

        im_original = axs[0].pcolormesh(original_dark_frame["pix_x"], original_dark_frame["pix_y"], original_dark_frame, shading="auto", vmin=vmin_original, vmax=vmax_original)
        im_integrated = axs[1].pcolormesh(integrated_dark_frame["q"], integrated_dark_frame["chi"], integrated_dark_frame, shading="auto")

        axs[0].set_title("Unintegrated Dark")
        axs[0].set_xlabel("pix_x")
        axs[0].set_ylabel("pix_y")
        axs[0].set_aspect("equal")

        axs[1].set_title("Integrated Dark (Postmask Applied)")
        axs[1].set_xlabel("q")
        axs[1].set_ylabel("chi")

        for qv, chv, c in [(0.02, -50, "red"), (0.005, 170, "cyan"), (0.08, -90, "lawngreen"), (0.12, -90, "goldenrod")]:
            intensity = integrated_dark_frame.sel(q=qv, chi=chv, method="nearest").values
            axs[1].plot(qv, chv, marker="o", color=c, markersize=8, label=f"{intensity:.2f}")

        axs[1].legend()
        fig.colorbar(im_original, ax=axs[0], shrink=0.7).set_label("Dark Counts")
        fig.colorbar(im_integrated, ax=axs[1], shrink=0.7).set_label("Dark Counts")
        energy_val = loaded_scan.energy.values[energy_idx]
        fig.suptitle(f"Dark Integration for Scan {scan_at_pol["scan_id"]} (pol={pol})\nBeamcenter ({scan_at_pol["beamcenter_x"]}, {scan_at_pol["beamcenter_y"]})   Energy={energy_val:.1f}", fontsize=16)

        plt.tight_layout()
        plt.show()
        plt.close()

    return scan_at_pol["darks_remeshed"], scan_at_pol["darks_remeshed_mask"], scan_at_pol["darks_remeshed_ones"], scan_at_pol["darks_clean"]

def calculate_q_cap(scan_at_pol):
    
    #calculate maximum accessible q (Å⁻¹) at each energy for a given detector radius, 
    #on the basis of where there is 360 degrees worth of info accessible in the chi direction.
    #outer_radius measured in pixels

    data = scan_at_pol["data"]
    outer_radius = scan_at_pol["outer_radius"]

    energies = np.array([e[0] for e in data.system.values])
    ny, nx = data.isel(system=0).shape

    #detector geometry
    pixel_size_A = scan_at_pol["pixel_size"] * 1e10   #m → Å
    D_A = scan_at_pol["sdd"] * 1e7                    #mm → Å

    beamcenter_x = scan_at_pol["beamcenter_x"]
    beamcenter_y = scan_at_pol["beamcenter_y"]

    x = (np.arange(nx) - beamcenter_x) * pixel_size_A
    y = (np.arange(ny) - beamcenter_y) * pixel_size_A
    xx, yy = np.meshgrid(x, y)

    radius_map = np.sqrt(xx**2 + yy**2)
    twotheta = np.arctan2(radius_map, D_A)

    radius_limit = outer_radius * pixel_size_A
    inside = radius_map <= radius_limit

    q_max = np.empty(len(energies))

    for i, energy in enumerate(energies):

        wavelength_A = (1239.841984 / energy) * 10  #eV → Å

        q_map = (4 * np.pi / wavelength_A) * np.sin(twotheta / 2)

        q_max[i] = q_map[inside].max()

    return xarray.DataArray(q_max, coords={"energy": energies}, dims=("energy",), name="q_max", attrs={"units": "Å^-1", "outer_radius_pixels": outer_radius})

In [ ]:
pol = 0

scan[pol]["beamcenter_x"] = 473.1
scan[pol]["beamcenter_y"] = 514.7
scan[pol]["threshold"] = 1

scan[pol]["outer_radius"] = outer_radius
scan[pol]["q_cap"] = calculate_q_cap(scan[pol])
scan[pol]["remeshed"], scan[pol]["remeshed_mask"], scan[pol]["remeshed_ones"], scan[pol]["clean_uncapped_q"], scan[pol]["clean"] = integrate_with_energy_masks(scan[pol], pol)
scan[pol]["darks_remeshed"], scan[pol]["darks_remeshed_mask"], scan[pol]["darks_remeshed_ones"], scan[pol]["darks_clean"], = integrate_darks_with_energy_masks(scan[pol], pol, high_ile) #high_ile only affects display here

In [ ]:
pol = 90

scan[pol]["beamcenter_x"] = 473.1
scan[pol]["beamcenter_y"] = 514.7
scan[pol]["threshold"] = 1

scan[pol]["outer_radius"] = outer_radius
scan[pol]["q_cap"] = calculate_q_cap(scan[pol])
scan[pol]["remeshed"], scan[pol]["remeshed_mask"], scan[pol]["remeshed_ones"], scan[pol]["clean_uncapped_q"], scan[pol]["clean"] = integrate_with_energy_masks(scan[pol], pol)
scan[pol]["darks_remeshed"], scan[pol]["darks_remeshed_mask"], scan[pol]["darks_remeshed_ones"], scan[pol]["darks_clean"], = integrate_darks_with_energy_masks(scan[pol], pol, high_ile) #high_ile only affects display here

In [ ]:
pol = 45

scan[pol]["beamcenter_x"] = 473.1
scan[pol]["beamcenter_y"] = 514.7
scan[pol]["threshold"] = 1

scan[pol]["outer_radius"] = outer_radius
scan[pol]["q_cap"] = calculate_q_cap(scan[pol])
scan[pol]["remeshed"], scan[pol]["remeshed_mask"], scan[pol]["remeshed_ones"], scan[pol]["clean_uncapped_q"], scan[pol]["clean"] = integrate_with_energy_masks(scan[pol], pol)
scan[pol]["darks_remeshed"], scan[pol]["darks_remeshed_mask"], scan[pol]["darks_remeshed_ones"], scan[pol]["darks_clean"], = integrate_darks_with_energy_masks(scan[pol], pol, high_ile) #high_ile only affects display here

In [ ]:
pol = 135

scan[pol]["beamcenter_x"] = 473.1
scan[pol]["beamcenter_y"] = 514.7
scan[pol]["threshold"] = 1

scan[pol]["outer_radius"] = outer_radius
scan[pol]["q_cap"] = calculate_q_cap(scan[pol])
scan[pol]["remeshed"], scan[pol]["remeshed_mask"], scan[pol]["remeshed_ones"], scan[pol]["clean_uncapped_q"], scan[pol]["clean"] = integrate_with_energy_masks(scan[pol], pol)
scan[pol]["darks_remeshed"], scan[pol]["darks_remeshed_mask"], scan[pol]["darks_remeshed_ones"], scan[pol]["darks_clean"], = integrate_darks_with_energy_masks(scan[pol], pol, high_ile) #high_ile only affects display here

# ANALYSIS

## Normalization

should talk this over with priyanka. also add notes justifying normalization in workups below (intensity vs q ; intensity vs chi)


In [ ]:
#main concern: are we correct in our normalization procedure? 
    #does it make sense to normalize each ring (q) by something different?
    #is the beamstop biasing the azimuthal mean? especially important to think about because beamstop will cover a different range for pol0/90, vs pol 45/135 


In [ ]:
for pol in scan:

    #since we're not automatically normalizing, we need a factor to normalize scans by
    scan[pol]["chi_avgd"] = scan[pol]["clean"].mean('chi')     # dims: (energy, q)

    #average across half of diffraction pattern spanning 90 to 270deg 
    #(which is the half with no beamstop and generally "better" detector panels)
    scan[pol]["hf_chi_avgd"] = (get_chi_slice(scan[pol]["clean"], 180, 90)).mean('chi')     # dims: (energy, q)

    #chi-average provides a reference intensity at each (energy, q)
    #normalization by this quantity removes overall scale differences between scans and preserves angular (chi-dependent) structure
    scan[pol]["chi_avgd_normed"] = scan[pol]["clean"]/scan[pol]["chi_avgd"]   # dims: (energy, chi, q)
    scan[pol]["hf_chi_avgd_normed"] = scan[pol]["clean"]/scan[pol]["hf_chi_avgd"]   # dims: (energy, chi, q)

print(f'scan[0]["clean"] dims:                  {scan[0]["clean"].dims}      expected: (energy, chi, q)')    #dims (energy, chi, q)
print(f'scan[0]["chi_avgd"] dims:               {scan[0]["chi_avgd"].dims}             expected: (energy, q)') #dims (energy, q)
print(f'scan[0]["hf_chi_avgd"] dims:            {scan[0]["hf_chi_avgd"].dims}             expected: (energy, q)') #dims (energy, q)
print(f'scan[0]["chi_avgd_normed"] dims:        {scan[0]["chi_avgd_normed"].dims}      expected: (energy, chi, q)') #dims (energy, chi, q)
print(f'scan[0]["hf_chi_avgd_normed"] dims:     {scan[0]["hf_chi_avgd_normed"].dims}      expected: (energy, chi, q)') #dims (energy, chi, q)

Testing that normalization makes sense:

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,4))

axs[0].plot(scan[0]["clean"].sel(energy=284.8, method="nearest").mean("chi").q.values, scan[0]["clean"].sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[0].plot(scan[90]["clean"].sel(energy=284.8, method="nearest").mean("chi").q.values, scan[90]["clean"].sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")
axs[1].plot(scan[0]["chi_avgd_normed"].sel(energy=284.8, method="nearest").mean("chi").q.values, scan[0]["chi_avgd_normed"].sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[1].plot(scan[90]["chi_avgd_normed"].sel(energy=284.8, method="nearest").mean("chi").q.values, scan[90]["chi_avgd_normed"].sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")

axs[0].set_yscale("log")
axs[0].set_ylabel("I")
axs[1].set_ylabel("I Ratio")
axs[1].set_ylim(-1,2)
axs[0].set_title("I.mean(chi) vs q")
axs[1].set_title("[I/I.mean(chi)].mean(chi) vs q \n(Ratio to the Mean)")
for ax in axs:
    ax.legend()
    ax.set_xlabel("q")

plt.tight_layout()
plt.show()
plt.close()

^Makes sense that once we've normalized something with respect to the chi-average, taking the chi average of that normalized thing comes out to 1. In other words, if the ratio I/(I.mean(chi)) describes how bright a given point is relaive to the azimuthal ring it lives on, then the average of that ratio all the way around the ring should be 1.

Below, let's continue examining this quantity describing how bright or dim a given point is relative to the average intensity on its azimuthal ring.

In [ ]:
test = (scan[0]["chi_avgd_normed"].sel(energy=284.8, method="nearest").mean("chi"))

print(test.min().item(), test.max().item())

In [ ]:
fig, axs = plt.subplots(1,3,figsize=(18,4))

axs[0].plot(get_chi_slice(scan[0]["chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[0]["chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[0].plot(get_chi_slice(scan[90]["chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[90]["chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")
axs[1].plot(get_chi_slice(scan[0]["chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[0]["chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[1].plot(get_chi_slice(scan[90]["chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[90]["chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")
#chi mean - 180 deg  slice
axs[2].plot(get_chi_slice(scan[0]["chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[0]["chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[2].plot(get_chi_slice(scan[90]["chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[90]["chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")

axs[0].set_title("[I.sel(chi=slice(135,-135)/I.mean(chi)].mean(chi) vs q\n(Dimensionless Ratio - 90 degree slice at 9oclock)")
axs[1].set_title("[I.sel(chi=slice(45,135)/I.mean(chi)].mean(chi) vs q\n(Dimensionless Ratio - 90 degree slice at 12oclock)")
axs[2].set_title("[I.sel(chi=slice(90,-90)/I.mean(chi)].mean(chi) vs q\n(Dimensionless Ratio - 180 degree slice at 9oclock)")

for ax in axs:
    ax.legend()
    ax.set_ylabel("Ratio of I/I.mean(chi)")
    ax.set_xlabel("q")
    
plt.tight_layout()
plt.show()
plt.close()

As we would expect, we see that the relative ratios invert between 9oclock and 12oclock, indicating the switching of the para and perp angles. Under the ideal of centrosymmetry, when we examine a 180deg slice, we are capturing the equivalent of a bright lobe and a dark lobe, so the pol=0 and pol=90 scans contain the same amount of information, and thus normalize to essentially the same shape. 

But now we need to think about the beamstop. It's snipping out some portion of our azimuthal ring, which is biasing the ring average Intensity. This is probably minimal for pol0 and pol90 pairs, since the beamstop sits at about -50 degrees. But for pol 45 and pol 135 pairs, it sits at a critical place such that the full azimuthal ring average is not an equal representation across each polarization. We will therefore consider the average intensity across half the ring, since half should cover one bright lobe and one dark lobe equivalent. It shouldn't super matter what slice we pick so long as it doesn't overlap the beamstop and is 180deg. To keep things consistent, we'll define this half chi normalization as that normalized by the average of the azimuthal ring spanning 90 to 270.

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,4))

axs[0].plot(get_chi_slice(scan[0]["chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[0]["chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0, chi normed")
axs[0].plot(get_chi_slice(scan[90]["chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[90]["chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90, chi normed")
axs[0].plot(get_chi_slice(scan[0]["hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[0]["hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0, half chi normed")
axs[0].plot(get_chi_slice(scan[90]["hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[90]["hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90, half chi normed")

axs[1].plot(get_chi_slice(scan[0]["chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[0]["chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0, chi normed")
axs[1].plot(get_chi_slice(scan[90]["chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[90]["chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90, chi normed")
axs[1].plot(get_chi_slice(scan[0]["hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[0]["hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0, half chi normed")
axs[1].plot(get_chi_slice(scan[90]["hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[90]["hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90, half chi normed")

axs[0].set_title("[I.sel(chi=slice(135,225)/I.mean(halfchi)].mean(chi) vs q \n(Dimensionless Ratio - 90 degree slice at 9oclock)")
axs[1].set_title("[I.sel(chi=slice(45,135)/I.mean(halfchi)].mean(chi) vs q \n(Dimensionless Ratio - 90 degree slice at 12oclock)")

for ax in axs:
    ax.legend()
    ax.set_ylabel("Ratio of I/I.mean(chi)")
    ax.set_xlabel("q")
    
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
ratio = (scan[0]["hf_chi_avgd"] / scan[0]["chi_avgd"])

fig, axs = plt.subplots(1,1,figsize=(4,3))

axs.plot(ratio.sel(energy=284.8, method="nearest"))

axs.set_xlabel("q")
axs.set_ylabel("I (hf chi avgd) / I (chi avgd)")

plt.show()
plt.close()

#this could be a problem. i don't think we should be seeing q dependent oscillations if ratios are truly equivalent
#it's minor though

#but wait, they're not equivalent. that's why we went ot the trouble of using half chi
#beamstop mask changes with q.
#so at small q, beamstop intercepts large angular sector of azimuthal ring. vice versa at big q. 
#so q-dependent skewing. so the whole chi average will have some q dependent skewing.
#so the ratio of hf_chi_avgd to chi_avgd will necessarily also have skewing

In [ ]:
#could do some simulations to support this interpretation.

We'll also want to take into account how beamline flux varies with energy, and thus determines the intensity strength of the readout. We'll normalize by upstream flux using I0 from the au mesh

In [ ]:
for pol in scan: 
    print(scan[pol]["au_mesh_current"])

In [ ]:
print("Ratio of au mesh current at 284.8eV vs at 286 eV")
for pol in scan:
    energy_ratio = scan[pol]["au_mesh_current"][0]/scan[pol]["au_mesh_current"][1]
    print(f"    pol{pol}: {energy_ratio}")

print("\nRatios of pol0/pol90")
print(f"284.8 eV: {scan[0]["au_mesh_current"][0]/scan[90]["au_mesh_current"][0]}")
print(f"286 eV: {scan[0]["au_mesh_current"][1]/scan[90]["au_mesh_current"][1]}")

print("\nRatios of pol45/pol135")
print(f"284.8 eV: {scan[45]["au_mesh_current"][0]/scan[135]["au_mesh_current"][0]}")
print(f"286 eV: {scan[45]["au_mesh_current"][1]/scan[135]["au_mesh_current"][1]}")

Energy dependence of incident flux is highly reproducible across all polarization measurements.

Incident flux is essentialy independent of polarization (is mesh current measured upstream of polarization spot?)

Key assumption: au mesh current is proportional to incident photon flux, F(E) = k * I_mesh (E)

In [ ]:
for pol in scan:
    scan[pol]["flux"] = xarray.DataArray(scan[pol]["au_mesh_current"], coords={"energy": scan[pol]["clean"].energy}, dims="energy")
    scan[pol]["flux_corrected"] = scan[pol]["clean"]/scan[pol]["flux"]   # dims: (energy, chi q)

#check that ordering of energies matches
print(f"energy values: {scan[0]["clean"].energy.values}")
print(f"flux values: {scan[0]["au_mesh_current"]}")
print(f"flux: \n{scan[0]["flux"]}")

print(f'\nscan[0]["clean"] dims:                  {scan[0]["clean"].dims}      expected: (energy, chi, q)')    #dims (energy, chi, q)
print(f'scan[0]["flux_corrected"] dims:         {scan[0]["flux_corrected"].dims}      expected: (energy, chi, q)') #dims (energy, chi, q)

In [ ]:
print(scan[0]["flux_corrected"].mean(["chi","q"]))
print(scan[90]["flux_corrected"].mean(["chi","q"]))

Appears to be some polarization-dependent throughout, which would match observations at the beamline

In [ ]:
#analogous chi averaging for flux corrected data

for pol in scan:
    scan[pol]["flxcorr_chi_avgd"] = scan[pol]["flux_corrected"].mean('chi')     # dims: (energy, q)
    scan[pol]["flxcorr_hf_chi_avgd"] = (get_chi_slice(scan[pol]["flux_corrected"], 180, 90)).mean('chi')     # dims: (energy, q)
    scan[pol]["flxcorr_chi_avgd_normed"] = scan[pol]["flux_corrected"]/scan[pol]["flxcorr_chi_avgd"]   # dims: (energy, chi, q)
    scan[pol]["flxcorr_hf_chi_avgd_normed"] = scan[pol]["flux_corrected"]/scan[pol]["flxcorr_hf_chi_avgd"]   # dims: (energy, chi, q)


print(f'scan[0]["flux_corrected"] dims:                 {scan[0]["flux_corrected"].dims}      expected: (energy, chi, q)')    #dims (energy, chi, q)
print(f'scan[0]["flxcorr_chi_avgd"] dims:               {scan[0]["flxcorr_chi_avgd"].dims}             expected: (energy, q)') #dims (energy, q)
print(f'scan[0]["flxcorr_hf_chi_avgd"] dims:            {scan[0]["flxcorr_hf_chi_avgd"].dims}             expected: (energy, q)') #dims (energy, q)
print(f'scan[0]["flxcorr_chi_avgd_normed"] dims:        {scan[0]["flxcorr_chi_avgd_normed"].dims}      expected: (energy, chi, q)') #dims (energy, chi, q)
print(f'scan[0]["flxcorr_hf_chi_avgd_normed"] dims:     {scan[0]["flxcorr_hf_chi_avgd_normed"].dims}      expected: (energy, chi, q)') #dims (energy, chi, q)

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,4))

axs[0].plot(scan[0]["flux_corrected"].sel(energy=284.8, method="nearest").mean("chi").q.values, scan[0]["flux_corrected"].sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[0].plot(scan[90]["flux_corrected"].sel(energy=284.8, method="nearest").mean("chi").q.values, scan[90]["flux_corrected"].sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")
axs[1].plot(scan[0]["flxcorr_chi_avgd_normed"].sel(energy=284.8, method="nearest").mean("chi").q.values, scan[0]["flxcorr_chi_avgd_normed"].sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[1].plot(scan[90]["flxcorr_chi_avgd_normed"].sel(energy=284.8, method="nearest").mean("chi").q.values, scan[90]["flxcorr_chi_avgd_normed"].sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")

axs[0].set_yscale("log")
axs[0].set_ylabel("Flux-Corrected I")
axs[1].set_ylabel("Flux-Corrected I Ratio")
axs[1].set_ylim(-1,2)
axs[0].set_title("Flux-Corrected I.mean(chi) vs q")
axs[1].set_title("[Flux-Corrected I / Flux-Corrected I.mean(chi)].mean(chi) vs q \n(Ratio to the Mean)")
for ax in axs:
    ax.legend()
    ax.set_xlabel("q")

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
fig, axs = plt.subplots(1,3,figsize=(18,4))

axs[0].plot(get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[0].plot(get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")
axs[1].plot(get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[1].plot(get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")
#chi mean - 180 deg  slice
axs[2].plot(get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[2].plot(get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")

axs[0].set_title("[Flux-Corrected I.sel(chi=slice(135,-135) / \nFlux-Corrected I.mean(chi)].mean(chi) vs q\n(Dimensionless Ratio - 90 degree slice at 9oclock)")
axs[1].set_title("[Flux-Corrected I.sel(chi=slice(45,135) / \nFlux-Corrected I.mean(chi)].mean(chi) vs q\n(Dimensionless Ratio - 90 degree slice at 12oclock)")
axs[2].set_title("[Flux-Corrected I.sel(chi=slice(90,-90) / \nFlux-Corrected I.mean(chi)].mean(chi) vs q\n(Dimensionless Ratio - 180 degree slice at 9oclock)")

for ax in axs:
    ax.legend()
    ax.set_ylabel("Ratio of Flux-Corrected I / Flux-Corrected I.mean(chi)")
    ax.set_xlabel("q")
    
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
fig, axs = plt.subplots(1,3,figsize=(18,4))

axs[0].plot(get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[0].plot(get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")
axs[1].plot(get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[1].plot(get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")
#chi mean - 180 deg  slice
axs[2].plot(get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0")
axs[2].plot(get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").q.values, 
                get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 180, 90).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90")

axs[0].set_title("[Flux-Corrected I.sel(chi=slice(135,-135) / \nFlux-Corrected I.mean(chi)].mean(chi) vs q\n(Dimensionless Ratio - 90 degree slice at 9oclock)")
axs[1].set_title("[Flux-Corrected I.sel(chi=slice(45,135) / \nFlux-Corrected I.mean(chi)].mean(chi) vs q\n(Dimensionless Ratio - 90 degree slice at 12oclock)")
axs[2].set_title("[Flux-Corrected I.sel(chi=slice(90,-90) / \nFlux-Corrected I.mean(chi)].mean(chi) vs q\n(Dimensionless Ratio - 180 degree slice at 9oclock)")

axs[2].set_ylim(-1,2)
for ax in axs:
    ax.legend()
    ax.set_ylabel("Ratio of Flux-Corrected I / Flux-Corrected I.mean(chi)")
    ax.set_xlabel("q")
    
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,4))

axs[0].plot(get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0, chi normed")
axs[0].plot(get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90, chi normed")
axs[0].plot(get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0, half chi normed")
axs[0].plot(get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 180, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90, half chi normed")

axs[1].plot(get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[0]["flxcorr_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0, chi normed")
axs[1].plot(get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[90]["flxcorr_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90, chi normed")
axs[1].plot(get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[0]["flxcorr_hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=0, half chi normed")
axs[1].plot(get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").q.values, get_chi_slice(scan[90]["flxcorr_hf_chi_avgd_normed"], 90, 45).sel(energy=284.8, method="nearest").mean("chi").values, label="pol=90, half chi normed")

axs[0].set_title("[Flux-Corrected I.sel(chi=slice(135,225) / \nFlux-Corrected I.mean(halfchi)].mean(chi) vs q \n(Dimensionless Ratio - 90 degree slice at 9oclock)")
axs[1].set_title("[Flux-Corrected I.sel(chi=slice(45,135) / \nFlux-Corrected I.mean(halfchi)].mean(chi) vs q \n(Dimensionless Ratio - 90 degree slice at 12oclock)")

for ax in axs:
    ax.legend()
    ax.set_ylabel("Ratio of Flux-Corrected I / Flux-Corrected I.mean(chi)")
    ax.set_xlabel("q")
    
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
scan[0]["flxcorr_hf_chi_avgd"] = (get_chi_slice(scan[0]["flux_corrected"], 180, 90)).mean('chi')
scan[90]["flxcorr_hf_chi_avgd"] = (get_chi_slice(scan[0]["flux_corrected"], 180, 90)).mean('chi')
scan[45]["flxcorr_hf_chi_avgd"] = (get_chi_slice(scan[0]["flux_corrected"], 180, 90)).mean('chi')
scan[135]["flxcorr_hf_chi_avgd"] = (get_chi_slice(scan[0]["flux_corrected"], 180, 90)).mean('chi')

scan[0]["flxcorr_hfchi_avgd_normed"] = scan[0]["flux_corrected"]/scan[0]["flxcorr_hf_chi_avgd"]   
scan[90]["flxcorr_hfchi_avgd_normed"] = scan[90]["flux_corrected"]/scan[90]["flxcorr_hf_chi_avgd"]
scan[45]["flxcorr_hfchi_avgd_normed"] = scan[45]["flux_corrected"]/scan[45]["flxcorr_hf_chi_avgd"]
scan[135]["flxcorr_hfchi_avgd_normed"] = scan[135]["flux_corrected"]/scan[135]["flxcorr_hf_chi_avgd"]

In [ ]:
#might want to come back later and think about how flux correction might be able to offer another way to anisotropy...

## Intensity

In [ ]:
pols = [0, 90, 45, 135]

plot_specs = [{"data": {p: scan[p]["clean"] for p in pols}, "title": f'{scan[0]["sample_name"]}: {scan[0]["base_description"]}\n(Note: No normalization performed across or between scans)'},
              {"data": {p: scan[p]["flux_corrected"] for p in pols}, "title": f'{scan[0]["sample_name"]}: {scan[0]["base_description"]}\n(Note: Flux corrected)'}]

for spec in plot_specs:

    vals = np.concatenate([spec["data"][p].mean("chi").values.flatten() for p in pols])
    vals = vals[np.isfinite(vals) & (vals > 0)]

    spec["vmin"] = 1e1 #np.percentile(vals, 1)
    spec["vmax"] = np.percentile(vals, 100)

    fig, axs = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)

    for ax, pol in zip(axs, pols):

        intensity = spec["data"][pol].mean("chi")

        im = intensity.plot(ax=ax, norm=LogNorm(vmin=spec["vmin"], vmax=spec["vmax"]), cmap="afmhot", add_colorbar=False)

        ax.set_xlabel(r"q $(\AA^{-1})$")
        ax.set_ylabel("energy (eV)")
        ax.set_title(f'Scan {scan[pol]["scan_id"]}: Polarization = {pol}°')

    cbar = fig.colorbar(im, ax=axs, location="right", shrink=0.4, label="Intensity")

    fig.suptitle(spec["title"], fontsize=16)

    print(f'vmin={spec["vmin"]:.3g}, vmax={spec["vmax"]:.3g}')

    plt.show()
    plt.close()

In [ ]:
#the x values in the function are placeholder values 
#the x-values used for vertical lines are defined as arguments at the bottom of this cell
#this allows easy editing near the plot & quick visual feedback during adjustments
def plot_i_vs_energy(scan, q_min, q_max, x1=284.8, x2=286, pol=0, flux_corrected=False):
    fig = plt.figure(figsize=(16, 4))

    if flux_corrected:
        data = scan["flux_corrected"]
        norm_label = "Flux corrected using Au mesh current"
    else:
        data = scan["clean"]
        norm_label = "No flux correction"
    intensity_vs_energy = (data.sel(q=slice(q_min, q_max)).mean("q").mean("chi"))
    intensity_vs_energy.plot(color="dodgerblue", label=scan["sample_name"])
    
    line_ymax = 1.1 * float(intensity_vs_energy.max())
    line_ymin = 0

    #keeping these separate in case I ever want to change color, line thickness/style, etc
    plt.vlines(x=x1, ymin=line_ymin, ymax=line_ymax, color='white', linestyle='--', linewidth=0.5, label=f'{x1} eV')
    plt.vlines(x=x2, ymin=line_ymin, ymax=line_ymax, color='white', linestyle='--', linewidth=0.5, label=f'{x2} eV')

    plt.xlim(240,360)
    plt.yscale("log") 
    plt.xlabel("Energy (eV)"); plt.ylabel("Average Intensity"); plt.legend(loc="upper right") 
    
    if flux_corrected:
        plt.title(f'Scan{scan["scan_id"]}: {scan["sample_name"]} at Polarization={pol}° \n{scan["base_description"]} \n(Flux-Corrected)')
    else:
        plt.title(f'Scan{scan["scan_id"]}: {scan["sample_name"]} at Polarization={pol}° \n{scan["base_description"]} \n(Note: No normalization performed across or between scans)')
    plt.show()
    plt.close()

In [ ]:
#Intensity vs. energy for pol=0 
#I don't find this plot particularly useful, but am including it in case you do.
#It gives the average intensity at each energy, as evaluated over a given q span. You may wish to adjust this q.

#might want to change these
q_min = 0.02
q_max = 0.08

x1=284.8
x2=286

#rather than adjusting the values in the function, can adjust them here for easy editing near plot and quick visual feedback
plot_i_vs_energy(scan[0], q_min, q_max, x1=x1, x2=x2, pol=0, flux_corrected=False)
plot_i_vs_energy(scan[90], q_min, q_max, x1=x1, x2=x2, pol=90, flux_corrected=False)
plot_i_vs_energy(scan[45], q_min, q_max, x1=x1, x2=x2, pol=45, flux_corrected=False)
plot_i_vs_energy(scan[135], q_min, q_max, x1=x1, x2=x2, pol=135, flux_corrected=False)

#remember you might want to adjust q range if plots are weird at high q

In [ ]:
#Intensity vs. energy for pol=0 
#I don't find this plot particularly useful, but am including it in case you do.
#It gives the average intensity at each energy, as evaluated over a given q span. You may wish to adjust this q.

#might want to change these
q_min = 0.02
q_max = 0.08

x1=284.8
x2=286

#rather than adjusting the values in the function, can adjust them here for easy editing near plot and quick visual feedback
plot_i_vs_energy(scan[0], q_min, q_max, x1=x1, x2=x2, pol=0, flux_corrected=True)
plot_i_vs_energy(scan[90], q_min, q_max, x1=x1, x2=x2, pol=90, flux_corrected=True)
plot_i_vs_energy(scan[45], q_min, q_max, x1=x1, x2=x2, pol=45, flux_corrected=True)
plot_i_vs_energy(scan[135], q_min, q_max, x1=x1, x2=x2, pol=135, flux_corrected=True)

#remember you might want to adjust q range if plots are weird at high q

In [ ]:
#energies listed in indices don't match those in the actual data, so let's find out what real energy vals are
#I suspect this will be fixed in the future and this step won't be necessary.
#(can't use .sel with method='nearest', presumably because that looks for labels, not values, and our energy coords don't have perfectly clean float labels.)

x1=284.8
x2=286

specified_energies = [x1, x2]
available_energies = scan[0]["clean"].coords["energy"].values

closest_energies = [float(min(available_energies, key=lambda x: abs(x - energy))) for energy in specified_energies]

print("Specified energies:", specified_energies)
print("Closest available energies:", closest_energies)

In [ ]:
# plot of intensity vs. q for various energy traces of interest

pols = [0, 90, 45, 135]

plot_specs = [
    {
        "scans": {0: scan[0]["clean"], 90: scan[90]["clean"], 45: scan[45]["clean"], 135: scan[135]["clean"]},
        "ylim": (1e2, 1e5),
        "title": f"{scan[0]['clean'].sample_name}: {base_description} \n(Note: No normalization performed across or between scans)"
    },
    {
        "scans": {0: scan[0]["flux_corrected"], 90: scan[90]["flux_corrected"], 45: scan[45]["flux_corrected"], 135: scan[135]["flux_corrected"]},
        "ylim": (1e2, 3e5),
        "title": f"{scan[0]['clean'].sample_name}: {base_description} \n(Note: Flux-Corrected)"
    }
]

for spec in plot_specs:

    fig, axs = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)

    integrated_scans = spec["scans"]

    for ax, pol in zip(axs, pols):
        integrated_scan = integrated_scans[pol]

        color_map = plt.get_cmap("viridis")

        for i, energy in enumerate(closest_energies):
            intensity_at_energy = integrated_scan.sel(energy=energy).mean("chi").values
            ax.plot(integrated_scan.coords["q"], intensity_at_energy, label=f"{energy:.1f} eV", color=color_map(i / len(closest_energies)))

        ax.set_xlabel(r"q $(\AA^{-1})$")
        ax.set_ylabel("Intensity")
        ax.set_ylim(*spec["ylim"])
        ax.set_yscale("log")
        ax.legend(loc="upper right")
        ax.set_title(f'Scan {scan[pol]["scan_id"]}:  Polarization = {pol}°')

    fig.suptitle(spec["title"], fontsize=16)
    plt.show()
    plt.close()

## Background Correction

We can try the slope matching method sometime when we have more energies. Since this is a repeats scan, we'll just try flat background subtraction for now.

Slope will refer to the exponent of the power law decay in high q. That is, the slope of log(I) vs log(q). Ideal porod slope for a sharp interface is -4.

In [ ]:
qmin, qmax = 0.07, 0.08  # Porod fit window. where we'll calculate slope from.
highq_min = 0.095         # define high-q window for flat subtraction
highq_max = 0.1

fig, axs = plt.subplots(1,4,figsize=(18,5))

for pol in scan:

    energies = scan[pol]["clean"].energy.values  # grab all energies

    energy_vals = []
    slope_uncorr_vals = []
    slope_corr_vals = []
    background_vals = []
    raw_spectra = []
    corr_spectra = []
    q_arrays = []

    for energy in energies:
        
        # get intensity vs q
        I_array = scan[pol]["clean"].sel(energy=energy).mean("chi")
        I = I_array.values
        q = I_array["q"].values

        #Estimate flat background from high-q region--------------------------------
        highq_mask = (q >= highq_min) & (q <= highq_max)
        if np.any(highq_mask):
            background = I[highq_mask].mean()  # high-q mean
        else:
            background = 0.0

        I_corr = I - background
        I_corr_for_fit = I_corr.copy()
        I_corr_for_fit[I_corr <= 0] = np.nan # avoid non-positive values for log-log fit

        raw_spectra.append(I.copy())
        corr_spectra.append(I_corr.copy())
        q_arrays.append(q.copy())

        #Fit slope in the designated q window --------------------------------------
        mask = (q >= qmin) & (q <= qmax)
        q_fit = q[mask]
        I_fit_uncorr = I[mask]
        I_fit_corr = I_corr_for_fit[mask]

        # skip if bad/empty
        valid_corr = (~np.isnan(I_fit_corr)) & (I_fit_corr > 0)
        if valid_corr.sum() < 2:
            continue
        valid_uncorr = (~np.isnan(I_fit_uncorr)) & (I_fit_uncorr > 0)
        if valid_uncorr.sum() < 2:
            continue

        # linear fit in log-log space, ignoring NaNs
        # uncorr slope
        valid_uncorr = ~np.isnan(I_fit_uncorr)
        slope_uncorr, _ = np.polyfit(np.log(q_fit[valid_uncorr]), np.log(I_fit_uncorr[valid_uncorr]), 1)
        # corr slope
        valid_corr = ~np.isnan(I_fit_corr)
        slope_corr, _ = np.polyfit(np.log(q_fit[valid_corr]), np.log(I_fit_corr[valid_corr]), 1)

        energy_vals.append(float(energy))
        slope_uncorr_vals.append(slope_uncorr)
        slope_corr_vals.append(slope_corr)
        background_vals.append(background)

    # convert to arrays
    energy_vals = np.array(energy_vals)
    slope_uncorr_vals = np.array(slope_uncorr_vals)
    slope_corr_vals = np.array(slope_corr_vals)
    
    colors = {0: "xkcd:bright blue", 45: "yellow", 90: "deeppink", 135: "springgreen"}
    widths = {0: 4, 45: 3, 90: 2, 135: 1}
    
    axs[0].scatter(energy_vals, background_vals, color=colors[pol], label=f"{pol}°")

    axs[1].scatter(energy_vals, slope_uncorr_vals, marker="+", color=colors[pol], label=f"{pol}° raw")
    axs[1].scatter(energy_vals, slope_corr_vals, marker="x", color=colors[pol], label=f"{pol}° corr")

    axs[2].plot(q_arrays[0], raw_spectra[0], color=colors[pol], linewidth=widths[pol], label=f"{pol}° raw")
    axs[2].plot(q_arrays[0], corr_spectra[0], "--", color=colors[pol], linewidth=widths[pol], label=f"{pol}° corr")

    axs[3].plot(q_arrays[1], raw_spectra[1], color=colors[pol], linewidth=widths[pol], label=f"{pol}° raw")
    axs[3].plot(q_arrays[1], corr_spectra[1], "--", color=colors[pol], linewidth=widths[pol], label=f"{pol}° corr")

axs[0].set_title("Subtracted Background")
axs[0].set_xlabel("Energy (eV)")
axs[0].legend()

axs[1].set_title(f"Porod Slope, Fitted from q={qmin} to {qmax}")
axs[1].set_xlabel("Energy (eV)")
axs[1].set_ylabel("Fitted Porod Slope")
axs[1].legend(ncol=2, fontsize=8)

for ax in axs[2:]:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("q (Å$^{-1}$)")
    ax.set_ylabel("Intensity")
    ax.legend(fontsize=8)

axs[2].set_title(f"I vs q, E={energies[0]:.1f} eV")
axs[3].set_title(f"I vs q, E={energies[1]:.1f} eV")
fig.suptitle(f'Subtracting Background via Flat Subtraction for Scans {scan[0]["scan_id"]}, {scan[90]["scan_id"]}, {scan[45]["scan_id"]}, & {scan[135]["scan_id"]}\nNote: Full valid q range, \nVarious pols not normalized to each other"')

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
def subtract_background(scan, pol, data_key, chi_center, chi_halfwidth, qmin, qmax, highq_min, highq_max):
    
    data = scan[pol][data_key]
    energies = data.energy.values

    energy_vals = []
    slope_uncorr_vals = []
    slope_corr_vals = []
    background_vals = []
    raw_spectra = []
    corr_spectra = []
    q_arrays = []

    for energy in energies:

        I_array = (get_chi_slice(data.sel(energy=energy), chi_center, chi_halfwidth).mean("chi"))
        I = I_array.values
        q = I_array.q.values

        highq_mask = (q >= highq_min) & (q <= highq_max)
        background = I[highq_mask].mean() if np.any(highq_mask) else 0.0
        background_vals.append(background)

        I_corr = I - background
        I_corr_for_fit = I_corr.copy()
        I_corr_for_fit[I_corr <= 0] = np.nan

        raw_spectra.append(I.copy())
        corr_spectra.append(I_corr.copy())
        q_arrays.append(q.copy())

        mask = (q >= qmin) & (q <= qmax)

        q_fit = q[mask]
        I_fit_uncorr = I[mask]
        I_fit_corr = I_corr_for_fit[mask]

        valid_uncorr = (~np.isnan(I_fit_uncorr)) & (I_fit_uncorr > 0)
        valid_corr = (~np.isnan(I_fit_corr)) & (I_fit_corr > 0)

        if valid_uncorr.sum() < 2 or valid_corr.sum() < 2:
            continue

        slope_uncorr, _ = np.polyfit(np.log(q_fit[valid_uncorr]), np.log(I_fit_uncorr[valid_uncorr]), 1)
        slope_corr, _ = np.polyfit(np.log(q_fit[valid_corr]), np.log(I_fit_corr[valid_corr]), 1)

        energy_vals.append(float(energy))
        slope_uncorr_vals.append(slope_uncorr)
        slope_corr_vals.append(slope_corr)

    return {
        "energies": np.array(energy_vals),
        "slope_uncorr": np.array(slope_uncorr_vals),
        "slope_corr": np.array(slope_corr_vals),
        "background": np.array(background_vals),
        "raw": raw_spectra,
        "corr": corr_spectra,
        "q": q_arrays,
    }

In [ ]:
# video_frames = []
# videoPath = Path.cwd() / "videos"
# videoPath.mkdir(parents=True, exist_ok=True)

# #---------------------------------------------------------------------------------------------------
# qmin, qmax = 0.07, 0.08  # Porod fit window. where we'll calculate slope from.
# highq_min = 0.095         # define high-q window for flat subtraction
# highq_max = 0.1
# data_key, descriptor = "flux_corrected", "I0 Corrected" #be sure to fix ylim too
# #data_key, descriptor = "clean", "Not I0 Corrected" #be sure to fix ylim too

# chi_halfwidth = 1
# chi_centers = np.arange(-179, 181, 1) # sweep centers
# plot_chi_centers = np.arange(-179, 181, 1) # only plot these


# #---------------------------------------------------------------------------------------------
# colors = {0: "xkcd:bright blue", 90: "yellow", 45: "deeppink", 135: "springgreen"}
# widths = {0: 4, 45: 3, 90: 2, 135: 1}

# for chi_center in chi_centers:

#     make_plot = chi_center in plot_chi_centers

#     if make_plot:
#         fig, axs = plt.subplots(2,3,figsize=(12,9))

#     for pol in scan:

#         energies = scan[pol][data_key].energy.values
#         results = subtract_background(scan, pol, data_key, chi_center, chi_halfwidth, qmin, qmax, highq_min, highq_max)

#         energy_vals = results["energies"]
#         slope_uncorr_vals = results["slope_uncorr"]
#         slope_corr_vals = results["slope_corr"]
#         background_vals = results["background"]
#         raw_spectra = results["raw"]
#         corr_spectra = results["corr"]
#         q_arrays = results["q"]

#         for i, energy in enumerate(energies):
#             background_vs_chi[pol][energy].append(background_vals[i])

#         if make_plot:
            
#             #energy 0 map
#             map0 = scan[pol][data_key].sel(energy=energies[0], method="nearest")
#             axs[0,0].pcolormesh(map0["q"], map0["chi"], map0, shading="auto")
#             axs[0,0].axhspan(chi_center - chi_halfwidth, chi_center + chi_halfwidth, color="white", alpha=0.3)

#             #energy 1 map
#             map1 = scan[pol][data_key].sel(energy=energies[1], method="nearest")
#             axs[0,1].pcolormesh(map1["q"], map1["chi"], map1, shading="auto")
#             axs[0,1].axhspan(chi_center - chi_halfwidth, chi_center + chi_halfwidth, color="white", alpha=0.3)

#             axs[1,0].plot(q_arrays[0], raw_spectra[0], color=colors[pol], alpha=0.2, linewidth=widths[pol], label=f"{pol}° raw")
#             axs[1,0].plot(q_arrays[0], corr_spectra[0], "--", color=colors[pol], linewidth=widths[pol], label=f"{pol}° corr")

#             axs[1,1].plot(q_arrays[1], raw_spectra[1], color=colors[pol], alpha=0.2, linewidth=widths[pol], label=f"{pol}° raw")
#             axs[1,1].plot(q_arrays[1], corr_spectra[1], "--",color=colors[pol], linewidth=widths[pol], label=f"{pol}° corr")
        
#             axs[0,2].scatter(energies, background_vals, color=colors[pol], label=f"{pol}°")

#             axs[1,2].scatter(energy_vals, slope_uncorr_vals, marker='o', facecolors='none', color=colors[pol], alpha=0.2, label=f"{pol}° raw")
#             axs[1,2].scatter(energy_vals, slope_corr_vals, marker="x", color=colors[pol], label=f"{pol}° corr")

#     if make_plot:
        
#         for ax in [axs[0,0], axs[0,1]]:
#             ax.set_xlabel("q (Å$^{-1}$)")
#             ax.set_ylabel("χ")
#             ax.set_ylim(-180,180)
        
#         axs[0,0].set_title(f"I, E={energies[0]:.1f} eV")
#         axs[0,1].set_title(f"I, E={energies[1]:.1f} eV")
#         axs[1,0].set_title(f"I vs q, E={energies[0]:.1f} eV")
#         axs[1,1].set_title(f"I vs q, E={energies[1]:.1f} eV")

#         for ax in [axs[1,0], axs[1,1]]:
#             ax.set_xscale("log")
#             ax.set_yscale("log")
#             ax.set_xlim(0.01, 0.11)
#             ax.set_ylim(1e0, 1e6) 
#             ax.set_xlabel("q (Å$^{-1}$)")
#             ax.set_ylabel("Intensity")
#             ax.legend(loc="lower left", fontsize=8)
#             ax.xaxis.set_major_locator(LogLocator(base=10))
#             ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:g}"))

#         axs[0,2].set_title("Subtracted Background")
#         axs[0,2].set_xlabel("Energy (eV)")
#         axs[0,2].set_ylim(100,1500) #100,800 for clean, 100, 1500 for flux_corrected
#         axs[0,2].legend(loc="upper right", fontsize=8)

#         axs[1,2].set_title(f"Porod Slope, Fitted from q={qmin} to {qmax}")
#         axs[1,2].set_xlabel("Energy (eV)")
#         axs[1,2].set_ylabel("Fitted Porod Slope")
#         axs[1,2].legend(ncol=2, fontsize=8, loc="upper right")
#         axs[1,2].set_ylim(-12,4) #-8,4 for clean, -12, 4 for flux_corrected
#         axs[1,2].axhline(-4, linestyle="--", color="gray", alpha=0.3)

#         for ax in [axs[0,2], axs[1,2]]:
#             ax.set_xlim(284.2,286.2)
#             ax.xaxis.set_major_locator(MultipleLocator(0.5))

#         fig.suptitle(
#             f"Flat Background Subtraction for Scans "
#             f"{scan[0]['scan_id']}, {scan[90]['scan_id']}, "
#             f"{scan[45]['scan_id']}, {scan[135]['scan_id']} at χ = {chi_center}° ± {chi_halfwidth}°\n"
#             f"Full valid q range, "
#             f"{descriptor}"
#         )

#         plt.tight_layout()
#         # Render figure to RGB before closing
#         canvas = FigureCanvas(fig)
#         canvas.draw()
#         width, height = canvas.get_width_height()
#         buf = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8)
#         img = buf.reshape(height, width, 4)[:,:,:3]  # drop alpha
#         video_frames.append(img)
#         plt.close(fig)

# fps = 5

# #REVERSE SWEEP------------------------------
# # # Append linger before reversing
# # linger_seconds = 2
# # linger_frames = int(linger_seconds * fps)
# # video_frames.extend([video_frames[-1]] * linger_frames) 
# # # Append the reversed frames to sweep backward
# # video_frames.extend(video_frames[::-1])
# #---------------------------------------------

# # Save video
# def save_video(frames, filename=f'{scan[0]['scan_id']}_{scan[90]['scan_id']}_{scan[45]['scan_id']}_{scan[135]['scan_id']}_p{workup_pass}_background_subtraction_{data_key}.mp4', fps=fps, repeat=1):
#     output_file = videoPath / filename
#     with imageio.get_writer(str(output_file), fps=fps, codec='libx264', quality=8, macro_block_size=None, format="ffmpeg") as writer:
#         for frame in frames:
#             for _ in range(repeat):
#                 writer.append_data(frame)
#     print(f"Video saved as {output_file}")

# save_video(video_frames)
            

In [ ]:
#^saved: 
#110789_110790_110791_110792_p05_background_subtraction_clean.mp4
#110789_110790_110791_110792_p05_background_subtraction_flux_corrected.mp4

In [ ]:
chi_halfwidth = 1
chi_centers = np.arange(-179, 181, 1) # sweep centers
plot_chi_centers = np.arange(0, 181, 180) # only plot these
data_key, descriptor = "flux_corrected", "I0 Corrected" #be sure to fix ylim too
#data_key, descriptor = "clean", "Not I0 Corrected" #be sure to fix ylim too

background_vs_chi_flux_corrected = {pol: {energy: [] for energy in scan[pol][data_key].energy.values} for pol in scan}
background_vs_chi = background_vs_chi_flux_corrected

#----------------------------------------------------------------
colors = {0: "xkcd:bright blue", 90: "yellow", 45: "deeppink", 135: "springgreen"}
widths = {0: 4, 45: 3, 90: 2, 135: 1}

for chi_center in chi_centers:

    make_plot = chi_center in plot_chi_centers

    if make_plot:
        fig, axs = plt.subplots(2,3,figsize=(12,9))

    for pol in scan:

        energies = scan[pol][data_key].energy.values
        results = subtract_background(scan, pol, data_key, chi_center, chi_halfwidth, qmin, qmax, highq_min, highq_max)

        energy_vals = results["energies"]
        slope_uncorr_vals = results["slope_uncorr"]
        slope_corr_vals = results["slope_corr"]
        background_vals = results["background"]
        raw_spectra = results["raw"]
        corr_spectra = results["corr"]
        q_arrays = results["q"]

        for i, energy in enumerate(energies):
            background_vs_chi[pol][energy].append(background_vals[i])

        if make_plot:
            
            #energy 0 map
            map0 = scan[pol][data_key].sel(energy=energies[0], method="nearest")
            axs[0,0].pcolormesh(map0["q"], map0["chi"], map0, shading="auto")
            axs[0,0].axhspan(chi_center - chi_halfwidth, chi_center + chi_halfwidth, color="white", alpha=0.3)

            #energy 1 map
            map1 = scan[pol][data_key].sel(energy=energies[1], method="nearest")
            axs[0,1].pcolormesh(map1["q"], map1["chi"], map1, shading="auto")
            axs[0,1].axhspan(chi_center - chi_halfwidth, chi_center + chi_halfwidth, color="white", alpha=0.3)

            if np.any(raw_spectra[0] > 0):
                axs[1,0].plot(q_arrays[0], raw_spectra[0], color=colors[pol], alpha=0.2, linewidth=widths[pol], label=f"{pol}° raw")
            if np.any(corr_spectra[0] > 0):
                axs[1,0].plot(q_arrays[0], corr_spectra[0], "--", color=colors[pol], linewidth=widths[pol], label=f"{pol}° corr")

            if np.any(raw_spectra[1] > 0):
                axs[1,1].plot(q_arrays[1], raw_spectra[1], color=colors[pol], alpha=0.2, linewidth=widths[pol], label=f"{pol}° raw")
            if np.any(corr_spectra[1] > 0):
                axs[1,1].plot(q_arrays[1], corr_spectra[1], "--", color=colors[pol], linewidth=widths[pol], label=f"{pol}° corr")
        
            axs[0,2].scatter(energies, background_vals, color=colors[pol], label=f"{pol}°")

            axs[1,2].scatter(energy_vals, slope_uncorr_vals, marker='o', facecolors='none', color=colors[pol], alpha=0.2, label=f"{pol}° raw")
            axs[1,2].scatter(energy_vals, slope_corr_vals, marker="x", color=colors[pol], label=f"{pol}° corr")

    if make_plot:
        
        for ax in [axs[0,0], axs[0,1]]:
            ax.set_xlabel("q (Å$^{-1}$)")
            ax.set_ylabel("χ")
            ax.set_ylim(-180,180)
        
        axs[0,0].set_title(f"I, E={energies[0]:.1f} eV")
        axs[0,1].set_title(f"I, E={energies[1]:.1f} eV")
        axs[1,0].set_title(f"I vs q, E={energies[0]:.1f} eV")
        axs[1,1].set_title(f"I vs q, E={energies[1]:.1f} eV")

        for ax in [axs[1,0], axs[1,1]]:
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlim(0.01, 0.11)
            ax.set_xlabel("q (Å$^{-1}$)")
            ax.set_ylabel("Intensity")
            ax.legend(loc="lower left", fontsize=8)
            ax.xaxis.set_major_locator(LogLocator(base=10))
            ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:g}"))

        axs[0,2].set_title("Subtracted Background")
        axs[0,2].set_xlabel("Energy (eV)")
        axs[0,2].legend(loc="upper right", fontsize=8)

        axs[1,2].set_title(f"Porod Slope, Fitted from q={qmin} to {qmax}")
        axs[1,2].set_xlabel("Energy (eV)")
        axs[1,2].set_ylabel("Fitted Porod Slope")
        axs[1,2].legend(ncol=2, fontsize=8, loc="upper right")
        axs[1,2].axhline(-4, linestyle="--", color="gray", alpha=0.3)

        for ax in [axs[0,2], axs[1,2]]:
            ax.set_xlim(284.5,286.2)

        fig.suptitle(
            f"Flat Background Subtraction for Scans {scan[0]['scan_id']}, {scan[90]['scan_id']}, "
            f"{scan[45]['scan_id']}, {scan[135]['scan_id']} at χ = {chi_center}° ± {chi_halfwidth}°\n"
            f"Full valid q range, {descriptor}"
        )

        plt.tight_layout()
        plt.show()
        plt.close()
            

In [ ]:
chi_halfwidth = 1
chi_centers = np.arange(-179, 181, 1) # sweep centers
plot_chi_centers = np.arange(0, 181, 180) # only plot these
#data_key, descriptor = "flux_corrected", "I0 Corrected" #be sure to fix ylim too
data_key, descriptor = "clean", "Not I0 Corrected" #be sure to fix ylim too

background_vs_chi_clean = {pol: {energy: [] for energy in scan[pol][data_key].energy.values} for pol in scan}
background_vs_chi = background_vs_chi_clean

#----------------------------------------------------------------
colors = {0: "xkcd:bright blue", 90: "yellow", 45: "deeppink", 135: "springgreen"}
widths = {0: 4, 45: 3, 90: 2, 135: 1}

for chi_center in chi_centers:

    make_plot = chi_center in plot_chi_centers

    if make_plot:
        fig, axs = plt.subplots(2,3,figsize=(12,9))

    for pol in scan:

        energies = scan[pol][data_key].energy.values
        results = subtract_background(scan, pol, data_key, chi_center, chi_halfwidth, qmin, qmax, highq_min, highq_max)

        energy_vals = results["energies"]
        slope_uncorr_vals = results["slope_uncorr"]
        slope_corr_vals = results["slope_corr"]
        background_vals = results["background"]
        raw_spectra = results["raw"]
        corr_spectra = results["corr"]
        q_arrays = results["q"]

        for i, energy in enumerate(energies):
            background_vs_chi[pol][energy].append(background_vals[i])

        if make_plot:
            
            #energy 0 map
            map0 = scan[pol][data_key].sel(energy=energies[0], method="nearest")
            axs[0,0].pcolormesh(map0["q"], map0["chi"], map0, shading="auto")
            axs[0,0].axhspan(chi_center - chi_halfwidth, chi_center + chi_halfwidth, color="white", alpha=0.3)

            #energy 1 map
            map1 = scan[pol][data_key].sel(energy=energies[1], method="nearest")
            axs[0,1].pcolormesh(map1["q"], map1["chi"], map1, shading="auto")
            axs[0,1].axhspan(chi_center - chi_halfwidth, chi_center + chi_halfwidth, color="white", alpha=0.3)

            if np.any(raw_spectra[0] > 0):
                axs[1,0].plot(q_arrays[0], raw_spectra[0], color=colors[pol], alpha=0.2, linewidth=widths[pol], label=f"{pol}° raw")
            if np.any(corr_spectra[0] > 0):
                axs[1,0].plot(q_arrays[0], corr_spectra[0], "--", color=colors[pol], linewidth=widths[pol], label=f"{pol}° corr")

            if np.any(raw_spectra[1] > 0):
                axs[1,1].plot(q_arrays[1], raw_spectra[1], color=colors[pol], alpha=0.2, linewidth=widths[pol], label=f"{pol}° raw")
            if np.any(corr_spectra[1] > 0):
                axs[1,1].plot(q_arrays[1], corr_spectra[1], "--", color=colors[pol], linewidth=widths[pol], label=f"{pol}° corr")
        
            axs[0,2].scatter(energies, background_vals, color=colors[pol], label=f"{pol}°")

            axs[1,2].scatter(energy_vals, slope_uncorr_vals, marker='o', facecolors='none', color=colors[pol], alpha=0.2, label=f"{pol}° raw")
            axs[1,2].scatter(energy_vals, slope_corr_vals, marker="x", color=colors[pol], label=f"{pol}° corr")

    if make_plot:
        
        for ax in [axs[0,0], axs[0,1]]:
            ax.set_xlabel("q (Å$^{-1}$)")
            ax.set_ylabel("χ")
            ax.set_ylim(-180,180)
        
        axs[0,0].set_title(f"I, E={energies[0]:.1f} eV")
        axs[0,1].set_title(f"I, E={energies[1]:.1f} eV")
        axs[1,0].set_title(f"I vs q, E={energies[0]:.1f} eV")
        axs[1,1].set_title(f"I vs q, E={energies[1]:.1f} eV")

        for ax in [axs[1,0], axs[1,1]]:
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlim(0.01, 0.11)
            ax.set_xlabel("q (Å$^{-1}$)")
            ax.set_ylabel("Intensity")
            ax.legend(loc="lower left", fontsize=8)
            ax.xaxis.set_major_locator(LogLocator(base=10))
            ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:g}"))

        axs[0,2].set_title("Subtracted Background")
        axs[0,2].set_xlabel("Energy (eV)")
        axs[0,2].legend(loc="upper right", fontsize=8)

        axs[1,2].set_title(f"Porod Slope, Fitted from q={qmin} to {qmax}")
        axs[1,2].set_xlabel("Energy (eV)")
        axs[1,2].set_ylabel("Fitted Porod Slope")
        axs[1,2].legend(ncol=2, fontsize=8, loc="upper right")
        axs[1,2].axhline(-4, linestyle="--", color="gray", alpha=0.3)

        for ax in [axs[0,2], axs[1,2]]:
            ax.set_xlim(284.5,286.2)

        fig.suptitle(
            f"Flat Background Subtraction for Scans {scan[0]['scan_id']}, {scan[90]['scan_id']}, "
            f"{scan[45]['scan_id']}, {scan[135]['scan_id']} at χ = {chi_center}° ± {chi_halfwidth}°\n"
            f"Full valid q range, {descriptor}"
        )

        plt.tight_layout()
        plt.show()
        plt.close()
            

In [ ]:
background_vs_chi = background_vs_chi_clean

for pol in background_vs_chi:
    for energy in background_vs_chi[pol]:
        background_vs_chi[pol][energy] = np.array(background_vs_chi[pol][energy])

colors = {0: "xkcd:bright blue", 90: "yellow", 45: "deeppink", 135: "springgreen"}

for energy in energies:

    fig, ax = plt.subplots(figsize=(8,4))

    for pol in scan:
        ax.plot(chi_centers, background_vs_chi[pol][energy], color=colors[pol], label=f"{pol}°")

    ax.set_xlabel("χ center (°)")
    ax.set_ylabel("High-q background")
    ax.set_title(f"Background vs χ for Scans {scan[0]['scan_id']}, {scan[90]['scan_id']}, {scan[45]['scan_id']}, {scan[135]['scan_id']}, E={energy:.1f} eV \nNote: Full valid q range, \nVarious pols not normalized to each other")
    ax.legend()

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
background_vs_chi = background_vs_chi_flux_corrected

for pol in background_vs_chi:
    for energy in background_vs_chi[pol]:
        background_vs_chi[pol][energy] = np.array(background_vs_chi[pol][energy])

colors = {0: "xkcd:bright blue", 90: "yellow", 45: "deeppink", 135: "springgreen"}

for energy in energies:

    fig, ax = plt.subplots(figsize=(8,4))

    for pol in scan:
        ax.plot(chi_centers, background_vs_chi[pol][energy], color=colors[pol], label=f"{pol}°")

    ax.set_xlabel("χ center (°)")
    ax.set_ylabel("High-q background")
    ax.set_title(f"Background vs χ for Scans {scan[0]['scan_id']}, {scan[90]['scan_id']}, {scan[45]['scan_id']}, {scan[135]['scan_id']}, E={energy:.1f} eV \nNote: Full valid q range, \nFlux Normed")
    ax.legend()

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
#in general, background follows the shape of the I vs q (unnormed)
#so now we need to ask some questions about the background, scale, flux. what order does stuff happen?

-------

In [ ]:
#find background
    #subtraction method
        #different q ranges
        #different chi slices --> does it appear to spike in areas where I vs chi spikes? does it appear multiplicative?
    
#porod slope matching method

#features vs chi:
    #darks
    #i vs chi
    #background
    #A ptwise (no bckgrd subtraction)
    #A ptwise (with bckgrd subtraction)



#

## Chi-Dependence Plots

In [ ]:
# The detector exhibits localized χ-dependent intensity artifacts that appear in the integrated intensity but not in the pointwise anisotropy. 
# This indicates that these artifacts do not substantially perturb the local para/perp ratio. 
# Because the conventional anisotropy estimator weights χ locations according to their measured intensity, it is susceptible to these localized intensity artifacts. 
# In contrast, the pointwise anisotropy estimator is intrinsically insensitive to common multiplicative changes in the local intensity. 
# Given the observed detector behavior, the pointwise estimator is therefore expected to provide a more faithful measure of the underlying molecular anisotropy.

#basically, assuming we have multiplicative errors, ptwise A with artifacts is closer to "real" ptwise A than conventional A with artifacts is to "real" conventional A

In [ ]:
#Intensity vs. chi for various energies

qmin, qmax = 0.035, 0.037

pol_colors = {0: "xkcd:bright blue", 90: "yellow", 45: "deeppink", 135: "springgreen"}
conditions = [("clean", "clean"), ("flux_corrected", "flux_corrected"), 
    ("hf_chi_avgd_normed", "hf_chi_avgd_normed"), ("flxcorr_hfchi_avgd_normed", "flxcorr_hfchi_avgd_normed")]
pol_pairs = [(0, 90), (45, 135)]

fig, axs = plt.subplots(4, 2, figsize=(10, 10), constrained_layout=True)

for row, (condition_name, condition_key) in enumerate(conditions):

    integrated_scans = {
        0: scan[0][condition_key],
        90: scan[90][condition_key],
        45: scan[45][condition_key],
        135: scan[135][condition_key],
    }

    for col, pol_pair in enumerate(pol_pairs):

        ax = axs[row, col]

        for pol in pol_pair:
            integrated_scan = integrated_scans[pol]
            intensity_vs_chi = (integrated_scan.isel(energy=0).sel(q=slice(qmin, qmax)).mean("q"))
            ax.plot(intensity_vs_chi.coords["chi"], intensity_vs_chi.values, label=f"pol={pol}°", color=pol_colors[pol])

        ax.set_title(f"{condition_name}: pol {pol_pair[0]}° vs {pol_pair[1]}°")
        ax.set_xlabel("chi")
        ax.set_ylabel("Intensity")
        ax.legend(loc="lower left")

        # optional dynamic y limits
        # all_vals = np.concatenate([line.get_ydata() for line in ax.lines])
        # ax.set_ylim(np.mean(all_vals)-1.4*np.std(all_vals),
        #             np.mean(all_vals)+1.5*np.std(all_vals))


fig.suptitle(
    f"Intensity vs Chi\n" 
    f"Scans {scan[0]['scan_id']}, {scan[90]['scan_id']}, {scan[45]['scan_id']}, & {scan[135]['scan_id']}\n"
    f"Integrated over q range {qmin}–{qmax}", fontsize=16
)

plt.show()
plt.close()

In [ ]:
#Intensity vs. chi for various energies

energies_of_interest = closest_energies #could pick other energies. this was just convenient

qmin, qmax = 0.035, 0.036

pol_colors = {0: "xkcd:bright blue", 90: "yellow", 45: "deeppink", 135: "springgreen"}
plot_specs = [
    {"integrated_scans": {0: scan[0]["clean"], 90: scan[90]["clean"], 45: scan[45]["clean"], 135: scan[135]["clean"]},
        "title": f'Intensity vs. Chi for Scans {scan[0]["scan_id"]}, {scan[90]["scan_id"]}, {scan[45]["scan_id"]}, & {scan[135]["scan_id"]}\nIntegrated over q range {qmin} to {qmax}'
                 f'\n(Note: No normalization performed across or between scans)\n'},
    {"integrated_scans": {0: scan[0]["hf_chi_avgd_normed"], 90: scan[90]["hf_chi_avgd_normed"], 45: scan[45]["hf_chi_avgd_normed"], 135: scan[135]["hf_chi_avgd_normed"]},
        "title": f'Intensity vs. Chi for Scans {scan[0]["scan_id"]}, {scan[90]["scan_id"]}, {scan[45]["scan_id"]}, & {scan[135]["scan_id"]}\nIntegrated over q range {qmin} to {qmax}'
                 f'\n(Note: Normalized by Chi- and q-averaged Intensity)\n'}
]

for spec in plot_specs:

    integrated_scans = spec["integrated_scans"]

    for pol_pair in [(0, 90), (45, 135)]:
        fig, axs = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

        for ax, energy in zip(axs.flat, energies_of_interest):
            for pol in pol_pair:
                integrated_scan = integrated_scans[pol]
                intensity_vs_chi = (integrated_scan.sel(energy=energy).sel(q=slice(qmin, qmax)).mean("q"))
                ax.plot(intensity_vs_chi.coords["chi"], intensity_vs_chi.values, label=f'pol={pol}°', color=pol_colors[pol])

            ax.set_title(f"{energy:.1f} eV")            
            ax.legend(loc="lower left")
            ax.set_ylabel("Intensity")
            ax.set_xlabel("chi")

            #adjusting ymin for axis display
            #may need some massaging if your data is cut off
            mean_y = np.mean(intensity_vs_chi.values)
            stdev_y = np.std(intensity_vs_chi.values)
            ymin = mean_y - 1.4*stdev_y
            ymax = mean_y + 1.5*stdev_y
            #ax.set_ylim(ymin, ymax)

        fig.suptitle(spec["title"], fontsize=16)

        plt.show()
        plt.close()

I'm still working out what to do with ^this data. For now, it's really only serving two purposes:
1. For my dataset, it confirms measured anisotropy because the intensity trace is sinusoidal and that of pol=0 is offset from that of pol=90. (However, this sinusoidal behavior and clean offset is not present at higher energies)
2. (At least for my data), it also confirms that intensities are not being normalized by the software, as the pol=0 curve and the pol=90 curves are at different heights (i.e., have different average intensities) for a significant proportion of my dataset. This tells us that we need to make corrections when we're calculating anisotropy so that we're not introducing systemic error.

## Anisotropy

In [ ]:
def wrap_chi(vals):
    return ((np.asarray(vals) + 180) % 360) - 180

def unwrap_chi(chi_vals):
    # Wrap into [-180, 180]
    chi_wrapped = ((chi_vals + 180) % 360) - 180
    # Then unwrap across ±180 boundary
    chi_unwrapped = np.where(chi_wrapped < -135, chi_wrapped + 360, chi_wrapped)
    return chi_unwrapped

def reorder_chi_for_9oclock(da):
    chi_vals = da['chi'].values
    chi_wrapped = ((chi_vals + 180) % 360) - 180
    sort_key = np.where(chi_wrapped < -135, chi_wrapped + 360, chi_wrapped)
    sort_order = np.argsort(sort_key)
    return da.isel(chi=sort_order)

def get_chi_slice(data, chi_center, chi_halfwidth): #lifted directly from pyhyperscattering
    """
    Return a slice centered at `chi_center` ± chi_halfwidth, handling wrapping manually.
    Assumes chi ranges from ~-180 to 180.
    """
    chi_vals = data['chi'].values
    chi_min = chi_center - chi_halfwidth
    chi_max = chi_center + chi_halfwidth

    # Wrap chi into [-180, 180)
    chi_vals_wrapped = ((chi_vals + 180) % 360) - 180
    chi_min_wrapped = ((chi_min + 180) % 360) - 180
    chi_max_wrapped = ((chi_max + 180) % 360) - 180

    if chi_min_wrapped <= chi_max_wrapped:
        mask = (chi_vals_wrapped >= chi_min_wrapped) & (chi_vals_wrapped <= chi_max_wrapped)
    else:
        mask = (chi_vals_wrapped >= chi_min_wrapped) | (chi_vals_wrapped <= chi_max_wrapped)

    return data.sel(chi=mask)


#a function to visualize the areas we're using to calculate anisotropy

def plot_wedges(ax, scan, polarization, quadrant, chi_widths):
    max_radius = 800

    quadrant_info = {"12oclock": 90, "9oclock": 180, "6oclock": -90, "3oclock": 0, "1p30oclock": 45, "4p30oclock": -45, "7p30oclock": -135, "10p30oclock": 135}

    if quadrant not in quadrant_info:
        raise ValueError(f"Invalid quadrant: {quadrant}")

    angle = quadrant_info[quadrant]
    chi_width = chi_widths.get(quadrant, 45)  # default 45 if not specified

    # Determine color based on parallel or perpendicular
    angle_diff = abs((angle - polarization) % 360)
    if angle_diff in [0, 180]:  # parallel
        color = "purple"
    else:  # perpendicular
        color = "white"

    # Select data and beam center based on polarization
    data = scan[polarization]["masked_data_static"]
    beam_center = (scan[0]["beamcenter_x"], scan[0]["beamcenter_y"])

    data.sel(energy=284.8).plot(ax=ax, norm=LogNorm(vmin=1e2, vmax=1e5), cmap='jet', add_colorbar=False, add_labels=False)
    wedge = mpatches.Wedge(center=beam_center, r=max_radius, theta1=angle - chi_width, theta2=angle + chi_width, facecolor=color, alpha=0.5)
    ax.add_patch(wedge)

    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xlim(0, 1025)
    ax.set_ylim(0, 1023)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')


In [ ]:
#visualization of areas we're using to calculate anisotropy

custom_chi_widths = {"12oclock": 45, "9oclock": 45, "6oclock": 45, "3oclock": 45, "1p30oclock": 45, "10p30oclock": 45, "7p30oclock": 45, "4p30oclock": 45}

plot_specs = [
    {"quadrants": ["12oclock", "9oclock", "6oclock", "3oclock"], "pols": [0, 90],
        "title": f'Quadrant and chi_width checks for Scans {scan[0]["scan_id"]} & {scan[90]["scan_id"]} \n{scan[0]["sample_name"]}: {scan[0]["base_description"]}'},
    {"quadrants": ["1p30oclock", "4p30oclock", "7p30oclock", "10p30oclock"], "pols": [45, 135],
        "title": f'Quadrant and chi_width checks for Scans {scan[45]["scan_id"]} & {scan[135]["scan_id"]} \n{scan[0]["sample_name"]}: {scan[0]["base_description"]}'}
]

for spec in plot_specs:

    fig, axs = plt.subplots(2, 4, figsize=(18, 12))

    quadrants = spec["quadrants"]

    for i, q in enumerate(quadrants):
        plot_wedges(axs[0, i], scan, polarization=spec["pols"][0], quadrant=q, chi_widths=custom_chi_widths)
        axs[0, i].set_title(f"{q} quadrant (pol={spec["pols"][0]}°),\nchi_width={custom_chi_widths[q]}°")

    for i, q in enumerate(quadrants):
        plot_wedges(axs[1, i], scan, polarization=spec["pols"][1], quadrant=q, chi_widths=custom_chi_widths)
        axs[1, i].set_title(f"{q} quadrant (pol={spec["pols"][1]}°),\nchi_width={custom_chi_widths[q]}°")

    #creating legend patches
    parallel_patch = mpatches.Patch(color='purple', label='Parallel')
    perp_patch = mpatches.Patch(color='white', label='Perpendicular')

    fig.legend(handles=[parallel_patch, perp_patch], loc='upper right', ncol=1, fontsize=12)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fig.suptitle(spec["title"], fontsize=16)
    plt.show()
    plt.close("all")

(JULY 2025: MIGHT NEED TO REVISE THIS EXPLANATION)

Next we are going to calculate anistropy by the standard formula: (I_para - I_perp)/(I_para + I_perp). Note that para and perp quadrants are defined relative to polarization the incident beam, not their location in the diffraction pattern.

To determine the intensity of a given wedge, we average over all values of chi within that wedge. This gives us a single intensity value per q, averaged across the angular range.

Given this setup, calculating anisotropy for each of the four quadrants should produce roughly consistent results. 

However, we need to take into account that beamstop mask has values of 0, not nan. This is important because if the intensity differs between pol=0° vs pol=90°, these zero values could skew the mean intensity by a different amount in each case.

To avoid this, we'll replace all zero values with NaN. That way, when we call xarray.mean() across chi, those points will simply be ignored.

We’ll also need to handle normalization during anisotropy calculation, but we'll do that shortly. For now, let’s start by setting all zeros to NaN.

(JULY 2025: MIGHT NEED TO REVISE THIS EXPLANATION)

Now we can calculate anisotropy for each quadrant. To ensure anisotropy results are consistent across quadrants (e.g., no sign inversion), we normalize each wedge’s intensity by the mean chi-averaged intensity over the full ring for that scan.

This normalization scales the wedge-averaged intensity at each (energy, q) by the full-ring average at the same (energy, q), preserving q-resolution while accounting for systematic differences between polarizations. This effectively removes global intensity offsets.

In [ ]:
#remember that during normalization we defined scan[pol]["chi-avgd_normed"]
#we'll grab some chi slices and from these normalized slices for the purposes of calculating anisotropy

#see how what chi looks like in the data
print(scan[0]["hf_chi_avgd_normed"]["chi"].values) #spans ~-179.5 to ~179.5 deg

In [ ]:
#slice normalized intensity vs chi
scan[90]["I_12"] = get_chi_slice(scan[90]["chi_avgd_normed"], 90, custom_chi_widths["12oclock"])   #para   #dims (energy, chi, q) 
scan[0]["I_12"] = get_chi_slice(scan[0]["chi_avgd_normed"], 90, custom_chi_widths["12oclock"])    #perp

scan[90]["I_6"] = get_chi_slice(scan[90]["chi_avgd_normed"], -90, custom_chi_widths["6oclock"])   #para
scan[0]["I_6"] = get_chi_slice(scan[0]["chi_avgd_normed"], -90, custom_chi_widths["6oclock"]) #perp

scan[0]["I_3"] = get_chi_slice(scan[0]["chi_avgd_normed"], 0, custom_chi_widths["3oclock"])   #para
scan[90]["I_3"] = get_chi_slice(scan[90]["chi_avgd_normed"], 0, custom_chi_widths["3oclock"]) #perp

scan[0]["I_9"] = get_chi_slice(scan[0]["chi_avgd_normed"], 180, custom_chi_widths["9oclock"]) #para
scan[90]["I_9"] = get_chi_slice(scan[90]["chi_avgd_normed"], 180, custom_chi_widths["9oclock"])   #perp

scan[45]["I_1p30"] = get_chi_slice(scan[45]["chi_avgd_normed"], 45, custom_chi_widths["1p30oclock"])  #para
scan[135]["I_1p30"] = get_chi_slice(scan[135]["chi_avgd_normed"], 45, custom_chi_widths["1p30oclock"])    #perp

scan[45]["I_7p30"] = get_chi_slice(scan[45]["chi_avgd_normed"], -135, custom_chi_widths["7p30oclock"])    #para
scan[135]["I_7p30"] = get_chi_slice(scan[135]["chi_avgd_normed"], -135, custom_chi_widths["7p30oclock"])  #perp

scan[135]["I_4p30"] = get_chi_slice(scan[135]["chi_avgd_normed"], -45, custom_chi_widths["4p30oclock"])   #para
scan[45]["I_4p30"] = get_chi_slice(scan[45]["chi_avgd_normed"], -45, custom_chi_widths["4p30oclock"])     #perp

scan[135]["I_10p30"] = get_chi_slice(scan[135]["chi_avgd_normed"], 135, custom_chi_widths["10p30oclock"]) #para
scan[45]["I_10p30"] = get_chi_slice(scan[45]["chi_avgd_normed"], 135, custom_chi_widths["10p30oclock"])   #perp

scan[0]["I_9"] = reorder_chi_for_9oclock(scan[0]["I_9"])  #para
scan[90]["I_9"] = reorder_chi_for_9oclock(scan[90]["I_9"])    #perp

#save to dict
A_calcs["I_12_para"] = scan[90]["I_12"]
A_calcs["I_12_perp"] = scan[0]["I_12"]
A_calcs["I_6_para"] = scan[90]["I_6"]
A_calcs["I_6_perp"] = scan[0]["I_6"]
A_calcs["I_3_para"] = scan[0]["I_3"]
A_calcs["I_3_perp"] = scan[90]["I_3"]
A_calcs["I_9_para"] = scan[0]["I_9"]
A_calcs["I_9_perp"] = scan[90]["I_9"]
A_calcs["I_1p30_para"] = scan[45]["I_1p30"]
A_calcs["I_1p30_perp"] = scan[135]["I_1p30"]
A_calcs["I_7p30_para"] = scan[45]["I_7p30"]
A_calcs["I_7p30_perp"] = scan[135]["I_7p30"] 
A_calcs["I_4p30_para"] = scan[135]["I_4p30"]
A_calcs["I_4p30_perp"] = scan[45]["I_4p30"]
A_calcs["I_10p30_para"] = scan[135]["I_10p30"]
A_calcs["I_10p30_perp"] = scan[45]["I_10p30"]

#calculate anisotropy
A_calcs["A_12"] = (A_calcs["I_12_para"].mean("chi") - A_calcs["I_12_perp"].mean("chi")) / (A_calcs["I_12_para"].mean("chi") + A_calcs["I_12_perp"].mean("chi"))
A_calcs["A_9"] = (A_calcs["I_9_para"].mean("chi") - A_calcs["I_9_perp"].mean("chi")) / (A_calcs["I_9_para"].mean("chi") + A_calcs["I_9_perp"].mean("chi"))
A_calcs["A_6"] = (A_calcs["I_6_para"].mean("chi") - A_calcs["I_6_perp"].mean("chi")) / (A_calcs["I_6_para"].mean("chi") + A_calcs["I_6_perp"].mean("chi"))
A_calcs["A_3"] = (A_calcs["I_3_para"].mean("chi") - A_calcs["I_3_perp"].mean("chi")) / (A_calcs["I_3_para"].mean("chi") + A_calcs["I_3_perp"].mean("chi"))

A_calcs["A_10p30"] = (A_calcs["I_10p30_para"].mean("chi") - A_calcs["I_10p30_perp"].mean("chi")) / (A_calcs["I_10p30_para"].mean("chi") + A_calcs["I_10p30_perp"].mean("chi"))
A_calcs["A_7p30"] = (A_calcs["I_7p30_para"].mean("chi") - A_calcs["I_7p30_perp"].mean("chi")) / (A_calcs["I_7p30_para"].mean("chi") + A_calcs["I_7p30_perp"].mean("chi"))
A_calcs["A_4p30"] = (A_calcs["I_4p30_para"].mean("chi") - A_calcs["I_4p30_perp"].mean("chi")) / (A_calcs["I_4p30_para"].mean("chi") + A_calcs["I_4p30_perp"].mean("chi"))
A_calcs["A_1p30"] = (A_calcs["I_1p30_para"].mean("chi") - A_calcs["I_1p30_perp"].mean("chi")) / (A_calcs["I_1p30_para"].mean("chi") + A_calcs["I_1p30_perp"].mean("chi"))


print(A_calcs["A_12"].dims)

print(f"I_para shape for 12, 9, 6, 3:\n{scan[90]["I_12"].shape, scan[0]["I_9"].shape, scan[90]["I_6"].shape, scan[0]["I_3"].shape}")
print(f"I_para dimensions for 12, 9, 6, 3:\n{scan[90]["I_12"].dims, scan[0]["I_9"].dims, scan[90]["I_6"].dims, scan[0]["I_3"].dims}")
print(f"I_perp shape for 12, 9, 6, 3:\n{scan[0]["I_12"].shape, scan[90]["I_9"].shape, scan[0]["I_6"].shape, scan[90]["I_3"].shape}")
print(f"I_perp dimensions for 12, 9, 6, 3:\n{scan[0]["I_12"].dims, scan[90]["I_9"].dims, scan[0]["I_6"].dims, scan[90]["I_3"].dims}")

print(f'\nA_calcs["A_12"].dims: {A_calcs["A_12"].dims}')

print(f"\nChi values in reorderd 9oclock slice:\n{scan[90]["I_9"]['chi'].values}")

In [ ]:
#slice normalized intensity vs chi
scan[90]["I_12_hf"] = get_chi_slice(scan[90]["hf_chi_avgd_normed"], 90, custom_chi_widths["12oclock"])   #para   #dims (energy, chi, q) 
scan[0]["I_12_hf"] = get_chi_slice(scan[0]["hf_chi_avgd_normed"], 90, custom_chi_widths["12oclock"])    #perp

scan[90]["I_6_hf"] = get_chi_slice(scan[90]["hf_chi_avgd_normed"], -90, custom_chi_widths["6oclock"])   #para
scan[0]["I_6_hf"] = get_chi_slice(scan[0]["hf_chi_avgd_normed"], -90, custom_chi_widths["6oclock"]) #perp

scan[0]["I_3_hf"] = get_chi_slice(scan[0]["hf_chi_avgd_normed"], 0, custom_chi_widths["3oclock"])   #para
scan[90]["I_3_hf"] = get_chi_slice(scan[90]["hf_chi_avgd_normed"], 0, custom_chi_widths["3oclock"]) #perp

scan[0]["I_9_hf"] = get_chi_slice(scan[0]["hf_chi_avgd_normed"], 180, custom_chi_widths["9oclock"]) #para
scan[90]["I_9_hf"] = get_chi_slice(scan[90]["hf_chi_avgd_normed"], 180, custom_chi_widths["9oclock"])   #perp

scan[45]["I_1p30_hf"] = get_chi_slice(scan[45]["hf_chi_avgd_normed"], 45, custom_chi_widths["1p30oclock"])  #para
scan[135]["I_1p30_hf"] = get_chi_slice(scan[135]["hf_chi_avgd_normed"], 45, custom_chi_widths["1p30oclock"])    #perp

scan[45]["I_7p30_hf"] = get_chi_slice(scan[45]["hf_chi_avgd_normed"], -135, custom_chi_widths["7p30oclock"])    #para
scan[135]["I_7p30_hf"] = get_chi_slice(scan[135]["hf_chi_avgd_normed"], -135, custom_chi_widths["7p30oclock"])  #perp

scan[135]["I_4p30_hf"] = get_chi_slice(scan[135]["hf_chi_avgd_normed"], -45, custom_chi_widths["4p30oclock"])   #para
scan[45]["I_4p30_hf"] = get_chi_slice(scan[45]["hf_chi_avgd_normed"], -45, custom_chi_widths["4p30oclock"])     #perp

scan[135]["I_10p30_hf"] = get_chi_slice(scan[135]["hf_chi_avgd_normed"], 135, custom_chi_widths["10p30oclock"]) #para
scan[45]["I_10p30_hf"] = get_chi_slice(scan[45]["hf_chi_avgd_normed"], 135, custom_chi_widths["10p30oclock"])   #perp

scan[0]["I_9_hf"] = reorder_chi_for_9oclock(scan[0]["I_9_hf"])  #para
scan[90]["I_9_hf"] = reorder_chi_for_9oclock(scan[90]["I_9_hf"])    #perp

#save to dict
A_calcs["I_12_para_hf"] = scan[90]["I_12_hf"]
A_calcs["I_12_perp_hf"] = scan[0]["I_12_hf"]
A_calcs["I_6_para_hf"] = scan[90]["I_6_hf"]
A_calcs["I_6_perp_hf"] = scan[0]["I_6_hf"]
A_calcs["I_3_para_hf"] = scan[0]["I_3_hf"]
A_calcs["I_3_perp_hf"] = scan[90]["I_3_hf"]
A_calcs["I_9_para_hf"] = scan[0]["I_9_hf"]
A_calcs["I_9_perp_hf"] = scan[90]["I_9_hf"]
A_calcs["I_1p30_para_hf"] = scan[45]["I_1p30_hf"]
A_calcs["I_1p30_perp_hf"] = scan[135]["I_1p30_hf"]
A_calcs["I_7p30_para_hf"] = scan[45]["I_7p30_hf"]
A_calcs["I_7p30_perp_hf"] = scan[135]["I_7p30_hf"] 
A_calcs["I_4p30_para_hf"] = scan[135]["I_4p30_hf"]
A_calcs["I_4p30_perp_hf"] = scan[45]["I_4p30_hf"]
A_calcs["I_10p30_para_hf"] = scan[135]["I_10p30_hf"]
A_calcs["I_10p30_perp_hf"] = scan[45]["I_10p30_hf"]

#calculate anisotropy
A_calcs["A_12_hf"] = (A_calcs["I_12_para_hf"].mean("chi") - A_calcs["I_12_perp_hf"].mean("chi")) / (A_calcs["I_12_para_hf"].mean("chi") + A_calcs["I_12_perp_hf"].mean("chi"))
A_calcs["A_9_hf"] = (A_calcs["I_9_para_hf"].mean("chi") - A_calcs["I_9_perp_hf"].mean("chi")) / (A_calcs["I_9_para_hf"].mean("chi") + A_calcs["I_9_perp_hf"].mean("chi"))
A_calcs["A_6_hf"] = (A_calcs["I_6_para_hf"].mean("chi") - A_calcs["I_6_perp_hf"].mean("chi")) / (A_calcs["I_6_para_hf"].mean("chi") + A_calcs["I_6_perp_hf"].mean("chi"))
A_calcs["A_3_hf"] = (A_calcs["I_3_para_hf"].mean("chi") - A_calcs["I_3_perp_hf"].mean("chi")) / (A_calcs["I_3_para_hf"].mean("chi") + A_calcs["I_3_perp_hf"].mean("chi"))

A_calcs["A_10p30_hf"] = (A_calcs["I_10p30_para_hf"].mean("chi") - A_calcs["I_10p30_perp_hf"].mean("chi")) / (A_calcs["I_10p30_para_hf"].mean("chi") + A_calcs["I_10p30_perp_hf"].mean("chi"))
A_calcs["A_7p30_hf"] = (A_calcs["I_7p30_para_hf"].mean("chi") - A_calcs["I_7p30_perp_hf"].mean("chi")) / (A_calcs["I_7p30_para_hf"].mean("chi") + A_calcs["I_7p30_perp_hf"].mean("chi"))
A_calcs["A_4p30_hf"] = (A_calcs["I_4p30_para_hf"].mean("chi") - A_calcs["I_4p30_perp_hf"].mean("chi")) / (A_calcs["I_4p30_para_hf"].mean("chi") + A_calcs["I_4p30_perp_hf"].mean("chi"))
A_calcs["A_1p30_hf"] = (A_calcs["I_1p30_para_hf"].mean("chi") - A_calcs["I_1p30_perp_hf"].mean("chi")) / (A_calcs["I_1p30_para_hf"].mean("chi") + A_calcs["I_1p30_perp_hf"].mean("chi"))


print(A_calcs["A_12"].dims)
print(A_calcs["A_12_hf"].dims)

print(f"I_para shape for 12_hf, 9_hf, 6_hf, 3_hf:\n{scan[90]["I_12_hf"].shape, scan[0]["I_9_hf"].shape, scan[90]["I_6_hf"].shape, scan[0]["I_3_hf"].shape}")
print(f"I_para dimensions for 12_hf, 9_hf, 6_hf, 3_hf:\n{scan[90]["I_12_hf"].dims, scan[0]["I_9_hf"].dims, scan[90]["I_6_hf"].dims, scan[0]["I_3_hf"].dims}")
print(f"I_perp shape for 12_hf, 9_hf, 6_hf, 3_hf:\n{scan[0]["I_12_hf"].shape, scan[90]["I_9_hf"].shape, scan[0]["I_6_hf"].shape, scan[90]["I_3_hf"].shape}")
print(f"I_perp dimensions for 12_hf, 9_hf, 6_hf, 3_hf:\n{scan[0]["I_12_hf"].dims, scan[90]["I_9_hf"].dims, scan[0]["I_6_hf"].dims, scan[90]["I_3_hf"].dims}")

print(f'\nA_calcs["A_12_hf"].dims: {A_calcs["A_12_hf"].dims}')

print(f"\nChi values in reorderd hf 9oclock slice:\n{scan[90]["I_9_hf"]['chi'].values}")

In [ ]:
plot_specs = [
    {"energy_slice": slice(280, 290), "title": f"Calculated Anisotropy by Quadrant for Scans {scan[0]["scan_id"]}, {scan[90]["scan_id"]}, {scan[45]["scan_id"]}, & {scan[135]["scan_id"]} \n{scan[0]["sample_name"]}: {scan[0]["base_description"]} \n Scans normalized by intensity at each (energy, q)"},
    #{"energy_slice": None, "title": f"Calculated Anisotropy by Quadrant for Scans {scan[0]["scan_id"]}, {scan[90]["scan_id"]}, {scan[45]["scan_id"]}, & {scan[135]["scan_id"]} (zoomed out) \n{scan[0]["sample_name"]}: {scan[0]["base_description"]} \n Scans normalized by intensity at each (energy, q)"}
]

A_keys = [["A_12", "A_9", "A_6", "A_3"], ["A_10p30", "A_7p30", "A_4p30", "A_1p30"]]
titles = [["12oclock", "9oclock", "6oclock", "3oclock"], ["10p30oclock", "7p30oclock", "4p30oclock", "1p30oclock"]]

for spec in plot_specs:

    fig, axs = plt.subplots(2, 4, figsize=(24, 12))

    for row in range(2):
        for col in range(4):

            key = A_keys[row][col]
            title = titles[row][col]

            if spec["energy_slice"] is not None:
                A_calcs[key].sel(energy=spec["energy_slice"]).plot(ax=axs[row,col], vmin=-0.05, vmax=0.05, cmap='coolwarm')
            else:
                A_calcs[key].plot(ax=axs[row,col], vmin=-0.05, vmax=0.05, cmap='coolwarm')

            axs[row,col].set_title(f"{title} quadrant \nchi_width={custom_chi_widths[title]}")

    fig.suptitle(spec["title"], fontsize=16)

    plt.tight_layout()
    plt.show()
    plt.close()

#all 8 quadrants should all look roughly the same

In [ ]:
plot_specs = [
    {"energy_slice": slice(280, 290), "title": f"Calculated Anisotropy by Quadrant for Scans {scan[0]["scan_id"]}, {scan[90]["scan_id"]}, {scan[45]["scan_id"]}, & {scan[135]["scan_id"]} \n{scan[0]["sample_name"]}: {scan[0]["base_description"]} \n Scans normalized by intensity at each (energy, q)"},
    #{"energy_slice": None, "title": f"Calculated Anisotropy by Quadrant for Scans {scan[0]["scan_id"]}, {scan[90]["scan_id"]}, {scan[45]["scan_id"]}, & {scan[135]["scan_id"]} (zoomed out) \n{scan[0]["sample_name"]}: {scan[0]["base_description"]} \n Scans normalized by intensity at each (energy, q)"}
]

A_keys = [["A_12_hf", "A_9_hf", "A_6_hf", "A_3_hf"], ["A_10p30_hf", "A_7p30_hf", "A_4p30_hf", "A_1p30_hf"]]
titles = [["12oclock", "9oclock", "6oclock", "3oclock"], ["10p30oclock", "7p30oclock", "4p30oclock", "1p30oclock"]]

for spec in plot_specs:

    fig, axs = plt.subplots(2, 4, figsize=(24, 12))

    for row in range(2):
        for col in range(4):

            key = A_keys[row][col]
            title = titles[row][col]

            if spec["energy_slice"] is not None:
                A_calcs[key].sel(energy=spec["energy_slice"]).plot(ax=axs[row,col], vmin=-0.05, vmax=0.05, cmap='coolwarm')
            else:
                A_calcs[key].plot(ax=axs[row,col], vmin=-0.05, vmax=0.05, cmap='coolwarm')

            axs[row,col].set_title(f"{title} quadrant \nchi_width={custom_chi_widths[title]}")

    fig.suptitle(spec["title"], fontsize=16)

    plt.tight_layout()
    plt.show()
    plt.close()

#all 8 quadrants should all look roughly the same

This is just to visualize, not actually for calculating A. Because this is pt by pint. so it's for evaluating how A(chi,q) deepneds on chi and stuff like darks.

NOT SURE THIS IS CORRECT!

In [ ]:
#add in for scans from pol 45, 135
#add in integrated darks (to see where streaks line up)
#rationalize result... make sure we're thinking about this correctly. does it make sense to have both pos and neg? rather than neg and ~0?
#some sort of A(chi) analysis, and A(overall) analysis... to see how backgrounds are affecting A calcs, particularly for low mw stuff
#need to think about how to integrate darks... because what is q (if there's no beam, so no energy?)... and will we need a new integrator?

#save I_para_all? I_perp_all before shiFting? after shifting?
#think hard about how averagnig A across quadrants. make sense to do I_para_all/I_perp_all in shifted quadrants, or to do avg(A_koclock) for all k?

In [ ]:
def split_at_zero(da2d):
    """Split a slice into negative-chi and positive-chi halves."""
    chi = da2d["chi"].values
    neg = da2d.isel(chi=chi < 0)
    pos = da2d.isel(chi=chi >= 0)
    return neg, pos

def split_at_180(da2d):
    """Split a slice into high-positive (>135) and low-negative (<-135) halves."""
    chi = da2d["chi"].values
    pos = da2d.isel(chi=chi > 0)   # ~135 to 180
    neg = da2d.isel(chi=chi < 0)   # ~-180 to -135
    return neg, pos

def plot_polarization_energy_0_90(A_calcs, energy, vrange):
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))

    pol_data = {}
    for pol in ["para", "perp"]:
        pol_data[pol] = {
            "3":  A_calcs[f"I_3_{pol}_hf"].sel(energy=energy, method="nearest"),
            "9":  A_calcs[f"I_9_{pol}_hf"].sel(energy=energy, method="nearest"),
            "6":  A_calcs[f"I_6_{pol}_hf"].sel(energy=energy, method="nearest"),
            "12": A_calcs[f"I_12_{pol}_hf"].sel(energy=energy, method="nearest"),
        }

    diff_data = {k: pol_data["para"][k] - pol_data["perp"][k] for k in ["3", "9", "6", "12"]}

    A_map = None

    for i, (label, data_dict) in enumerate([
        ("Para", pol_data["para"]),
        ("Perp", pol_data["perp"]),
        ("Para - Perp", diff_data)
    ]):

        da3_neg, da3_pos = split_at_zero(data_dict["3"])
        da9_neg, da9_pos = split_at_180(data_dict["9"])
        patches = [da9_neg, data_dict["6"], da3_neg, da3_pos, data_dict["12"], da9_pos]

        if label == "Para - Perp":
            A_map = xarray.concat(patches, dim="chi").sortby("chi")

        all_data = [np.ma.masked_invalid(p.values) for p in patches]

        if i == 2:
            vmin, vmax = -vrange, vrange
        else:
            all_data = np.concatenate([p.values.ravel() for p in patches])
            all_data = all_data[np.isfinite(all_data)]
            vmin = np.nanpercentile(all_data, 5)
            vmax = np.nanpercentile(all_data, 95)

        m = None
        for da_patch in patches:
            chi = da_patch["chi"].values
            idx = np.argsort(chi)
            data = np.ma.masked_invalid(da_patch.values[idx, :])
            chi_sorted = chi[idx]
            q = da_patch["q"].values

            cmap = "coolwarm" if i == 2 else "jet"
            m = axs[i].pcolormesh(q, chi_sorted, data,
                                 shading="auto",
                                 vmin=vmin, vmax=vmax,
                                 cmap=cmap)

        axs[i].set_xlabel("q")
        axs[i].set_ylabel("chi")
        axs[i].set_title(f"{label}\nEnergy={energy} eV")
        fig.colorbar(m, ax=axs[i])

    plt.tight_layout()
    plt.show()
    plt.close()

    return A_map

def plot_polarization_energy_45_135(A_calcs, energy, vrange):
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))

    pol_data = {}
    for pol in ["para", "perp"]:
        pol_data[pol] = {
            "1p30":  A_calcs[f"I_1p30_{pol}_hf"].sel(energy=energy, method="nearest"),
            "7p30":  A_calcs[f"I_7p30_{pol}_hf"].sel(energy=energy, method="nearest"),
            "4p30":  A_calcs[f"I_4p30_{pol}_hf"].sel(energy=energy, method="nearest"),
            "10p30": A_calcs[f"I_10p30_{pol}_hf"].sel(energy=energy, method="nearest"),
        }

    diff_data = {k: pol_data["para"][k] - pol_data["perp"][k] for k in ["1p30", "7p30", "4p30", "10p30"]}

    A_map = None

    for i, (label, data_dict) in enumerate([
        ("Para", pol_data["para"]),
        ("Perp", pol_data["perp"]),
        ("Para - Perp", diff_data)
    ]):

        #da3_neg, da3_pos = split_at_zero(data_dict["3"])
        #da9_neg, da9_pos = split_at_180(data_dict["9"])
        patches = [data_dict["1p30"], data_dict["4p30"], data_dict["7p30"], data_dict["10p30"]]

        if label == "Para - Perp":
            A_map = xarray.concat(patches, dim="chi").sortby("chi")

        all_data = [np.ma.masked_invalid(p.values) for p in patches]

        if i == 2:
            vmin, vmax = -vrange, vrange
        else:
            all_data = np.concatenate([p.values.ravel() for p in patches])
            all_data = all_data[np.isfinite(all_data)]
            vmin = np.nanpercentile(all_data, 5)
            vmax = np.nanpercentile(all_data, 95)

        m = None
        for da_patch in patches:
            chi = da_patch["chi"].values
            idx = np.argsort(chi)
            data = np.ma.masked_invalid(da_patch.values[idx, :])
            chi_sorted = chi[idx]
            q = da_patch["q"].values

            cmap = "coolwarm" if i == 2 else "jet"
            m = axs[i].pcolormesh(q, chi_sorted, data,
                                 shading="auto",
                                 vmin=vmin, vmax=vmax,
                                 cmap=cmap)

        axs[i].set_xlabel("q")
        axs[i].set_ylabel("chi")
        axs[i].set_title(f"{label}\nEnergy={energy} eV")
        fig.colorbar(m, ax=axs[i])

    plt.tight_layout()
    plt.show()
    plt.close()

    return A_map

def plot_polarization_energy(A_calcs, energy, vrange, mode):
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))

    if mode == "0_90":
        sectors = {
            "3":  "3",
            "9":  "9",
            "6":  "6",
            "12": "12",
        }

        def make_patches(data_dict):
            da3_neg, da3_pos = split_at_zero(data_dict["3"])
            da9_neg, da9_pos = split_at_180(data_dict["9"])
            return [da9_neg, data_dict["6"], da3_neg, da3_pos, data_dict["12"], da9_pos]

    elif mode == "45_135":
        sectors = {
            "1p30":  "1p30",
            "7p30":  "7p30",
            "4p30":  "4p30",
            "10p30": "10p30",
        }

        def make_patches(data_dict):
            return [
                data_dict["1p30"],
                data_dict["4p30"],
                data_dict["7p30"],
                data_dict["10p30"]
            ]

    else:
        raise ValueError(f"Unknown mode: {mode}")

    pol_data = {}
    for pol in ["para", "perp"]:
        pol_data[pol] = {
            key: A_calcs[f"I_{val}_{pol}_hf"].sel(energy=energy, method="nearest")
            for key, val in sectors.items()
        }

    diff_data = {
        key: pol_data["para"][key] - pol_data["perp"][key]
        for key in sectors
    }

    A_map = None

    for i, (label, data_dict) in enumerate([
        ("Para", pol_data["para"]),
        ("Perp", pol_data["perp"]),
        ("Para - Perp", diff_data)
    ]):

        patches = make_patches(data_dict)

        if label == "Para - Perp":
            A_map = xarray.concat(patches, dim="chi").sortby("chi")

        if i == 2:
            vmin, vmax = -vrange, vrange
        else:
            all_data = np.concatenate([p.values.ravel() for p in patches])
            all_data = all_data[np.isfinite(all_data)]
            vmin = np.nanpercentile(all_data, 5)
            vmax = np.nanpercentile(all_data, 95)

        m = None
        for da_patch in patches:
            chi = da_patch["chi"].values
            idx = np.argsort(chi)

            data = np.ma.masked_invalid(da_patch.values[idx, :])
            chi_sorted = chi[idx]
            q = da_patch["q"].values

            cmap = "coolwarm" if i == 2 else "jet"

            m = axs[i].pcolormesh(
                q,
                chi_sorted,
                data,
                shading="auto",
                vmin=vmin,
                vmax=vmax,
                cmap=cmap
            )

        axs[i].set_xlabel("q")
        axs[i].set_ylabel("chi")
        axs[i].set_title(f"{label}\nEnergy={energy} eV")
        fig.colorbar(m, ax=axs[i])

    plt.tight_layout()
    plt.show()
    plt.close()

    return A_map

In [ ]:
A_calcs["A_map"] = {
    mode: {
        energy: plot_polarization_energy(A_calcs, energy, 0.15, mode)
        for energy in [284.8, 286]
    }
    for mode in ["0_90", "45_135"]
}

In [ ]:
    # q_min_for_sweep = 0.035
    # q_interval_for_sweep = 0.005
    
    # fig, axs = plt.subplots(2,2,figsize=(12,8))

    # axs[0,0].pcolormesh(A_map_284p8["q"], A_map_284p8["chi"], A_map_284p8, shading="auto", vmin=-0.15, vmax=0.15, cmap="coolwarm")
    # axs[0,1].pcolormesh(A_map_286["q"], A_map_286["chi"], A_map_286, shading="auto", vmin=-0.15, vmax=0.15, cmap="coolwarm")
    # axs[0,0].axvspan(q_min_for_sweep, q_min_for_sweep + q_interval_for_sweep, alpha=0.3, color="gray")
    # axs[0,1].axvspan(q_min_for_sweep, q_min_for_sweep + q_interval_for_sweep, alpha=0.3, color="gray")


    # axs[1,0].plot(A_map_284p8.sel(q=slice(q_min_for_sweep,(q_min_for_sweep + q_interval_for_sweep))).mean("q").chi.values, A_map_284p8.sel(q=slice(q_min_for_sweep,(q_min_for_sweep + q_interval_for_sweep))).mean("q").values)
    # axs[1,1].plot(A_map_286.sel(q=slice(q_min_for_sweep,(q_min_for_sweep + q_interval_for_sweep))).mean("q").chi.values, A_map_286.sel(q=slice(q_min_for_sweep,(q_min_for_sweep + q_interval_for_sweep))).mean("q").values)
    # for ax in axs[1, :]:
    #     ax.set_ylim(-.2, 0.05)

    # axs[0,0].set_title("Energy = 284.8 eV")
    # axs[0,1].set_title("Energy = 286 eV")
    # axs[0,0].set_ylabel("Chi")
    # axs[0,1].set_ylabel("Chi")
    # axs[0,0].set_xlabel("q")
    # axs[0,1].set_xlabel("q")
    # axs[1,0].set_xlabel("Chi")
    # axs[1,1].set_xlabel("Chi")
    # axs[1,0].set_ylabel("Pointwise A")
    # axs[1,1].set_ylabel("Pointwise A")

    # plt.tight_layout()
    # plt.show()
    # plt.close()

In [ ]:

# video_frames = []

# videoPath = Path.cwd() / "videos"
# videoPath.mkdir(parents=True, exist_ok=True)

# q_min_sweep_vals = np.arange(0,0.8,0.005)
# q_interval_for_sweep = 0.005

# for q in q_min_sweep_vals:

#     fig, axs = plt.subplots(2,2,figsize=(12,8))

#     axs[0,0].pcolormesh(A_map_284p8["q"], A_map_284p8["chi"], A_map_284p8, shading="auto", vmin=-0.15, vmax=0.15, cmap="coolwarm")
#     axs[0,1].pcolormesh(A_map_286["q"], A_map_286["chi"], A_map_286, shading="auto", vmin=-0.15, vmax=0.15, cmap="coolwarm")
#     axs[0,0].axvspan(q, q + q_interval_for_sweep, alpha=0.3, color="gray")
#     axs[0,1].axvspan(q, q + q_interval_for_sweep, alpha=0.3, color="gray")


#     axs[1,0].plot(A_map_284p8.sel(q=slice(q,(q + q_interval_for_sweep))).mean("q").chi.values, A_map_284p8.sel(q=slice(q,(q + q_interval_for_sweep))).mean("q").values)
#     axs[1,1].plot(A_map_286.sel(q=slice(q,(q + q_interval_for_sweep))).mean("q").chi.values, A_map_286.sel(q=slice(q,(q + q_interval_for_sweep))).mean("q").values)
#     for ax in axs[1, :]:
#         ax.set_ylim(-.2, 0.05)

#     axs[0,0].set_title("Energy = 284.8 eV")
#     axs[0,1].set_title("Energy = 286 eV")
#     axs[0,0].set_ylabel("Chi")
#     axs[0,1].set_ylabel("Chi")
#     axs[0,0].set_xlabel("q")
#     axs[0,1].set_xlabel("q")
#     axs[1,0].set_xlabel("Chi")
#     axs[1,1].set_xlabel("Chi")
#     axs[1,0].set_ylabel("Pointwise A")
#     axs[1,1].set_ylabel("Pointwise A")

#     plt.tight_layout()

#     # Render figure to RGB before closing
#     canvas = FigureCanvas(fig)
#     canvas.draw()
#     width, height = canvas.get_width_height()
#     buf = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8)
#     img = buf.reshape(height, width, 4)[:,:,:3]  # drop alpha
#     video_frames.append(img)
#     plt.close(fig)

# # Append linger before reversing
# linger_seconds = 2
# fps = 3
# linger_frames = int(linger_seconds * fps)

# video_frames.extend([video_frames[-1]] * linger_frames)

# # Append the reversed frames to sweep backward
# video_frames.extend(video_frames[::-1])

# # Save video
# def save_video(frames, filename='q_sweep_over_ptwise_A.mp4', fps=3, repeat=1):
#     output_file = videoPath / filename
#     with imageio.get_writer(str(output_file), fps=fps, codec='libx264', quality=8, macro_block_size=None, format="ffmpeg") as writer:
#         for frame in frames:
#             for _ in range(repeat):
#                 writer.append_data(frame)
#     print(f"Video saved as {output_file}")

# save_video(video_frames)

In [ ]:

# video_frames = []

# videoPath = Path.cwd() / "videos"
# videoPath.mkdir(parents=True, exist_ok=True)

# q_min_sweep_vals = np.arange(0,0.08,0.0005)
# q_interval_for_sweep = 0.005

# for q in q_min_sweep_vals:

#     fig, axs = plt.subplots(2,2,figsize=(12,8))

#     axs[0,0].pcolormesh(A_map_284p8["q"], A_map_284p8["chi"], A_map_284p8, shading="auto", vmin=-0.15, vmax=0.15, cmap="coolwarm")
#     axs[0,1].pcolormesh(A_map_286["q"], A_map_286["chi"], A_map_286, shading="auto", vmin=-0.15, vmax=0.15, cmap="coolwarm")
#     axs[0,0].axvspan(q, q + q_interval_for_sweep, alpha=0.3, color="gray")
#     axs[0,1].axvspan(q, q + q_interval_for_sweep, alpha=0.3, color="gray")


#     axs[1,0].plot(A_map_284p8.sel(q=slice(q,(q + q_interval_for_sweep))).mean("q").chi.values, A_map_284p8.sel(q=slice(q,(q + q_interval_for_sweep))).mean("q").values)
#     axs[1,1].plot(A_map_286.sel(q=slice(q,(q + q_interval_for_sweep))).mean("q").chi.values, A_map_286.sel(q=slice(q,(q + q_interval_for_sweep))).mean("q").values)
#     for ax in axs[1, :]:
#         ax.set_ylim(-.2, 0.05)

#     axs[0,0].set_title("Energy = 284.8 eV")
#     axs[0,1].set_title("Energy = 286 eV")
#     axs[0,0].set_ylabel("Chi")
#     axs[0,1].set_ylabel("Chi")
#     axs[0,0].set_xlabel("q")
#     axs[0,1].set_xlabel("q")
#     axs[1,0].set_xlabel("Chi")
#     axs[1,1].set_xlabel("Chi")
#     axs[1,0].set_ylabel("Pointwise A")
#     axs[1,1].set_ylabel("Pointwise A")

#     plt.tight_layout()

#     # Render figure to RGB before closing
#     canvas = FigureCanvas(fig)
#     canvas.draw()
#     width, height = canvas.get_width_height()
#     buf = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8)
#     img = buf.reshape(height, width, 4)[:,:,:3]  # drop alpha
#     video_frames.append(img)
#     plt.close(fig)

# # Append linger before reversing
# linger_seconds = 2
# fps = 3
# linger_frames = int(linger_seconds * fps)

# video_frames.extend([video_frames[-1]] * linger_frames)

# # Append the reversed frames to sweep backward
# video_frames.extend(video_frames[::-1])

# # Save video
# def save_video(frames, filename='q_sweep_over_ptwise_A_fixed.mp4', fps=3, repeat=1):
#     output_file = videoPath / filename
#     with imageio.get_writer(str(output_file), fps=fps, codec='libx264', quality=8, macro_block_size=None, format="ffmpeg") as writer:
#         for frame in frames:
#             for _ in range(repeat):
#                 writer.append_data(frame)
#     print(f"Video saved as {output_file}")

# save_video(video_frames)

In [ ]:
# print(videoPath)

In [ ]:
# scan[0]["darks_residual_remeshed_clean"].isel(system=0).dims

In [ ]:
# A_map.dims

# fig, axs = plt.subplots(1,2,figsize=(12,6))

# axs[0].plot(A_map.sel(q=slice(0.035,0.04)).mean("q").chi.values, A_map.sel(q=slice(0.035,0.04)).mean("q").values)
# axs[0].set_ylim(-.2, 0.05)

# axs[1].plot(scan[0]["darks_residual_remeshed_clean"].isel(system=0).sel(q=slice(0.035,0.04)).mean("q").chi.values, scan[0]["darks_residual_remeshed_clean"].isel(system=0).sel(q=slice(0.035,0.04)).mean("q").values)

# plt.tight_layout()
# plt.show()
# plt.close()

In [ ]:
#for darks: compared to average of quadrant intensity, does this point in the quadrant more or less intense? by how much? additive/substractive? 
    # --> does that scale with A calculations? in pt by pt? in averaged? (remember to keep same q. and chi, if it applies.)
        #ultimately wnat a map of "outliers" in darks (integrated) and then want a I vs chi and a A (ptbypt) vs chi for many q. coould do a sum across q. or blocks/bins of q.
    #to see variation in darks. probably do c alcs on the darks. like normalize each quadrant. 

#other features to think about: intensity edge effects in quadrants (heat in pixels), also spikes in the data (are they isotropic/anisotrpic/do they dilute A?)

#ultimatley looking for th emost "honest" value of A


#reasoning: pt by pt analysis of A should give the same vals in 180 degree reflections. can check to see if it does or doesn't. 
# and whether the degree to which it doesn't is correlated to darks background or to intensity weirdness in pixels

In [ ]:
#local smoothing in q-chi space. then take residual. then flag outliers.
#remove radial and angular structure separately, get 2 separate residuals.
#fourier filtering in chi. keep low modes (real physics: 0,2-fold anisotropy, discard high-frequency noise)
#z-score in local neighborhood

We aim to shift the chi range of each quadrant to cover the same range so we can concatenate them and get an A value that takes all four quadrants into account, without doing something like an elementwise average where pts that are not evenly spaced might introduce artifacts if averaged diretly

Let's aim to put everything in the 0 to 90 range. Need to make sure no chi values fall outside a 90deg range:

In [ ]:
def chi_distance(a, b):
    return ((b - a + 180) % 360) - 180

def circ_mean(vals):
    vals = np.deg2rad(vals)
    return np.rad2deg(np.arctan2(np.sin(vals).mean(), np.cos(vals).mean()))

def shift_to_ref(chi_vals, ref):
    chi = wrap_chi(chi_vals)
    shifted = chi - (circ_mean(chi) - ref)
    return wrap_chi(shifted)

quadrant_centers = {
    "1p30": circ_mean(wrap_chi(A_calcs["I_1p30_para"]["chi"].values)),
    "3": circ_mean(wrap_chi(A_calcs["I_3_para"]["chi"].values)),
    "4p30": circ_mean(wrap_chi(A_calcs["I_4p30_para"]["chi"].values)),
    "6": circ_mean(wrap_chi(A_calcs["I_6_para"]["chi"].values)),
    "7p30": circ_mean(wrap_chi(A_calcs["I_7p30_para"]["chi"].values)),
    "9": circ_mean(wrap_chi(A_calcs["I_9_para"]["chi"].values)),
    "10p30": circ_mean(wrap_chi(A_calcs["I_10p30_para"]["chi"].values)),
    "12": circ_mean(wrap_chi(A_calcs["I_12_para"]["chi"].values)),
}

#compute reference center (1p30)
ref_center = circ_mean(wrap_chi(A_calcs["I_1p30_para"]["chi"].values))
shifted_centers = {
    "1p30": circ_mean(shift_to_ref(A_calcs["I_1p30_para"]["chi"].values, ref_center)),
    "3": circ_mean(shift_to_ref(A_calcs["I_3_para"]["chi"].values, ref_center)),
    "4p30": circ_mean(shift_to_ref(A_calcs["I_4p30_para"]["chi"].values, ref_center)),
    "6": circ_mean(shift_to_ref(A_calcs["I_6_para"]["chi"].values, ref_center)),
    "7p30": circ_mean(shift_to_ref(A_calcs["I_7p30_para"]["chi"].values, ref_center)),
    "9": circ_mean(shift_to_ref(A_calcs["I_9_para"]["chi"].values, ref_center)),
    "10p30": circ_mean(shift_to_ref(A_calcs["I_10p30_para"]["chi"].values, ref_center)),
    "12": circ_mean(shift_to_ref(A_calcs["I_12_para"]["chi"].values, ref_center)),
}

order = ["1p30", "3", "4p30", "6", "7p30", "9", "10p30", "12"]
print(f"quadrant        initial center         shifted center")
for k in order:
    print(f"{k}:        {quadrant_centers[k]}       {shifted_centers[k]}")
print("\n")

vals = [quadrant_centers[k] for k in order]
for i in range(len(vals) - 1):
    d = chi_distance(vals[i], vals[i + 1])
    print(f"{order[i]} → {order[i+1]}: {d}")

In [ ]:
#shifting each quadrant to the same chi range so that we can concatenate for an average

def apply_chi_shift(da, shift):
    out = da.copy()
    out["chi"] = wrap_chi(out["chi"].values + shift)
    return out

chi_shifts = {k: ref_center - quadrant_centers[k] for k in quadrant_centers}

I_para_shifted = {}
I_perp_shifted = {}

fig, axs = plt.subplots(1,2,figsize=(10, 3))

for k in quadrant_centers.keys():
    I_para_shifted[k] = apply_chi_shift(A_calcs[f"I_{k}_para"], chi_shifts[k])
    I_perp_shifted[k] = apply_chi_shift(A_calcs[f"I_{k}_perp"], chi_shifts[k])

for k in quadrant_centers.keys():
    chi_vals_para = I_para_shifted[k]["chi"].values
    chi_vals_perp = I_perp_shifted[k]["chi"].values
    axs[0].scatter(chi_vals_para, np.full_like(chi_vals_para, list(quadrant_centers.keys()).index(k)), s=5, label=f"{k} oclock")
    axs[1].scatter(chi_vals_perp, np.full_like(chi_vals_perp, list(quadrant_centers.keys()).index(k)), s=5, label=f"{k} oclock")

for ax in axs:
    ax.set_xlabel("chi (shifted)")
    ax.set_ylabel("quadrant")
    ax.legend()
axs[0].set_title("Chi vals after shifting: Para")
axs[1].set_title("Chi vals after shifting: Perp")
plt.tight_layout()
plt.show()
plt.close()

for k in quadrant_centers.keys():
    print(f"Para chi values in shifted {k} oclock slice:\n {I_para_shifted[k]['chi'].values}\n")
    print(f"Perp chi values in shifted {k} oclock slice:\n {I_perp_shifted[k]['chi'].values}\n")

In [ ]:
#sanity check: if we shifted chi vals correctly, then there should be no difference between quadrant-wise plots of A_koclock and of A shifted

for k in quadrant_centers.keys():

    I_para_mean = I_para_shifted[k].mean("chi")
    I_perp_mean = I_perp_shifted[k].mean("chi")

    A_shifted = (I_para_mean - I_perp_mean) / (I_para_mean + I_perp_mean)

    print(f"{k}:    {np.nanmean(A_calcs[f'A_{k}'] - A_shifted)**2}")

In [ ]:
#what does np.allclose mean? what does atol mean? seems like we have some systemic misalignment on 9oclock slices. need to fix.

In [ ]:
tol = 1e-6
ref_k = list(I_para_shifted.keys())[0]
print("\nnp.allclose check:")
for k in I_para_shifted.keys():
    print(f'{k},    {np.allclose(I_para_shifted[k]["chi"].values, I_para_shifted[ref_k]["chi"].values, atol=tol)}')

print("\nnp.array_equal check:")
for k in I_para_shifted.keys():
    print(f'{k},    {np.array_equal(I_para_shifted[k]["chi"].values, I_para_shifted[ref_k]["chi"].values)}')


#be sure to do rounding

It's not identical, but it's really close. We're going to collapse all vals down to a single chi, so let's force chi vals to be exactly equal. We'll use slice 1p30p as reference.

In [ ]:
print(f"Master chi vals taken from quadrant {ref_k}     (should be 1p30)") 
chi_master = I_para_shifted[ref_k]["chi"]

#re-indexing everyting onto chi_master
I_para_shifted_aligned = {}
I_perp_shifted_aligned = {}

for k in I_para_shifted.keys():
    I_para_shifted_aligned[k] = I_para_shifted[k].interp(chi=chi_master)
    I_perp_shifted_aligned[k] = I_perp_shifted[k].interp(chi=chi_master)

tol = 1e-6
print("\nnp.allclose check:")
for k in I_para_shifted.keys():
    print(f'{k},    {np.allclose(I_para_shifted_aligned[k]["chi"].values, I_para_shifted_aligned[ref_k]["chi"].values, atol=tol)}')

print("\nnp.array_equal check:")
for k in I_para_shifted.keys():
    print(f'{k},    {np.array_equal(I_para_shifted_aligned[k]["chi"].values, I_para_shifted_aligned[ref_k]["chi"].values)}')

In [ ]:
#Concatenating I vals of shifted quadrants 
A_calcs["I_para_concat"] = xarray.concat([I_para_shifted_aligned[k] for k in quadrant_centers.keys()], dim="quadrant")
A_calcs["I_perp_concat"] = xarray.concat([I_perp_shifted_aligned[k] for k in quadrant_centers.keys()], dim="quadrant")

print(f"I_para_concat shape: {A_calcs["I_para_concat"].shape}")
print(f"I_perp_concat shape: {A_calcs["I_perp_concat"].shape}")

print(f"I_para_concat dims: {A_calcs["I_para_concat"].dims}")
print(f"I_perp_concat dims: {A_calcs["I_perp_concat"].dims}")

#avg across I para and perp at given chi vals, ignoring nans
A_calcs["I_para_mean"] = A_calcs["I_para_concat"].mean("quadrant", skipna=True)
A_calcs["I_perp_mean"] = A_calcs["I_perp_concat"].mean("quadrant", skipna=True)

print(f"\nI_para_mean shape: {A_calcs["I_para_mean"].shape}")
print(f"I_perp_mean shape: {A_calcs["I_perp_mean"].shape}")

print(f"I_para_mean dims: {A_calcs["I_para_mean"].dims}")
print(f"I_perp_mean dims: {A_calcs["I_perp_mean"].dims}")

METHOD 1: GLOBAL INTENSITY RATIO

METHOD 2: POINTWISE ANISOTROPY

METHOD 3: QUADRANT ANISTROPY AVERAGE

Going to look at multiple ways of calculating an "overall" A:

* **Method 1: Get global I_para and I_perp from whole scan images, then calculate anisotropy**

  * A calculation: 
    * Starting from $I_{quadrant,\parallel}$ and $I_{quadrant,\parallel}$ (`shape (energy, chi, q)`) for each quadrant, 
    * concatenate into $I_{concat,\parallel}$ and $I_{concat,\perp}$ (`shape: (quadrant, energy, chi, q)`), 
    * collapse across quadrant dimension to get $I_{mean,\parallel}$ and $I_{mean,\perp}$ (`shape: (energy, chi, q)`),  
    * average along chi dimension to get $I_{mean,chiavgd,\parallel}$ and $I_{mean,chiavg,\perp}$ (`shape: (energy, q)`)
    * then calculate an overall A via $A_{avg} = (I_{mean,chiavgd,\parallel} - I_{mean,chiavgd,\perp}) / (I_{mean,chiavgd\parallel} + I_{mean,chiavgd\perp})$ (`shape: (energy,q)`)
  * Most robust against noise, missing pixels, beamstop masking, and unstable denominators because intensities are pooled during concatenation before taking ratios
  * Drawback is quadrants with higher absolute intensity contribute more strongly, so detector gain/brightness artifacts can bias the result if detector response differs by quadrant (which ours does)
  * Best if dectector artifacts are approximately multiplicative and polarization-independent, especially when masking or incomplete chi coverage is significant

* **Method 2: Calculate anisotropy pixel-by-pixel before averaging**
  * A calculation:
    * Starting from $I_{quadrant\parallel}$ and $I_{quadrant\parallel}$ (`shape (energy, chi, q)`) for each quadrant, 
    * concatenate into $I_{concat,\parallel}$ and $I_{concat,\perp}$ (`shape: (quadrant, energy, chi, q)`), 
    * calculate A for each point in (`(quadrant, energy, chi, q)`) via $A_{pt} = (I_{concat,\parallel} - I_{concat,\perp}) / (I_{concat,\parallel} + I_{concat,\perp})$ (`shape: (quadrant, energy, chi, q)`)
    * average across chi and quadrant dimensions to get an overall A via $A_{avg} = A_{pt}$.mean(chi, quadrant) (`shape: (energy,q)`)
  * Preserves local anisotropy structure and treats all local regions equally before averaging.
  * Drawback is this method is highly sensitive to noise, low-intensity regions, unstable denominators, NaNs, and masking artifacts. Small denominator regions can dominate the avg.
  * This would be best for systems where local anisotropy distributions themselves are of interest and signal-to-noise is uniformly high. Probably not preferred for robust ensemble anisotropy estimation like we're trying to do here


* **Method 3: Calculate anisotropy independently per quadrant, then average quadrants**
  * A calculation:
    * Starting from $I_{quadrant\parallel}$ and $I_{quadrant\parallel}$ (`shape (energy, chi, q)`) for each quadrant, 
    * chi-average each quadrant to get $I_{quadrant,chiavgd\parallel}$ and $I_{quadrant,chiavgd\parallel}$ (`shape (energy, q)`)
    * $A_{quadrant} = (I_{quadrant,chiavgd\parallel} - I_{quadrant,chiavgd,\perp}) / (I_{quadrant,chiavgd\parallel} + I_{quadrant,chiavgd\perp})$ (`shape: (energy,q)`)
    * then average A across all quadrants to get an overall A $A_{avg} = A_{quadrant}$.mean(quadrant) (`shape: (energy,q)`)
  * Compute a single anisotropy value for each quadrant independently, then average the quadrant anisotropies together.
  * Reduces sensitivity to detector brightness/gain differences because each quadrant is normalized internally before averaging.
  * Drawback is if a quadrant contains beamstop masking or loses regions with strong para/perp contrast, its anisotropy can become systematically biased but it still contributes equally to the final average.
  * Best for systems where detector quadrant gain differences dominate the error budget and chi coverage is approximately equivalent across quadrants.


For your use case, I would actually favor Method 2, with one caveat.

The earlier argument still applies: after normalization, Method 1 is still vulnerable to localized spikes because the spike enters the numerator and denominator before the ratio. Method 2 limits the influence of a single bad χ bin.

The caveat is the one you already identified: Method 2 can become noisy where:

I
∥
∗
	​

+I
⊥
∗
	​

→0

because those pixels get huge leverage. But your workflow already has:

mask propagation,
remeshed pixel fraction filtering,
NaN masking,
χ averaging over a defined good region,

so you have already addressed the main failure modes.

I would slightly rewrite the interpretation of Method 2 in your notes:

Method 2: Calculate anisotropy pointwise before averaging
This is the least biased estimator when the goal is an ensemble-average anisotropy and localized intensity artifacts remain after masking. Because each χ/quadrant location contributes equally, isolated bright pixels or detector spikes cannot dominate the result. Requires careful masking of low-intensity regions because the denominator is evaluated locally.

Also, one subtle point: since you normalize each polarization independently by χ-average, Method 2 is no longer "preserving absolute anisotropy amplitude" in the same sense as Method 1. You are asking a slightly different question:

Method 1 asks: "After throughput correction, what is the anisotropy of the total scattered intensity?"
Method 2 asks: "What anisotropy does the typical angular location exhibit?"

For your RSoXS case, where you care about a subtle orientational signature and not absolute scattering-weighted contrast, Method 2 is probably the more defensible metric.

In [ ]:
#METHOD 1: CALCULCATE CONCATENATED I_PARA_ALL (MEANCHI), I_PERP_ALL (MEANCHI) BASED ON ALL 8 QUADRANTS, CALCULATE ANISOTROPY DIRECTLY FROM THAT
#METHOD 2: CALCULCATE CONCATENATED I_PARA_ALL (CHI RESOLVED), I_PERP_ALL (CHI RESOLVED ) BASED ON ALL 8 QUADRANTS, CALCULATE DIFFERENCE, TAKE MEANCHI MEANQUADRANT OF THAT (SEEMS SKETCHY, BUT IS WHAT I'VE DOEN IN THE PAST FOR SOME REASON)
#METHOD 2: CALCULCATE CONCATENATED A BASED ON ALL 8 QUADRANTS, AVERAGE THOSE

#METHOD 1:
print(f'Method 1: A_avg = [I_para_mean.mean("chi") - I_perp_mean.mean("chi")] / [I_para_mean.mean("chi") - I_perp_mean.mean("chi")]')
A_method1 = (A_calcs["I_para_mean"].mean("chi") - A_calcs["I_perp_mean"].mean("chi")) / (A_calcs["I_para_mean"].mean("chi") + A_calcs["I_perp_mean"].mean("chi"))
A_calcs["A_avg_mthd1"] = A_method1
print(f'Method 1 A shape: {A_method1.shape}')
print(f'Method 1 A dims: {A_method1.dims}')

#METHOD 2:
print(f'\nMethod 2: I_vs_chi_quadrant = I_para_concat - I_perp_mean] / [I_para_mean - I_para_mean]')
print(f"          A = I_vs_chi_quadrant.mean(dim=['chi', 'quadrant'], skipna=True)")
I_vs_chi_quadrant_method2 = (A_calcs["I_para_concat"] - A_calcs["I_perp_concat"]) / (A_calcs["I_para_concat"] + A_calcs["I_perp_concat"])
A_method2 = I_vs_chi_quadrant_method2.mean(dim=['chi', 'quadrant'], skipna=True)
A_calcs["A_avg_mthd2"] = A_method2
print(f'Method 2 I_vs_chi_quadrant_method2 shape: {I_vs_chi_quadrant_method2.shape}')
print(f'Method 2 I_vs_chi_quadrant_method2 dims: {I_vs_chi_quadrant_method2.dims}')
print(f'Method 2 A shape: {A_method2.shape}')
print(f'Method 2 A dims: {A_method2.dims}')

#METHOD 3:
print(f"\nMethod 3: Simply average A of all 8 quadrants")
A_stack = xarray.concat([A_calcs[k] for k in ["A_1p30", "A_3", "A_4p30", "A_6", "A_7p30", "A_9", "A_10p30", "A_12"]], dim="quadrant")
A_method3 = A_stack.mean(dim=['quadrant'], skipna=True)
A_calcs["A_avg_mthd3"] = A_method3
print(f'A_1p30 shape: {A_calcs["A_1p30"].shape}')
print(f'A_1p30 dims: {A_calcs["A_1p30"].dims}')
print(f'A_stack shape: {A_stack.shape}')
print(f'A_stack dims: {A_stack.dims}')
print(f'Method 3 A shape: {A_method3.shape}')
print(f'Method 3 A dims: {A_method3.dims}')


#COMPUTING DIFFERENCES
diff_12 = A_method1 - A_method2
diff_13 = A_method1 - A_method3
diff_23 = A_method2 - A_method3

fig, axs = plt.subplots(2,3,figsize=(16,8))

A_method1.sel(energy=slice(280, 290)).plot(ax=axs[0,0], vmin=-0.05, vmax=0.05, cmap='coolwarm')
A_method2.sel(energy=slice(280, 290)).plot(ax=axs[0,1], vmin=-0.05, vmax=0.05, cmap='coolwarm')
A_method3.sel(energy=slice(280, 290)).plot(ax=axs[0,2], vmin=-0.05, vmax=0.05, cmap='coolwarm')

vrange = 0.0005
diff_12.sel(energy=slice(280, 290)).plot(ax=axs[1,0], vmin=-vrange, vmax=vrange, cmap='coolwarm')
diff_23.sel(energy=slice(280, 290)).plot(ax=axs[1,1], vmin=-vrange, vmax=vrange, cmap='coolwarm')
diff_13.sel(energy=slice(280, 290)).plot(ax=axs[1,2], vmin=-vrange, vmax=vrange, cmap='coolwarm')

axs[0,0].set_title("Method1")
axs[0,1].set_title("Method2")
axs[0,2].set_title("Method3")
axs[1,0].set_title("Method1 - Method2")
axs[1,1].set_title("Method2 - Method3")
axs[1,2].set_title("Method1 - Method3")

plt.tight_layout()
plt.show()
plt.close()


Let's stick with Method 1 - NO METHOD 2 NOW:

In [ ]:
#look at zoomed in and full scale energies:

fig, axs = plt.subplots(1,2,figsize=(18,7))

A_calcs["A_avg_mthd2"].sel(energy=slice(280, 290)).plot(ax=axs[0], vmin=-.05, vmax=0.05, cmap='coolwarm')
axs[0].set_title("Zoomed in")
A_calcs["A_avg_mthd2"].plot(ax=axs[1], vmin=-.05, vmax=0.05, cmap='coolwarm')
axs[1].set_title("Zoomed out")

fig.suptitle(f"Average A take from full diffraction pattern (Method 2), \n" 
                          f"chi_widths: 12oclock={custom_chi_widths["12oclock"]}, 9oclock={custom_chi_widths["9oclock"]}, 6oclock={custom_chi_widths["6oclock"]}, 3oclock={custom_chi_widths["3oclock"]} \n"
                          f"Scans {scan[0]["scan_id"]} & {scan[0]["scan_id"]}: {sample_name}: {base_description}", fontsize=16)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
savgol_window_length = 21
A_calcs["A_avg_mthd2_savgol"] = A_calcs["A_avg_mthd2"].copy(data=savgol_filter(A_calcs["A_avg_mthd2"].values, window_length=savgol_window_length, polyorder=3, axis=-1))

A_calcs["dA_dq_mthd2"] = A_calcs["A_avg_mthd2_savgol"].differentiate(coord="q") #doing it this way to preserve xarray so we can keep metadata... is it smart enough to know this is q savgol, not raw q?

fig, axs = plt.subplots(1,3,figsize=(20, 6))

# now can use .sel and .plot
#moving forward, dA/dq refers to data where A has been averaged across all 4 quadrants and savgol smoothed
A_calcs["A_avg_mthd2"].sel(energy=slice(280, 290)).plot(ax=axs[0],vmin=-.05, vmax=0.05, cmap='coolwarm'), axs[0].set_title(f"A av, full diffraction image")
A_calcs["A_avg_mthd2_savgol"].sel(energy = slice(280,290)).plot(ax=axs[1], vmin=-.05, vmax = 0.05, cmap = 'coolwarm'), axs[1].set_title(f"Savgol smooth, window_length={savgol_window_length}")
A_calcs["dA_dq_mthd2"].sel(energy = slice(280,290)).plot(ax=axs[2], vmin=-15, vmax = 15, cmap = 'coolwarm'), axs[2].set_title(f"dA/dq, after av and savgol smooth, window_length={savgol_window_length}")

fig.suptitle(f"Smoothing and Differentiating A (averaged across all quadrants via Method 2)\n"
             f"From Scans {scan[0]['scan_id']} & {scan[0]['scan_id']}: {sample_name}: {base_description} \n"
             f"From Average A of all 4 Quadrants, chi_widths: 12oclock={custom_chi_widths['12oclock']}, 9oclock={custom_chi_widths['9oclock']}, 6oclock={custom_chi_widths['6oclock']}, 3oclock={custom_chi_widths['3oclock']} \n", fontsize=16
)               

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# #plotting only "good" q range, smoothing it, and differentiating
# #this is the max possible q window (if we're averaging all 4 quuadrants) - essentially here we're identifying the min possible q and max possible q, after we've made all corrections

# fig, axs = plt.subplots(1,3,figsize=(18, 6))

# #at the edges of the dataset, the scipy savgol filter works like this: 
# #at the start, the missing points to the left are mirrored from the existing right-side values
# #at the end, the missing points to the right are mirrored from the existing left-side values
# #q resolution is 500 pts, so half of the savol window (~10 pts) is equal to 0.02 q

# q_vals = A_calcs["A_avg_mthd2"].q.values

# #using the qmin we got from eyeballing, and the qmax we got from max radius calcs & setting to e=280
# in_bounds = (q_vals >= q_min) & (q_vals <= q_max[energy_sel]) #energy_sel will probably be 280. was specified in previous cell
# q_within_bounds = q_vals[in_bounds]

# pts_trimmed = math.ceil(savgol_window_length / 2)

# # skipping number of pts at edges equal to half of savgol window
# q_trimmed = q_within_bounds[pts_trimmed:-pts_trimmed]
# q_min_trimmed = q_trimmed[0]
# q_max_trimmed = q_trimmed[-1]

# print(f'q_min: {q_min}')
# print(f'q_min_trimmed (lowest possible q_min): {q_min_trimmed:.4f}')
# print(f'q_max: {q_max[energy_sel]:.4f}')
# print(f'q_max_trimmed (highest possible q_max): {q_max_trimmed:.4f}')

# #trimming so that we will be able to sum across the dataset without edge effects
# A_calcs["A_avg_mthd1"].sel(energy=slice(280, 290)).sel(q=slice(q_min_trimmed, q_max_trimmed)).plot(ax=axs[0],vmin=-.05, vmax=0.05, cmap='coolwarm')
# axs[0].set_title(f"A av, all quadrants")
# A_calcs["A_avg_mthd1_savgol"].sel(energy = slice(280,290)).sel(q=slice(q_min_trimmed, q_max_trimmed)).plot(ax=axs[1], vmin=-.05, vmax = 0.05, cmap = 'coolwarm')
# axs[1].set_title(f"Savgol smoothed, window_length={savgol_window_length}")
# A_calcs["dA_dq_mthd1"].sel(energy = slice(280,290)).sel(q=slice(q_min_trimmed, q_max_trimmed)).plot(ax=axs[2], vmin=-15, vmax = 15, cmap = 'coolwarm')
# axs[2].set_title(f"dA/dq of averaged & smoothed A")

# fig.suptitle(f"Maximum Extent of q After Edge Corrections Involving Savgol Smoothing (Method 1):\n"
#              f"From Scans {scan[0]['scan_id']} & {scan[0]['scan_id']}: {scan[0]["sample_name"]}: {scan[0]["base_description"]} \n"
#              f"From Average A of all 4 Quadrants, chi_widths: 12oclock={custom_chi_widths['12oclock']}, 9oclock={custom_chi_widths['9oclock']}, 6oclock={custom_chi_widths['6oclock']}, 3oclock={custom_chi_widths['3oclock']} \n"
#              f"10p30oclock={custom_chi_widths['10p30oclock']}, 7p30oclock={custom_chi_widths['7p30oclock']}, 4p30oclock={custom_chi_widths['4p30oclock']}, 1p30oclock={custom_chi_widths['1p30oclock']} \n"
#              f"(After trimming savgol edge effects): Lowest possible qmin={q_min_trimmed:.4f}, Highest possible qmax={q_max_trimmed:.4f} \n", fontsize=16
# )               

# plt.tight_layout()
# plt.show()
# plt.close()

# #might want to play with the vmin, vmax to enhance contrast/readability

In [ ]:
# def ARmap_energy_selection(e_vals, q_min, q_max, emin, emax, vscale_A=None, vscale_dA_dq=None):

#     #trimming q bounds if necessary, let be if not
#     q_min_used = max(q_min, q_min_trimmed) 
#     q_max_used = min(q_max, q_max_trimmed) 

#     #we're going to examine only the energy and q range of interest
#     A_selected = A_calcs["A_avg_mthd1_savgol"].sel(energy=slice(emin, emax), q=slice(q_min_used, q_max_used))
#     dA_dq_selected = A_calcs["dA_dq_mthd1"].sel(energy=slice(emin, emax), q=slice(q_min_used, q_max_used)) #above, we defined dA_dq as allquad average and savgol smoothed of full e and q ranges

#     def make_plots(fig):
#         gs = GridSpec(nrows=6, ncols=4, width_ratios=[1.8, 0.001, 1, 1])  #0.001 is a spacer column

#         #making emply subplot for big anisotropy map
#         ax_top = fig.add_subplot(gs[:3, 0])
#         ax_bottom = fig.add_subplot(gs[3:, 0])
#         #shifting a bit to improve spacing between top and bottom axs
#         ax_top.set_position(ax_top.get_position().translated(0, 0.04)) 
#         ax_bottom.set_position(ax_bottom.get_position().translated(0, 0))
#         #making empty subplots for statistical traces
#         axs = np.empty((6, 2), dtype=object)

#         #filling in axs with designated subplot axes, skipping the first 2 columns
#         for i in range(6):
#             for j in range(2):
#                 axs[i, j] = fig.add_subplot(gs[i, j + 2])
        
#         if vscale_A is None:
#             vmax_A = np.nanmax(np.abs(A_selected.values))
#         else:
#             vmax_A=vscale_A
#         vmin_A = -vmax_A

#         if vscale_dA_dq is None:
#             vmax_dA_dq = np.nanmax(np.abs(dA_dq_selected.values))
#         else:
#             vmax_dA_dq=vscale_dA_dq
#         vmin_dA_dq = -vmax_dA_dq

#         A_selected.plot(ax=ax_top, vmin=vmin_A, vmax=vmax_A, cmap='coolwarm', cbar_kwargs={"label": "A", "shrink": 0.5})
#         dA_dq_selected.plot(ax=ax_bottom, vmin=vmin_dA_dq, vmax=vmax_dA_dq, cmap='coolwarm', cbar_kwargs={"label": "dA/dq","shrink": 0.5})

#         #TOP ROW: A and dA/dq
#         #A - mean of A over q, for each energy
#         A_mean_q = A_selected.mean('q')
#         A_mean_q.plot(ax=axs[0, 0])
#         axs[0, 0].set_ylabel("Mean", fontsize=8)
#         #dA/dq - mean of dA/dq across q, for each energy
#         dA_dq_mean_q = dA_dq_selected.mean('q')
#         dA_dq_mean_q.plot(ax=axs[0, 1])
        
#         #METHOD 1: CUMULATIVE ABSOLUTE SUM 
#         #A - cumulative sum of abs(A) across q, per energy
#         cumulative_abs_A = np.nansum(np.abs(A_selected.values), axis=-1)
#         axs[1,0].plot(A_selected.coords["energy"].values, cumulative_abs_A)
#         axs[1,0].set_ylabel("Cumulative \nabs(A)", fontsize=8)
#         axs[1,0].set_xlabel("energy", fontsize=8)
#         axs[1,0].tick_params(labelsize=8)
#         #dA/dq - cumulative sum of abs(dA/dq) across q, per energy
#         cumulative_abs_dA_dq = np.nansum(np.abs(dA_dq_selected.values), axis=-1)
#         axs[1,1].plot(dA_dq_selected.coords["energy"].values, cumulative_abs_dA_dq)
#         axs[1,1].set_xlabel("energy", fontsize=8)
#         axs[1,1].tick_params(labelsize=8)

#         #METHOD 2: TOTAL VARIATION
#         # Total variation is the 1D arclength of a curve.
#         # A - total variation of A vs q (i.e., sum of the absolute change between neighboring q-points), per energy
#         total_variation_A = np.nansum(np.abs(np.diff(A_selected.values, axis=-1)), axis=-1) #-1 refers to last axis, which is q, since A_selected has shape (n_energy, n_q)
#         xarr_total_variation_A = xarray.DataArray(total_variation_A, coords={"energy": A_selected.coords["energy"]}, dims=["energy"])
#         xarr_total_variation_A.plot(ax=axs[2, 0])
#         axs[2,0].set_ylabel("Total \nVariation", fontsize=8)
#         #dA/dq - total variation of dA/dq vs q
#         total_variation_dA_dq = np.nansum(np.abs(np.diff(dA_dq_selected.values, axis=-1)), axis=-1)
#         xarr_total_variation_dA_dq = xarray.DataArray(total_variation_dA_dq, coords={"energy": dA_dq_selected.coords["energy"]}, dims=["energy"])
#         xarr_total_variation_dA_dq.plot(ax=axs[2, 1])

#         #METHOD 3: ROOT MEAN SQUARE
#         #A - RMS of A across q, per energy
#         rms_A = np.sqrt(np.nanmean(A_selected**2, axis=-1))
#         axs[3,0].plot(A_selected.coords["energy"].values, rms_A)
#         axs[3,0].set_ylabel("Root mean \nsquare", fontsize=8)
#         #dA/dq - RMS of dA/dq across q, per energy
#         rms_dA_dq = np.sqrt(np.nanmean(dA_dq_selected**2, axis=-1))
#         axs[3,1].plot(dA_dq_selected.coords["energy"].values, rms_dA_dq)

#         #METHOD 4: SPECTRAL ENTROPY
#         #spectral entropy quantifies the complexity or "disorder" in the frequency content of a signal. 
#         #It’s computed by taking the power spectral density (via a periodogram) of A as a function of q, at each energy, 
#         #and then calculating the Shannon entropy of that normalized spectrum.
#         #essentially, if variation in A is characterized by fewer frequencies (i.e., is periodic/structured), spectral entropy is lower. 
#         #but if variation in A is flat or noisy (i.e., lots of frequencies), spectral entropy is higher
#         def spectral_entropy(row):
#             if np.all(np.isnan(row)) or np.all(row == 0):
#                 return np.nan
#             f, Pxx = periodogram(row)
#             total_power = np.sum(Pxx)
#             if total_power == 0:
#                 return np.nan
#             Pxx = Pxx / total_power
#             return entropy(Pxx)
#         #A - spectral entropy of A across q, per energy
#         entropy_A = np.array([spectral_entropy(row) for row in np.nan_to_num(A_selected.values, nan=0.0)])
#         axs[4,0].plot(A_selected.coords["energy"].values, entropy_A)
#         axs[4,0].set_ylabel("Spectral \nentropy", fontsize=8)
#         #dA/dq - spectral entropy of dA/dq across q, per energy
#         entropy_dA_dq = np.array([spectral_entropy(row) for row in np.nan_to_num(dA_dq_selected.values, nan=0.0)])
#         axs[4,1].plot(dA_dq_selected.coords["energy"].values, entropy_dA_dq)

#         #METHOD 5: ZERO CROSSINGS
#         #A - for each energy, counts how many times A(q) crosses zero along the selected q range.
#         zero_crossings_A = np.sum(np.diff(np.signbit(A_selected), axis=-1), axis=-1)
#         axs[5,0].plot(A_selected.coords["energy"].values, zero_crossings_A)
#         axs[5,0].set_ylabel("Zero \ncrossings", fontsize=8)
#         #dA/dq - same idea, applied to the derivative dA/dq
#         zero_crossings_dA_dq = np.sum(np.diff(np.signbit(dA_dq_selected), axis=-1), axis=-1)
#         axs[5,1].plot(dA_dq_selected.coords["energy"].values, zero_crossings_dA_dq)

#         #adjusting subplot labels, titles, ticks
#         axs[5,0].set_xlabel("energy", fontsize=8)
#         axs[5,0].tick_params(labelsize=8)
#         axs[5,1].set_xlabel("energy", fontsize=8)
#         axs[5,1].tick_params(labelsize=8)
#         axs[0,0].set_title("Measures of A \nAveraged Across q at Each Energy", fontsize=10)
#         axs[0,1].set_title("Measures of dA/dq \nAveraged Across q at Each Energy", fontsize=10)
#         #hiding axes labels/ticks in all but the bottom row of subplots for statistical traces
#         for i in range(5):
#             for j in range(2):
#                 ax = axs[i, j]
#                 ax.tick_params(labelsize=8)
#                 if i < 5:
#                     ax.tick_params(labelbottom=False)
#                     ax.set_xlabel("")

#         fig.suptitle(
#             f"Scans {scan[0]['scan_id']} & {scan[90]['scan_id']}: {sample_name}: {base_description}\n"
#             f"(Here, A is the 4-quadrant average & is savgol smoothed (window={savgol_window_length}). "
#             f"chi_widths: 12oclock={custom_chi_widths['12oclock']}, 9oclock={custom_chi_widths['9oclock']}, "
#             f"6oclock={custom_chi_widths['6oclock']}, 3oclock={custom_chi_widths['3oclock']})\n"
#             f"q range = [{q_min_used:.4f}, {q_max_used:.4f}]",
#             y=1.01 #shifting title to allow some extra spacing
#         )

#         #adding lines at specified energies
#         for ax in axs.flat:
#             ax.set_xlim(emin, emax)

#         return fig, ax_top, ax_bottom, axs, vmin_A, vmax_A, vmin_dA_dq, vmax_dA_dq

#     #return figure without energy lines
#     fig1, ax_top1, ax_bottom1, axs1, vmin_A, vmax_A, vmin_dA_dq, vmax_dA_dq = make_plots(plt.figure(figsize=(18, 9), constrained_layout=True))
    
#     #add lines for selected energies
#     for ax in axs1.flat:
#         for e in e_vals:
#             ax.axvline(e, color='pink', linestyle='--', linewidth=1, zorder=0)
#     for y in e_vals:
#         ax_top1.axhline(y, color='deeppink', linestyle='--', linewidth=1)
#         ax_bottom1.axhline(y, color='deeppink', linestyle='--', linewidth=1)

#     plt.show()

#     #SUBPLOT DESCRIPTIONS:
#     #[0,0]: A - The anisotropy signal `A` averaged over the selected q-range, plotted as a function of energy
#     #[0,1]: dA/dE - The numerical derivative of the left trace (A vs energy), i.e., how the average anisotropy signal changes with energy
#     #[1,0]: A - cumulative sum of abs(A) across q, per energy. Essentially total magnitude of anisotropy signal
#     #[1,1]: dA/dq - cumulative sum of abs(dA/dq) across q, per energy. How much total spatial change occurs in the anisotropy, aggregating all directional changes
#     #Total variation is the arclength of a curve, e.g., high total variation is sharp changes, low total variation is more flat data
#     #[2,0]: A - total variation across q: sums the absolute change between neighboring q-points, per energy. This measures how much A fluctuates across q — effectively the 1D arclength of A vs q
#     #[2,1]: dA/dq - same idea applied to dA/dq: shows how sharply the derivative varies across q.
#     #[3,0]: A - root mean square (RMS) of A across q, per energy. This gives the typical magnitude of anisotropy at each energy, regardless of sign — useful for seeing where A is strongest overall.
#     #[3,1]: dA/dq - RMS of dA/dq across q, per energy. Captures the average strength of spatial variation in anisotropy, i.e., how strongly A changes with q at each energy.
#     #spectral entropy quantifies how regular variation is. flat/noisy signals = high spectral entropy. regular fluctuations (e.g. bragg peaks) = lower spectral entropy
#     #[4,0]: A - spectral entropy of A across q, per energy
#     #[4,1]: dA/dq - spectral entropy of dA/dq across q, per energy
#     #[5,0]: A - number of times A(q) changes sign across the qrange, per energy: counts spatial oscillations
#     #[5,1]: dA/dq - number of sign changes in dA/dq(q): captures how often slope direction flips

#     print("IDENTIFYING SUITABLE ENERGIES:\n"
#         "Energies with strong anisotropy and Bragg-like features are expected to have co-occuring:\n"
#         "HIGH values in axs:\n"
#         "  [1,0], [2,0], [3,0] — for overall anisotropy strength\n"
#         "  [1,1], [2,1], [3,1] — for spatial modulation strength\n"
#         "LOW values in axs:\n"
#         "  [4,0], [4,1] — low entropy implies regular, structured signal\n"
#         "MODERATE vlaues in axs:\n"
#         "  [5,0], [5,1] — very high counts suggest noise; very low may indicate flat signals\n"
#         "See documentation in function for further details\n")

#     print(f"(q_min_used, q_max_used): ({q_min_used:.4f}, {q_max_used:.4f})")

#     return fig1, q_min_used, q_max_used, vmin_A, vmax_A, vmin_dA_dq, vmax_dA_dq, emin, emax

In [ ]:
# #max q window possible
# q_min = q_min_trimmed
# q_max = q_max_trimmed
# e_vals=[284.8, 286.0]

# _, _, _, _, _, _, _, _, _ = ARmap_energy_selection(
#     e_vals=e_vals, q_min=q_min, q_max=q_max, emin=280, emax=290, vscale_A=0.03, vscale_dA_dq=10)


In [ ]:
# #max q window possible
# q_min = q_min_trimmed
# q_max = 0.03
# e_vals=[284.8, 286.0]

# _, _, _, _, _, _, _, _, _ = ARmap_energy_selection(
#     e_vals=e_vals, q_min=q_min, q_max=q_max, emin=280, emax=290, vscale_A=0.03, vscale_dA_dq=10)


In [ ]:
# energy_list = [284.8, 286]  
# q_min_used = q_min_trimmed
# q_max_used = q_max_trimmed

# color_map = plt.get_cmap("viridis")

# # FOR SELECTED ENERGIES
# fig, axs = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)

# q_vals = A_calcs["A_avg_mthd1_savgol"].sel(q=slice(q_min_used, q_max_used)).coords["q"]

# for i, energy in enumerate(energy_list):
#     A_at_energy = A_calcs["A_avg_mthd1_savgol"].sel(energy=energy, method="nearest").sel(q=slice(q_min_used, q_max_used)).values
#     axs[0].plot(q_vals, A_at_energy, label=f"{energy:.1f} eV", color=color_map(i / len(energy_list)))
#     axs[1].axhline(energy, linestyle="--", color=color_map(i / len(energy_list)))

# axs[0].set_xlabel(r"q $(\AA^{-1})$")
# axs[0].set_ylabel("Anisotropy")
# axs[0].set_xlim(q_min_trimmed, q_max_trimmed)
# axs[0].legend()

# fig.suptitle(
#     f"Energies selected for q range = [{q_min_used:.4f}, {q_max_used:.4f}]\n"
#     f"Scans {scan[0]['scan_id']}, {scan[90]['scan_id']}, {scan[45]['scan_id']} & {scan[135]['scan_id']}: {scan[0]["sample_name"]}: {scan[0]["base_description"]}\n"
#     f"(A is 8-quadrant average, savgol smoothed window={savgol_window_length})\n", 
#     y=1.15
# )

# A_calcs["A_avg_mthd1_savgol"].sel(energy=slice(280, 290)).sel(q=slice(q_min_used, q_max_used)).plot(ax=axs[1], vmin=-.05, vmax=0.05, cmap='coolwarm', cbar_kwargs={"label": "A", "shrink": 0.5})

# plt.show()
# plt.close()

Finaly, let's smooth the individual quadrant A calculations. Might want to use those later.

In [ ]:
# def smooth_quadrant_A(A_avg, name, savgol_window_length):

#     A_avg_savgol = A_avg.copy()
#     A_avg_savgol.values = savgol_filter(A_avg.values, window_length=savgol_window_length, polyorder=3, axis=-1)

#     dA_dq = A_avg.differentiate(coord="q") #doing it this way to preserve xarray so we can keep metadata... is it smart enough to know this is q savgol, not raw q?

#     fig, axs = plt.subplots(1,3,figsize=(20, 5))

#     # now can use .sel and .plot
#     #moving forward, dA/dq refers to data where A has been averaged across all 4 quadrants and savgol smoothed
#     A_avg.sel(energy=slice(280, 290)).plot(ax=axs[0],vmin=-.05, vmax=0.05, cmap='coolwarm')
#     A_avg_savgol.sel(energy = slice(280,290)).plot(ax=axs[1], vmin=-.05, vmax = 0.05, cmap = 'coolwarm')
#     dA_dq.sel(energy = slice(280,290)).plot(ax=axs[2], vmin=-15, vmax = 15, cmap = 'coolwarm')

#     axs[0].set_title(name)
#     axs[1].set_title(f"Savgol smooth, window_length={savgol_window_length}")
#     axs[2].set_title(f"dA/dq, after av and savgol smooth, window_length={savgol_window_length}")

#     fig.suptitle(
#         f"Smoothing and Differentiating A (individual quadrants)\n"
#         f"From Scans {scan[0]['scan_id']}, {scan[90]['scan_id']}, {scan[45]['scan_id']}, & {scan[135]['scan_id']}: {sample_name}: {base_description}\n"
#         f"A chi_widths: 12oclock={custom_chi_widths['12oclock']}, 9oclock={custom_chi_widths['9oclock']}, "
#         f"6oclock={custom_chi_widths['6oclock']}, 3oclock={custom_chi_widths['3oclock']}\n"
#         f"10p30oclock={custom_chi_widths['10p30oclock']}, 7p30oclock={custom_chi_widths['7p30oclock']}, "
#         f"4p30oclock={custom_chi_widths['4p30oclock']}, 1p30oclock={custom_chi_widths['1p30oclock']}",
#         fontsize=16
#     )

#     plt.tight_layout()
#     plt.show()
#     plt.close()

#     return A_avg_savgol, dA_dq

In [ ]:
# quadrant_keys = ["A_12", "A_9", "A_6", "A_3", "A_10p30", "A_7p30", "A_4p30", "A_1p30"]

# for k in quadrant_keys:
#     A_smooth, dA_dq = smooth_quadrant_A(A_calcs[k], k, savgol_window_length=21)

#     A_calcs[f"{k}_savgol"] = A_smooth
#     A_calcs[f"{k}_dA_dq"] = dA_dq

In [ ]:
# energy_1 = 284.8
# energy_2 = 286

# quadrants = ["A_12_savgol", "A_9_savgol", "A_6_savgol", "A_3_savgol", "A_10p30_savgol", "A_7p30_savgol", "A_4p30_savgol", "A_1p30_savgol"]
# color_map = plt.get_cmap("viridis")

# fig, axs = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)

# q_vals = A_calcs[quadrants[0]].sel(q=slice(q_min_used, q_max_used)).coords["q"]

# for i, k in enumerate(quadrants):

#     A1 = (A_calcs[k].sel(energy=energy_1, method="nearest").sel(q=slice(q_min_used, q_max_used)).values)
#     A2 = (A_calcs[k].sel(energy=energy_2, method="nearest").sel(q=slice(q_min_used, q_max_used)).values    )

#     axs[0].plot(q_vals, A1, label=k.replace("_savgol", ""), color=color_map(i / len(quadrants)))
#     axs[1].plot(q_vals, A2, label=k.replace("_savgol", ""), color=color_map(i / len(quadrants)))

# axs[0].set_title(f"Energy = {energy_1} eV")
# axs[1].set_title(f"Energy = {energy_2} eV")

# for ax in axs:
#     ax.set_xlabel(r"q $(\AA^{-1})$")
#     ax.set_ylabel("Anisotropy")
#     ax.legend()

# Save Results

In [ ]:
for pol, info in scan.items():
    print(f"Polarization {pol} keys:")
    for key in info.keys():
        print(f"  - {key}")

In [ ]:
print("A_calcs keys:")
for key in A_calcs.keys():
    print(f"  - {key}")

In [ ]:
save_dir = "/homes/ksr2/rsoxs_workup/sst1_2026apr/pickles"
scan_id_list = [str(sid) for sid in scan_ids.values()]

filename_scan = f"scan_dict_{'_'.join(scan_id_list)}_p{workup_pass}.pkl"
filepath_scan = os.path.join(save_dir, filename_scan)

filename_A = f"scan_dict_{'_'.join(scan_id_list)}_p{workup_pass}_Acalcs.pkl"
filepath_A = os.path.join(save_dir, filename_A)

print("starting scan save")
t0 = time.time()
with open(filepath_scan, "wb") as f:
    pickle.dump(scan, f)
print(f"finished scan save in {time.time() - t0:.1f} s")

print("starting A_calcs save")
t1 = time.time()
with open(filepath_A, "wb") as f:
    pickle.dump(A_calcs, f)
print(f"finished A_calcs save in {time.time() - t1:.1f} s")

# BREAK

General workflow: Average --> Savgol --> eyeball lower bound, calculate upper bound @ 280 --> IF we're at the very edges, need to trim in the q upper and lower bounds. but if not we can leave it.

From here on out, we're going to focus mainly on the energy range 280 to 290, because that's where the interesting stuff seems to be happening

In [ ]:
# for pol, info in scan.items():
#     print(f"Polarization {pol} keys:")
#     for key in info.keys():
#         print(f"  - {key}")

In [ ]:
#reorder (at integration) so all pols have the same key order

In [ ]:
# #creating a dictionary with A calc info
# A_calcs = {
#     "A_avg": A_avg,
#     "A_avg_savgol": A_avg_savgol,
#     "dA_dq": dA_dq,
#     "I_para_all": I_para_all,
#     "I_perp_all": I_perp_all,
#     "saved_1": saved_versions["saved_1"],
#     "saved_2": saved_versions["saved_2"],
#     "saved_3": saved_versions["saved_3"],
#     "base_description": base_description,
#     "sample_name": sample_name,
#     "diameter": diameter,
#     "mw": mw,
#     "polymer": polymer,
#     "wash_state": wash_state,
# }

# print("A_calcs keys:")
# for key in A_calcs.keys():
#     print(f"  - {key}")
        

BE SURE TO UPDATE WHERE PICKLES ARE GOING. WNAT TO USE ACTUAL PICLES FOLDER

In [ ]:
# save_dir = "/homes/ksr2/rsoxs_workup/sst1_2026apr/pickles"
# scan_id_list = [str(sid) for sid in scan_ids.values()]

# filename_scan = f"scan_dict_{'_'.join(scan_id_list)}_p01.pkl"
# filepath_scan = os.path.join(save_dir, filename_scan)

# filename_A = f"scan_dict_{'_'.join(scan_id_list)}_p01_Acalcs.pkl"
# filepath_A = os.path.join(save_dir, filename_A)

# print("starting scan save")
# t0 = time.time()
# with open(filepath_scan, "wb") as f:
#     pickle.dump(scan, f)
# print(f"finished scan save in {time.time() - t0:.1f} s")

# print("starting A_calcs save")
# t1 = time.time()
# with open(filepath_A, "wb") as f:
#     pickle.dump(A_calcs, f)
# print(f"finished A_calcs save in {time.time() - t1:.1f} s")

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# I_para_mean = A_calcs["I_para_mean"]
# I_perp_mean = A_calcs["I_perp_mean"]

# A_global = (I_para_mean - I_perp_mean) / (I_para_mean + I_perp_mean)

# quadrants = list(quadrant_centers.keys())
# nq = len(quadrants)

# bias = {}
# weight = {}
# contrib = {}

# for k in quadrants:
#     I_para = A_calcs[f"I_{k}_para"]
#     I_perp = A_calcs[f"I_{k}_perp"]

#     I_sum = I_para + I_perp
#     I_diff = I_para - I_perp

#     A_q = I_diff / I_sum

#     I_sum_m = I_sum.mean("chi", skipna=True)
#     A_q_m = A_q.mean("chi", skipna=True)

#     bias[k] = (A_q_m - A_global).mean("energy")

# weight_denom = sum(
#     (A_calcs[f"I_{kk}_para"] + A_calcs[f"I_{kk}_perp"]).mean("chi", skipna=True)
#     for kk in quadrants
# )

# for k in quadrants:
#     I_para = A_calcs[f"I_{k}_para"]
#     I_perp = A_calcs[f"I_{k}_perp"]

#     I_sum = I_para + I_perp
#     I_diff = I_para - I_perp
#     A_q = I_diff / I_sum

#     I_sum_m = I_sum.mean("chi", skipna=True)
#     A_q_m = A_q.mean("chi", skipna=True)

#     weight[k] = I_sum_m / weight_denom
#     contrib[k] = (A_q_m * weight[k]).mean("energy")


# # dynamic subplot sizing
# fig, axs = plt.subplots(3, nq, figsize=(3*nq, 10), squeeze=False)

# for i, k in enumerate(quadrants):

#     axs[0, i].plot(bias[k].values)
#     axs[0, i].set_title(f"{k}: A bias")

#     axs[1, i].plot(weight[k].mean("q").values)
#     axs[1, i].set_title(f"{k}: weight")

#     axs[2, i].plot(contrib[k].values)
#     axs[2, i].set_title(f"{k}: contribution")

# plt.tight_layout()
# plt.show()

In [ ]:
# #getting a sense of stability of quadrant brightness, contrast

# avg_denom = xarray.concat([
#     (A_calcs["I_12_para"] + A_calcs["I_12_perp"]),
#     (A_calcs["I_6_para"] + A_calcs["I_6_perp"]),
#     (A_calcs["I_3_para"] + A_calcs["I_3_perp"]),
#     (A_calcs["I_9_para"] + A_calcs["I_9_perp"]),
#     (A_calcs["I_10p30_para"] + A_calcs["I_10p30_perp"]),
#     (A_calcs["I_7p30_para"] + A_calcs["I_7p30_perp"]),
#     (A_calcs["I_4p30_para"] + A_calcs["I_4p30_perp"]),
#     (A_calcs["I_1p30_para"] + A_calcs["I_1p30_perp"]),
# ], dim="stack").mean("stack", skipna=True)

# print(A_calcs["I_12_para"].dims)
# print(avg_denom.dims)



In [ ]:
#do A vs E

#do plots of A vs q for each individual q. do at main energies. 